# Northwind: sales analysis in SQL

**The data.** Northwind is Microsoft's sample database of a small food importer: 8 tables, 91 customers, 830 orders and
2,155 order lines from July 1996 to May 1998. The CSVs in this folder are built from Microsoft's original install script
by `build_data.py` (MIT licence, see `NOTICE.md`), with the original table and column names.

**The questions.** How big is the business, how is it trending, what drove the last change, and can the numbers be trusted?

**How to read this notebook.** Part 1 runs four classic SQL mistakes next to the correct query, so that each error is
measured on this data. Part 2 is the full analysis AutoSQL Explorer generates, in its seven steps:
schema & grain → validate → build with CTEs → KPIs → window functions → root cause → fast & correct.
Every step explains 🔍 what we found before, 💡 why it matters, 🎯 what we're going to do and 🛠 what the SQL does;
after the code runs, 📌 says what we found, computed from the result so that the text can't contradict the output.
The SQL in the code cells is the DuckDB version that runs here; the MySQL 8 and PostgreSQL versions are in the
collapsed blocks under each step.

## Setup

In [1]:
import sys
from pathlib import Path

import duckdb
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

sys.path.insert(0, str(Path.cwd().resolve().parents[1]))  # the repository root, so that `autosql` can be imported
from autosql import build_database, read_folder

metrics = {}  # every cell stores its key numbers here


def note(text):
    """Write a plain-English finding under the cell."""
    try:
        from IPython.display import Markdown, display
        display(Markdown("📌 " + text))
    except ImportError:
        print("📌 " + text)


catalog = read_folder(".", "*.csv")        # the 8 original CSVs; names become snake_case (OrderID -> order_id)
con = build_database(catalog)              # in-memory DuckDB, one table per CSV
source = {name: t.df for name, t in catalog.tables.items()}
scratch = duckdb.connect()                 # an empty database, for the CREATE TABLE check


def sql(query, db=None):
    cursor = (db or con).execute(query)
    return cursor.df() if cursor.description else pd.DataFrame()


print({name: len(df) for name, df in source.items()})

import matplotlib as mpl
from matplotlib.colors import LinearSegmentedColormap

C_SERIES, C_DRIVER, C_TOTAL, C_MUTED = "#2a78d6", "#eb6834", "#52514e", "#c3c2b7"
C_UP, C_DOWN = "#2a78d6", "#e34948"
C_SEQ = LinearSegmentedColormap.from_list("autosql_blue", ["#cde2fb", "#86b6ef", "#3987e5", "#256abf", "#184f95", "#0d366b"])
mpl.rcParams.update({
    "axes.spines.top": False, "axes.spines.right": False, "axes.edgecolor": "#c3c2b7", "axes.linewidth": 0.8,
    "axes.grid": True, "axes.grid.axis": "y", "grid.color": "#e1e0d9", "grid.linewidth": 0.8, "axes.axisbelow": True,
    "axes.titlesize": 11, "axes.titleweight": "bold", "axes.labelcolor": "#52514e", "axes.titlecolor": "#0b0b0b",
    "xtick.color": "#52514e", "ytick.color": "#52514e", "font.size": 9.5, "lines.linewidth": 2, "figure.facecolor": "white",
})

{'categories': 8, 'customers': 91, 'employees': 9, 'order_details': 2155, 'orders': 830, 'products': 77, 'shippers': 3, 'suppliers': 29}


# Part 1: what a naive analysis gets wrong

### 1. Summing an order-level number after joining order lines (fan-out)

🔍 **What we found before:** `orders.freight` is stored once per order, but `order_details` has 1 to 25 lines per order.  
💡 **Why it matters:** Joining first repeats the freight on every line, so its total is multiplied by the number of lines.

In [2]:
naive = sql("""
SELECT SUM(o.freight) AS freight
FROM   orders o
JOIN   order_details od ON od.order_id = o.order_id
""").iloc[0, 0]
correct = sql("SELECT SUM(freight) AS freight FROM orders").iloc[0, 0]
note(f"Naive: **{naive:,.2f}**. Correct: **{correct:,.2f}**. The join inflates freight **{naive / correct:.2f}×**.")

📌 Naive: **207,306.10**. Correct: **64,942.69**. The join inflates freight **3.19×**.

### 2. Comparing a partial quarter with a full one

🔍 **What we found before:** The data ends on 6 May 1998, so 1998-Q2 has 36 of its 91 days.  
💡 **Why it matters:** Any part-period looks like a collapse. Compare complete periods only, or the same period a year earlier.

In [3]:
per_quarter = sql("""
SELECT EXTRACT(YEAR FROM o.order_date)::int AS order_year,
       EXTRACT(QUARTER FROM o.order_date)::int AS quarter_number,
       SUM(od.unit_price * od.quantity * (1 - od.discount)) AS revenue,
       MIN(o.order_date) AS first_day, MAX(o.order_date) AS last_day
FROM   order_details od
JOIN   orders o ON o.order_id = od.order_id
GROUP  BY 1, 2
ORDER  BY 1, 2
""")
display(per_quarter.tail(3))
naive = 100 * (per_quarter.revenue.iloc[-1] / per_quarter.revenue.iloc[-2] - 1)
correct = 100 * (per_quarter.revenue.iloc[-2] / per_quarter.revenue.iloc[-3] - 1)
note(f"Naive 'latest quarter': **{naive:+.1f}%**. Last two *complete* quarters (1997-Q4 → 1998-Q1): **{correct:+.1f}%**. "
     f"The data stops on {per_quarter.last_day.iloc[-1]:%d %B %Y}.")

,order_year,quarter_number,revenue,first_day,last_day
5,1997,4,181681.4635,1997-10-01,1997-12-31
6,1998,1,298491.5530,1998-01-01,1998-03-31
7,1998,2,142132.3130,1998-04-01,1998-05-06


📌 Naive 'latest quarter': **-52.4%**. Last two *complete* quarters (1997-Q4 → 1998-Q1): **+64.3%**. The data stops on 06 May 1998.

### 3. Averaging order lines instead of orders

🔍 **What we found before:** Revenue is stored per line; an order has 2.6 lines on average.  
💡 **Why it matters:** Average order value is revenue ÷ the number of distinct orders. The average of lines counts each line as if it were an order.

In [4]:
row = sql("""
SELECT AVG(unit_price * quantity * (1 - discount)) AS avg_of_lines,
       SUM(unit_price * quantity * (1 - discount)) / COUNT(DISTINCT order_id) AS avg_order_value
FROM   order_details
""").iloc[0]
note(f"Average of lines: **{row.avg_of_lines:,.2f}**. Average order value: **{row.avg_order_value:,.2f}** "
     f"({row.avg_order_value / row.avg_of_lines:.1f}× larger).")

📌 Average of lines: **587.37**. Average order value: **1,525.05** (2.6× larger).

### 4. NOT IN when the sub-query contains a NULL

🔍 **What we found before:** `employees.reports_to` is NULL for the one employee who reports to nobody.  
💡 **Why it matters:** `x NOT IN (…, NULL)` is never true, so NOT IN silently returns no rows. NOT EXISTS is safe.

In [5]:
row = sql("""
SELECT (SELECT COUNT(*) FROM employees e
        WHERE  e.employee_id NOT IN (SELECT reports_to FROM employees)) AS not_in,
       (SELECT COUNT(*) FROM employees e
        WHERE  NOT EXISTS (SELECT 1 FROM employees m WHERE m.reports_to = e.employee_id)) AS not_exists
""").iloc[0]
note(f"Employees who manage nobody: NOT IN says **{row.not_in}**, NOT EXISTS says **{row.not_exists}**.")

📌 Employees who manage nobody: NOT IN says **0**, NOT EXISTS says **7**.

# Part 2: the AutoSQL Explorer analysis

Generated from the Northwind CSVs in this folder. Run it from the folder that contains `data/` (the project ZIP has both).

Every step explains **what we found before**, **why it matters**, **what we're going to do** and **what the SQL does**. The SQL below is the DuckDB version that runs here; the MySQL and PostgreSQL versions are in the collapsed blocks. After a cell runs, the 📌 lines say **what we found**.

## Assumptions

- Fact table: `order_details` (one row per `order_id` × `product_id`).
- Revenue = unit_price × quantity × (1 − discount) (net of discount; gross is shown in the reconciliation).
- `order_facts` has one row per order (`order_id`); `line_facts` one row per line. AOV = revenue ÷ distinct orders.
- Periods are quarters of `orders.order_date`.
- 1998-Q2 is partial (40% of its days have data) and is excluded from period-over-period verdicts.
- ship_days = shipped_date − order_date, in days; orders without `shipped_date` are left out of its averages.
- Recency is measured to the last date in the data, not to today.
- Dimensions are joined with LEFT JOIN; a missing match is labelled 'Unknown'.

## Decision trace

- You uploaded 8 table(s) with 3,202 rows in total: `categories`, `customers`, `employees`, `order_details`, `orders`, `products` and 2 more. → We profile every table and column first, so that every later query is built on checked facts instead of guesses.
- 75 column name(s) were not plain SQL names (e.g. `CategoryID` → `category_id`, `CategoryName` → `category_name`, `Description` → `description`). → We renamed them to snake_case (`OrderID` → `order_id`), so that the SQL is readable and runs the same in MySQL and PostgreSQL. The column profile lists every original name.
- In `categories` (8 rows), `category_id` is unique and never empty. → We treat `category_id` as the primary key, so that joins and counts are checked against it.
- In `customers` (91 rows), `customer_id` is unique and never empty. → We treat `customer_id` as the primary key, so that joins and counts are checked against it.
- In `employees` (9 rows), `employee_id` is unique and never empty. → We treat `employee_id` as the primary key, so that joins and counts are checked against it.
- In `order_details` (2,155 rows), no single column is unique, but `order_id` + `product_id` together are. → We use (`order_id`, `product_id`) as the primary key, so that the grain is one row per `order_id` × `product_id` and order-level numbers are rolled up before they are joined.
- In `orders` (830 rows), `order_id` is unique and never empty. → We treat `order_id` as the primary key, so that joins and counts are checked against it.
- In `products` (77 rows), `product_id` is unique and never empty. → We treat `product_id` as the primary key, so that joins and counts are checked against it.
- In `shippers` (3 rows), `shipper_id` is unique and never empty. → We treat `shipper_id` as the primary key, so that joins and counts are checked against it.
- In `suppliers` (29 rows), `supplier_id` is unique and never empty. → We treat `supplier_id` as the primary key, so that joins and counts are checked against it.
- `order_details.order_id` → `orders.order_id`: named like the key of `orders`, and 100% of its values exist there. → We join on it as many-to-one, so that `orders` details can be added to `order_details` rows without multiplying them.
- `order_details.product_id` → `products.product_id`: named like the key of `products`, and 100% of its values exist there. → We join on it as many-to-one, so that `products` details can be added to `order_details` rows without multiplying them.
- `orders.customer_id` → `customers.customer_id`: named like the key of `customers`, and 100% of its values exist there. → We join on it as many-to-one, so that `customers` details can be added to `orders` rows without multiplying them.
- `orders.employee_id` → `employees.employee_id`: named like the key of `employees`, and 100% of its values exist there. → We join on it as many-to-one, so that `employees` details can be added to `orders` rows without multiplying them.
- `products.supplier_id` → `suppliers.supplier_id`: named like the key of `suppliers`, and 100% of its values exist there. → We join on it as many-to-one, so that `suppliers` details can be added to `products` rows without multiplying them.
- `products.category_id` → `categories.category_id`: named like the key of `categories`, and 100% of its values exist there. → We join on it as many-to-one, so that `categories` details can be added to `products` rows without multiplying them.
- `employees.reports_to` → `employees.employee_id`: named like a link to another row of `employees` (an org chart / tree), and every value is a `employee_id`. → We record the link and use it in the NULL-trap check, so that the classic `NOT IN` mistake is shown on your own data.
- `orders.ship_via` → `shippers.shipper_id`: its values are exactly the keys of `shippers` and the names share 'ship'. Matching values alone can be a coincidence, so tick it to use it. → We only suggest this link. Tick it under the diagram if it is real, so that it is checked and used like the others.
- `orders.freight` is stored once per `orders` row, but each `orders` row has up to 25 `order_details` rows. → We roll `order_details` up to one row per `orders` **before** joining, so that `freight` is counted once. The cardinality step shows both totals.
- `products.units_in_stock` is stored once per `products` row, but each `products` row has up to 54 `order_details` rows. → We roll `order_details` up to one row per `products` **before** joining, so that `units_in_stock` is counted once. The cardinality step shows both totals.
- You let the engine decide what to measure. → We look for the fact table, the revenue formula and the date, so that the analysis answers 'how much, how many, and how is it changing?'.
- `order_details` has 2,155 rows and holds `unit_price`, `quantity`, `discount`. → We use `order_details` as the fact table, so that revenue is computed once, at its true grain.
- `order_details` has 2,155 rows but only 830 distinct `order_id` values: one row per order **line**. → We roll lines up to one row per order first (`order_facts`), and keep a separate line-level view (`line_facts`) for product questions, so that each KPI uses its true grain.
- `order_details` holds a price (`unit_price`) and a quantity (`quantity`) and a discount (`discount`). → We define **revenue = SUM(unit_price × quantity × (1 - discount))**, so that every KPI uses one documented formula.
- The order date (and the other order-level columns) is in `orders`, one join away from `order_details` (via `order_id`). → We join `orders` after rolling the lines up, so that order-level columns are never repeated.
- `orders.order_date` runs from 1996-07-04 to 1998-05-06. → We use it for every trend and group the data by **quarter** (year *and* quarter, because the data spans more than one year), so that periods are comparable.
- 1998-Q2 only has data from 1998-04-01 to 1998-05-06: 36 of 91 days (40%). → We show it, flag it as **partial**, and leave it out of every period-over-period verdict, so that changes compare like with like.
- `shipped_date` is a later date than `order_date` in 100% of rows. → We compute **ship_days** per order and put it next to revenue in every trend, so that a change in the symptom can be checked against this driver.
- `shipped_date` is empty for 21 order(s) (2.5%). → We say so next to every average and count those orders separately, so that the average isn't misread.
- `required_date` holds the promised date. → We add **shipped_late** (1 = after `required_date`), so that the late share can be tracked per period and per group.
- `orders.customer_id` identifies the customer. → We use it for the customer-level steps, so that each customer is counted once.
- The best columns to slice by are `customers.country` (geography, 21 values), `categories.category_name` (product, 8 values), `employees.last_name` (people, 9 values). → We use them in the drill-down, so that a change can be traced to one segment the business can act on.
- The fact table has one row per order line. → We compute **average order value = revenue ÷ COUNT(DISTINCT order)**, and show the average line value next to it so that the difference is visible.
- Queries join on 6 foreign key column(s) and filter on `order_date`. → We propose CREATE INDEX statements for them, so that the same queries stay fast on a real database.

## 1. Write down the schema

*1 · Schema & grain*

🔍 **What we found before:** There are 8 tables and 7 trusted relationship(s) between them.  
💡 **Why it matters:** A schema written down as CREATE TABLE statements, with its primary and foreign keys, is the map every later query relies on. It also documents types: dates as DATE, money as DECIMAL, codes as text so leading zeros survive.  
🎯 **What we're going to do:** We write the CREATE TABLE statements, parents first, and test them on an empty database, so that the keys are declared exactly as the data supports.

🛠 **What this SQL does:** One CREATE TABLE per table, in an order where every referenced table already exists. Types come from the data (text length rounded up, NOT NULL when no value is missing). PRIMARY KEY is the key found during profiling; FOREIGN KEY is declared only for trusted links with no orphans. In the app the statements run on an empty database, as a syntax check.

<details><summary>MySQL 8</summary>

```sql
-- Why: We write the CREATE TABLE statements, parents first, and test them on an empty database, so
--      that the keys are declared exactly as the data supports.

-- 1) `categories`: one row per `category_id`
CREATE TABLE categories (
    category_id    INT NOT NULL,
    category_name  VARCHAR(20) NOT NULL,
    description    VARCHAR(100) NOT NULL,
    PRIMARY KEY (category_id)
);

-- 2) `customers`: one row per `customer_id`
CREATE TABLE customers (
    customer_id    VARCHAR(10) NOT NULL,
    company_name   VARCHAR(50) NOT NULL,
    contact_name   VARCHAR(50) NOT NULL,
    contact_title  VARCHAR(50) NOT NULL,
    address        VARCHAR(50) NOT NULL,
    city           VARCHAR(20) NOT NULL,
    region         VARCHAR(20),
    postal_code    VARCHAR(10),
    country        VARCHAR(20) NOT NULL,
    phone          VARCHAR(20) NOT NULL,
    fax            VARCHAR(20),
    PRIMARY KEY (customer_id)
);

-- 3) `employees`: one row per `employee_id`
CREATE TABLE employees (
    employee_id        INT NOT NULL,
    last_name          VARCHAR(10) NOT NULL,
    first_name         VARCHAR(10) NOT NULL,
    title              VARCHAR(50) NOT NULL,
    title_of_courtesy  VARCHAR(10) NOT NULL,
    birth_date         DATE NOT NULL,
    hire_date          DATE NOT NULL,
    address            VARCHAR(50) NOT NULL,
    city               VARCHAR(10) NOT NULL,
    region             VARCHAR(10),
    postal_code        VARCHAR(10) NOT NULL,
    country            VARCHAR(10) NOT NULL,
    home_phone         VARCHAR(20) NOT NULL,
    extension          INT NOT NULL,
    notes              VARCHAR(1000) NOT NULL,
    reports_to         INT,
    photo_path         VARCHAR(50) NOT NULL,
    PRIMARY KEY (employee_id),
    FOREIGN KEY (reports_to) REFERENCES employees (employee_id)
);

-- 4) `orders`: one row per `order_id`
CREATE TABLE orders (
    order_id          INT NOT NULL,
    customer_id       VARCHAR(10) NOT NULL,
    employee_id       INT NOT NULL,
    order_date        DATE NOT NULL,
    required_date     DATE NOT NULL,
    shipped_date      DATE,
    ship_via          INT NOT NULL,
    freight           DECIMAL(18, 2) NOT NULL,
    ship_name         VARCHAR(50) NOT NULL,
    ship_address      VARCHAR(50) NOT NULL,
    ship_city         VARCHAR(20) NOT NULL,
    ship_region       VARCHAR(20),
    ship_postal_code  VARCHAR(10),
    ship_country      VARCHAR(20) NOT NULL,
    PRIMARY KEY (order_id),
    FOREIGN KEY (customer_id) REFERENCES customers (customer_id),
    FOREIGN KEY (employee_id) REFERENCES employees (employee_id)
);

-- 5) `suppliers`: one row per `supplier_id`
CREATE TABLE suppliers (
    supplier_id    INT NOT NULL,
    company_name   VARCHAR(50) NOT NULL,
    contact_name   VARCHAR(50) NOT NULL,
    contact_title  VARCHAR(50) NOT NULL,
    address        VARCHAR(50) NOT NULL,
    city           VARCHAR(20) NOT NULL,
    region         VARCHAR(10),
    postal_code    VARCHAR(10) NOT NULL,
    country        VARCHAR(20) NOT NULL,
    phone          VARCHAR(20) NOT NULL,
    fax            VARCHAR(20),
    home_page      VARCHAR(100),
    PRIMARY KEY (supplier_id)
);

-- 6) `products`: one row per `product_id`
CREATE TABLE products (
    product_id         INT NOT NULL,
    product_name       VARCHAR(50) NOT NULL,
    supplier_id        INT NOT NULL,
    category_id        INT NOT NULL,
    quantity_per_unit  VARCHAR(20) NOT NULL,
    unit_price         DECIMAL(18, 2) NOT NULL,
    units_in_stock     INT NOT NULL,
    units_on_order     INT NOT NULL,
    reorder_level      INT NOT NULL,
    discontinued       INT NOT NULL,
    PRIMARY KEY (product_id),
    FOREIGN KEY (supplier_id) REFERENCES suppliers (supplier_id),
    FOREIGN KEY (category_id) REFERENCES categories (category_id)
);

-- 7) `order_details`: one row per `order_id` × `product_id`
CREATE TABLE order_details (
    order_id    INT NOT NULL,
    product_id  INT NOT NULL,
    unit_price  DECIMAL(18, 2) NOT NULL,
    quantity    INT NOT NULL,
    discount    DECIMAL(18, 4) NOT NULL,
    PRIMARY KEY (order_id, product_id),
    FOREIGN KEY (order_id) REFERENCES orders (order_id),
    FOREIGN KEY (product_id) REFERENCES products (product_id)
);

-- 8) `shippers`: one row per `shipper_id`
CREATE TABLE shippers (
    shipper_id    INT NOT NULL,
    company_name  VARCHAR(20) NOT NULL,
    phone         VARCHAR(20) NOT NULL,
    PRIMARY KEY (shipper_id)
);
```
</details>

<details><summary>PostgreSQL</summary>

```sql
-- Why: We write the CREATE TABLE statements, parents first, and test them on an empty database, so
--      that the keys are declared exactly as the data supports.

-- 1) `categories`: one row per `category_id`
CREATE TABLE categories (
    category_id    INTEGER NOT NULL,
    category_name  VARCHAR(20) NOT NULL,
    description    VARCHAR(100) NOT NULL,
    PRIMARY KEY (category_id)
);

-- 2) `customers`: one row per `customer_id`
CREATE TABLE customers (
    customer_id    VARCHAR(10) NOT NULL,
    company_name   VARCHAR(50) NOT NULL,
    contact_name   VARCHAR(50) NOT NULL,
    contact_title  VARCHAR(50) NOT NULL,
    address        VARCHAR(50) NOT NULL,
    city           VARCHAR(20) NOT NULL,
    region         VARCHAR(20),
    postal_code    VARCHAR(10),
    country        VARCHAR(20) NOT NULL,
    phone          VARCHAR(20) NOT NULL,
    fax            VARCHAR(20),
    PRIMARY KEY (customer_id)
);

-- 3) `employees`: one row per `employee_id`
CREATE TABLE employees (
    employee_id        INTEGER NOT NULL,
    last_name          VARCHAR(10) NOT NULL,
    first_name         VARCHAR(10) NOT NULL,
    title              VARCHAR(50) NOT NULL,
    title_of_courtesy  VARCHAR(10) NOT NULL,
    birth_date         DATE NOT NULL,
    hire_date          DATE NOT NULL,
    address            VARCHAR(50) NOT NULL,
    city               VARCHAR(10) NOT NULL,
    region             VARCHAR(10),
    postal_code        VARCHAR(10) NOT NULL,
    country            VARCHAR(10) NOT NULL,
    home_phone         VARCHAR(20) NOT NULL,
    extension          INTEGER NOT NULL,
    notes              VARCHAR(1000) NOT NULL,
    reports_to         INTEGER,
    photo_path         VARCHAR(50) NOT NULL,
    PRIMARY KEY (employee_id),
    FOREIGN KEY (reports_to) REFERENCES employees (employee_id)
);

-- 4) `orders`: one row per `order_id`
CREATE TABLE orders (
    order_id          INTEGER NOT NULL,
    customer_id       VARCHAR(10) NOT NULL,
    employee_id       INTEGER NOT NULL,
    order_date        DATE NOT NULL,
    required_date     DATE NOT NULL,
    shipped_date      DATE,
    ship_via          INTEGER NOT NULL,
    freight           NUMERIC(18, 2) NOT NULL,
    ship_name         VARCHAR(50) NOT NULL,
    ship_address      VARCHAR(50) NOT NULL,
    ship_city         VARCHAR(20) NOT NULL,
    ship_region       VARCHAR(20),
    ship_postal_code  VARCHAR(10),
    ship_country      VARCHAR(20) NOT NULL,
    PRIMARY KEY (order_id),
    FOREIGN KEY (customer_id) REFERENCES customers (customer_id),
    FOREIGN KEY (employee_id) REFERENCES employees (employee_id)
);

-- 5) `suppliers`: one row per `supplier_id`
CREATE TABLE suppliers (
    supplier_id    INTEGER NOT NULL,
    company_name   VARCHAR(50) NOT NULL,
    contact_name   VARCHAR(50) NOT NULL,
    contact_title  VARCHAR(50) NOT NULL,
    address        VARCHAR(50) NOT NULL,
    city           VARCHAR(20) NOT NULL,
    region         VARCHAR(10),
    postal_code    VARCHAR(10) NOT NULL,
    country        VARCHAR(20) NOT NULL,
    phone          VARCHAR(20) NOT NULL,
    fax            VARCHAR(20),
    home_page      VARCHAR(100),
    PRIMARY KEY (supplier_id)
);

-- 6) `products`: one row per `product_id`
CREATE TABLE products (
    product_id         INTEGER NOT NULL,
    product_name       VARCHAR(50) NOT NULL,
    supplier_id        INTEGER NOT NULL,
    category_id        INTEGER NOT NULL,
    quantity_per_unit  VARCHAR(20) NOT NULL,
    unit_price         NUMERIC(18, 2) NOT NULL,
    units_in_stock     INTEGER NOT NULL,
    units_on_order     INTEGER NOT NULL,
    reorder_level      INTEGER NOT NULL,
    discontinued       INTEGER NOT NULL,
    PRIMARY KEY (product_id),
    FOREIGN KEY (supplier_id) REFERENCES suppliers (supplier_id),
    FOREIGN KEY (category_id) REFERENCES categories (category_id)
);

-- 7) `order_details`: one row per `order_id` × `product_id`
CREATE TABLE order_details (
    order_id    INTEGER NOT NULL,
    product_id  INTEGER NOT NULL,
    unit_price  NUMERIC(18, 2) NOT NULL,
    quantity    INTEGER NOT NULL,
    discount    NUMERIC(18, 4) NOT NULL,
    PRIMARY KEY (order_id, product_id),
    FOREIGN KEY (order_id) REFERENCES orders (order_id),
    FOREIGN KEY (product_id) REFERENCES products (product_id)
);

-- 8) `shippers`: one row per `shipper_id`
CREATE TABLE shippers (
    shipper_id    INTEGER NOT NULL,
    company_name  VARCHAR(20) NOT NULL,
    phone         VARCHAR(20) NOT NULL,
    PRIMARY KEY (shipper_id)
);
```
</details>

In [6]:
outputs = {}
# `categories`: one row per `category_id`
outputs['create_categories'] = sql(r"""
CREATE TABLE categories (
    category_id    INTEGER NOT NULL,
    category_name  VARCHAR(20) NOT NULL,
    description    VARCHAR(100) NOT NULL,
    PRIMARY KEY (category_id)
)
""", scratch)
# `customers`: one row per `customer_id`
outputs['create_customers'] = sql(r"""
CREATE TABLE customers (
    customer_id    VARCHAR(10) NOT NULL,
    company_name   VARCHAR(50) NOT NULL,
    contact_name   VARCHAR(50) NOT NULL,
    contact_title  VARCHAR(50) NOT NULL,
    address        VARCHAR(50) NOT NULL,
    city           VARCHAR(20) NOT NULL,
    region         VARCHAR(20),
    postal_code    VARCHAR(10),
    country        VARCHAR(20) NOT NULL,
    phone          VARCHAR(20) NOT NULL,
    fax            VARCHAR(20),
    PRIMARY KEY (customer_id)
)
""", scratch)
# `employees`: one row per `employee_id`
outputs['create_employees'] = sql(r"""
CREATE TABLE employees (
    employee_id        INTEGER NOT NULL,
    last_name          VARCHAR(10) NOT NULL,
    first_name         VARCHAR(10) NOT NULL,
    title              VARCHAR(50) NOT NULL,
    title_of_courtesy  VARCHAR(10) NOT NULL,
    birth_date         DATE NOT NULL,
    hire_date          DATE NOT NULL,
    address            VARCHAR(50) NOT NULL,
    city               VARCHAR(10) NOT NULL,
    region             VARCHAR(10),
    postal_code        VARCHAR(10) NOT NULL,
    country            VARCHAR(10) NOT NULL,
    home_phone         VARCHAR(20) NOT NULL,
    extension          INTEGER NOT NULL,
    notes              VARCHAR(1000) NOT NULL,
    reports_to         INTEGER,
    photo_path         VARCHAR(50) NOT NULL,
    PRIMARY KEY (employee_id),
    FOREIGN KEY (reports_to) REFERENCES employees (employee_id)
)
""", scratch)
# `orders`: one row per `order_id`
outputs['create_orders'] = sql(r"""
CREATE TABLE orders (
    order_id          INTEGER NOT NULL,
    customer_id       VARCHAR(10) NOT NULL,
    employee_id       INTEGER NOT NULL,
    order_date        DATE NOT NULL,
    required_date     DATE NOT NULL,
    shipped_date      DATE,
    ship_via          INTEGER NOT NULL,
    freight           DECIMAL(18, 2) NOT NULL,
    ship_name         VARCHAR(50) NOT NULL,
    ship_address      VARCHAR(50) NOT NULL,
    ship_city         VARCHAR(20) NOT NULL,
    ship_region       VARCHAR(20),
    ship_postal_code  VARCHAR(10),
    ship_country      VARCHAR(20) NOT NULL,
    PRIMARY KEY (order_id),
    FOREIGN KEY (customer_id) REFERENCES customers (customer_id),
    FOREIGN KEY (employee_id) REFERENCES employees (employee_id)
)
""", scratch)
# `suppliers`: one row per `supplier_id`
outputs['create_suppliers'] = sql(r"""
CREATE TABLE suppliers (
    supplier_id    INTEGER NOT NULL,
    company_name   VARCHAR(50) NOT NULL,
    contact_name   VARCHAR(50) NOT NULL,
    contact_title  VARCHAR(50) NOT NULL,
    address        VARCHAR(50) NOT NULL,
    city           VARCHAR(20) NOT NULL,
    region         VARCHAR(10),
    postal_code    VARCHAR(10) NOT NULL,
    country        VARCHAR(20) NOT NULL,
    phone          VARCHAR(20) NOT NULL,
    fax            VARCHAR(20),
    home_page      VARCHAR(100),
    PRIMARY KEY (supplier_id)
)
""", scratch)
# `products`: one row per `product_id`
outputs['create_products'] = sql(r"""
CREATE TABLE products (
    product_id         INTEGER NOT NULL,
    product_name       VARCHAR(50) NOT NULL,
    supplier_id        INTEGER NOT NULL,
    category_id        INTEGER NOT NULL,
    quantity_per_unit  VARCHAR(20) NOT NULL,
    unit_price         DECIMAL(18, 2) NOT NULL,
    units_in_stock     INTEGER NOT NULL,
    units_on_order     INTEGER NOT NULL,
    reorder_level      INTEGER NOT NULL,
    discontinued       INTEGER NOT NULL,
    PRIMARY KEY (product_id),
    FOREIGN KEY (supplier_id) REFERENCES suppliers (supplier_id),
    FOREIGN KEY (category_id) REFERENCES categories (category_id)
)
""", scratch)
# `order_details`: one row per `order_id` × `product_id`
outputs['create_order_details'] = sql(r"""
CREATE TABLE order_details (
    order_id    INTEGER NOT NULL,
    product_id  INTEGER NOT NULL,
    unit_price  DECIMAL(18, 2) NOT NULL,
    quantity    INTEGER NOT NULL,
    discount    DECIMAL(18, 4) NOT NULL,
    PRIMARY KEY (order_id, product_id),
    FOREIGN KEY (order_id) REFERENCES orders (order_id),
    FOREIGN KEY (product_id) REFERENCES products (product_id)
)
""", scratch)
# `shippers`: one row per `shipper_id`
outputs['create_shippers'] = sql(r"""
CREATE TABLE shippers (
    shipper_id    INTEGER NOT NULL,
    company_name  VARCHAR(20) NOT NULL,
    phone         VARCHAR(20) NOT NULL,
    PRIMARY KEY (shipper_id)
)
""", scratch)
result = list(outputs.values())[-1] if outputs else pd.DataFrame()
# 📌 What we found
metrics["tables"] = 8
metrics["foreign_keys"] = 7
note(f"All 8 CREATE TABLE statements ran on an empty database, with 8 primary key(s) and 7 foreign key(s) declared.")

📌 All 8 CREATE TABLE statements ran on an empty database, with 8 primary key(s) and 7 foreign key(s) declared.

## 2. Prove the grain

*1 · Schema & grain*

🔍 **What we found before:** Profiling found a key for each table: `categories`: one row per `category_id`; `customers`: one row per `customer_id`; `employees`: one row per `employee_id`; `order_details`: one row per `order_id` × `product_id`….  
💡 **Why it matters:** If a key is really unique, COUNT(*) equals the number of distinct keys. If not, every join on it multiplies rows. COUNT(DISTINCT key) ignores NULLs, so empty keys are counted separately.  
🎯 **What we're going to do:** We compare rows with distinct keys in every table, so that the grain is proven by SQL, not assumed.

🛠 **What this SQL does:** For each table: COUNT(*), the number of distinct key values (for a two-column key, the number of distinct pairs), and how many rows have an empty key. One SELECT per table, stacked with UNION ALL.

<details><summary>MySQL 8</summary>

```sql
-- Why: We compare rows with distinct keys in every table, so that the grain is proven by SQL, not
--      assumed.

-- Rows vs distinct keys per table
SELECT 'categories' AS table_name,
       'category_id' AS key_columns,
       COUNT(*) AS row_count,
       COUNT(DISTINCT category_id) AS distinct_keys,
       SUM(CASE WHEN category_id IS NULL THEN 1 ELSE 0 END) AS null_keys
FROM   categories
UNION ALL
SELECT 'customers' AS table_name,
       'customer_id' AS key_columns,
       COUNT(*) AS row_count,
       COUNT(DISTINCT customer_id) AS distinct_keys,
       SUM(CASE WHEN customer_id IS NULL THEN 1 ELSE 0 END) AS null_keys
FROM   customers
UNION ALL
SELECT 'employees' AS table_name,
       'employee_id' AS key_columns,
       COUNT(*) AS row_count,
       COUNT(DISTINCT employee_id) AS distinct_keys,
       SUM(CASE WHEN employee_id IS NULL THEN 1 ELSE 0 END) AS null_keys
FROM   employees
UNION ALL
SELECT 'order_details' AS table_name,
       'order_id + product_id' AS key_columns,
       COUNT(*) AS row_count,
       (SELECT COUNT(*) FROM (SELECT DISTINCT order_id, product_id FROM order_details) k) AS distinct_keys,
       SUM(CASE WHEN order_id IS NULL OR product_id IS NULL THEN 1 ELSE 0 END) AS null_keys
FROM   order_details
UNION ALL
SELECT 'orders' AS table_name,
       'order_id' AS key_columns,
       COUNT(*) AS row_count,
       COUNT(DISTINCT order_id) AS distinct_keys,
       SUM(CASE WHEN order_id IS NULL THEN 1 ELSE 0 END) AS null_keys
FROM   orders
UNION ALL
SELECT 'products' AS table_name,
       'product_id' AS key_columns,
       COUNT(*) AS row_count,
       COUNT(DISTINCT product_id) AS distinct_keys,
       SUM(CASE WHEN product_id IS NULL THEN 1 ELSE 0 END) AS null_keys
FROM   products
UNION ALL
SELECT 'shippers' AS table_name,
       'shipper_id' AS key_columns,
       COUNT(*) AS row_count,
       COUNT(DISTINCT shipper_id) AS distinct_keys,
       SUM(CASE WHEN shipper_id IS NULL THEN 1 ELSE 0 END) AS null_keys
FROM   shippers
UNION ALL
SELECT 'suppliers' AS table_name,
       'supplier_id' AS key_columns,
       COUNT(*) AS row_count,
       COUNT(DISTINCT supplier_id) AS distinct_keys,
       SUM(CASE WHEN supplier_id IS NULL THEN 1 ELSE 0 END) AS null_keys
FROM   suppliers;
```
</details>

<details><summary>PostgreSQL</summary>

```sql
-- Why: We compare rows with distinct keys in every table, so that the grain is proven by SQL, not
--      assumed.

-- Rows vs distinct keys per table
SELECT 'categories' AS table_name,
       'category_id' AS key_columns,
       COUNT(*) AS row_count,
       COUNT(DISTINCT category_id) AS distinct_keys,
       SUM(CASE WHEN category_id IS NULL THEN 1 ELSE 0 END) AS null_keys
FROM   categories
UNION ALL
SELECT 'customers' AS table_name,
       'customer_id' AS key_columns,
       COUNT(*) AS row_count,
       COUNT(DISTINCT customer_id) AS distinct_keys,
       SUM(CASE WHEN customer_id IS NULL THEN 1 ELSE 0 END) AS null_keys
FROM   customers
UNION ALL
SELECT 'employees' AS table_name,
       'employee_id' AS key_columns,
       COUNT(*) AS row_count,
       COUNT(DISTINCT employee_id) AS distinct_keys,
       SUM(CASE WHEN employee_id IS NULL THEN 1 ELSE 0 END) AS null_keys
FROM   employees
UNION ALL
SELECT 'order_details' AS table_name,
       'order_id + product_id' AS key_columns,
       COUNT(*) AS row_count,
       (SELECT COUNT(*) FROM (SELECT DISTINCT order_id, product_id FROM order_details) k) AS distinct_keys,
       SUM(CASE WHEN order_id IS NULL OR product_id IS NULL THEN 1 ELSE 0 END) AS null_keys
FROM   order_details
UNION ALL
SELECT 'orders' AS table_name,
       'order_id' AS key_columns,
       COUNT(*) AS row_count,
       COUNT(DISTINCT order_id) AS distinct_keys,
       SUM(CASE WHEN order_id IS NULL THEN 1 ELSE 0 END) AS null_keys
FROM   orders
UNION ALL
SELECT 'products' AS table_name,
       'product_id' AS key_columns,
       COUNT(*) AS row_count,
       COUNT(DISTINCT product_id) AS distinct_keys,
       SUM(CASE WHEN product_id IS NULL THEN 1 ELSE 0 END) AS null_keys
FROM   products
UNION ALL
SELECT 'shippers' AS table_name,
       'shipper_id' AS key_columns,
       COUNT(*) AS row_count,
       COUNT(DISTINCT shipper_id) AS distinct_keys,
       SUM(CASE WHEN shipper_id IS NULL THEN 1 ELSE 0 END) AS null_keys
FROM   shippers
UNION ALL
SELECT 'suppliers' AS table_name,
       'supplier_id' AS key_columns,
       COUNT(*) AS row_count,
       COUNT(DISTINCT supplier_id) AS distinct_keys,
       SUM(CASE WHEN supplier_id IS NULL THEN 1 ELSE 0 END) AS null_keys
FROM   suppliers;
```
</details>

In [7]:
outputs = {}
# Rows vs distinct keys per table
outputs['grain'] = sql(r"""
SELECT 'categories' AS table_name,
       'category_id' AS key_columns,
       COUNT(*) AS row_count,
       COUNT(DISTINCT category_id) AS distinct_keys,
       SUM(CASE WHEN category_id IS NULL THEN 1 ELSE 0 END) AS null_keys
FROM   categories
UNION ALL
SELECT 'customers' AS table_name,
       'customer_id' AS key_columns,
       COUNT(*) AS row_count,
       COUNT(DISTINCT customer_id) AS distinct_keys,
       SUM(CASE WHEN customer_id IS NULL THEN 1 ELSE 0 END) AS null_keys
FROM   customers
UNION ALL
SELECT 'employees' AS table_name,
       'employee_id' AS key_columns,
       COUNT(*) AS row_count,
       COUNT(DISTINCT employee_id) AS distinct_keys,
       SUM(CASE WHEN employee_id IS NULL THEN 1 ELSE 0 END) AS null_keys
FROM   employees
UNION ALL
SELECT 'order_details' AS table_name,
       'order_id + product_id' AS key_columns,
       COUNT(*) AS row_count,
       (SELECT COUNT(*) FROM (SELECT DISTINCT order_id, product_id FROM order_details) k) AS distinct_keys,
       SUM(CASE WHEN order_id IS NULL OR product_id IS NULL THEN 1 ELSE 0 END) AS null_keys
FROM   order_details
UNION ALL
SELECT 'orders' AS table_name,
       'order_id' AS key_columns,
       COUNT(*) AS row_count,
       COUNT(DISTINCT order_id) AS distinct_keys,
       SUM(CASE WHEN order_id IS NULL THEN 1 ELSE 0 END) AS null_keys
FROM   orders
UNION ALL
SELECT 'products' AS table_name,
       'product_id' AS key_columns,
       COUNT(*) AS row_count,
       COUNT(DISTINCT product_id) AS distinct_keys,
       SUM(CASE WHEN product_id IS NULL THEN 1 ELSE 0 END) AS null_keys
FROM   products
UNION ALL
SELECT 'shippers' AS table_name,
       'shipper_id' AS key_columns,
       COUNT(*) AS row_count,
       COUNT(DISTINCT shipper_id) AS distinct_keys,
       SUM(CASE WHEN shipper_id IS NULL THEN 1 ELSE 0 END) AS null_keys
FROM   shippers
UNION ALL
SELECT 'suppliers' AS table_name,
       'supplier_id' AS key_columns,
       COUNT(*) AS row_count,
       COUNT(DISTINCT supplier_id) AS distinct_keys,
       SUM(CASE WHEN supplier_id IS NULL THEN 1 ELSE 0 END) AS null_keys
FROM   suppliers
""")
display(outputs['grain'].head(20))
result = list(outputs.values())[-1] if outputs else pd.DataFrame()
# 📌 What we found
bad = result[(result["row_count"] != result["distinct_keys"]) | (result["null_keys"] > 0)]
metrics["grain_ok"] = bad.empty
for row in result.itertuples():
    ok = row.row_count == row.distinct_keys and row.null_keys == 0
    metrics.setdefault("grain", {})[row.table_name] = {"rows": int(row.row_count), "distinct": int(row.distinct_keys), "ok": bool(ok)}
if bad.empty:
    note(f"Every key is unique and never empty: in all {len(result)} tables, COUNT(*) equals the number of distinct keys.")
for row in bad.itertuples():
    note(f"`{row.table_name}`: {int(row.row_count):,} rows but {int(row.distinct_keys):,} distinct `{row.key_columns}` "
         f"and {int(row.null_keys):,} empty key(s). Joins on this key will repeat rows until it is de-duplicated.")

,table_name,key_columns,row_count,distinct_keys,null_keys
0,categories,category_id,8,8,0.0
1,customers,customer_id,91,91,0.0
2,employees,employee_id,9,9,0.0
3,order_details,order_id + product_id,2155,2155,0.0
4,orders,order_id,830,830,0.0
5,products,product_id,77,77,0.0
6,shippers,shipper_id,3,3,0.0
7,suppliers,supplier_id,29,29,0.0


📌 Every key is unique and never empty: in all 8 tables, COUNT(*) equals the number of distinct keys.

## 3. Join cardinality and fan-out

*1 · Schema & grain*

🔍 **What we found before:** There are 7 trusted relationship(s), and `orders.freight` sits on the 'one' side of a one-to-many join.  
💡 **Why it matters:** A 1:N join repeats every parent row once per child. Anything stored on the parent (freight, a budget, a target) is then summed several times: the classic fan-out that silently inflates totals.  
🎯 **What we're going to do:** We measure children per parent for every link and show `freight` summed before and after the join, so that the size of the trap is visible in your own numbers.

🛠 **What this SQL does:** For each relationship, a LEFT JOIN from parent to child counts the children of every parent (0 when none), then MIN / AVG / MAX summarise them. The fan-out demo sums the parent column once per parent, and again after an inner join to the child table.

<details><summary>MySQL 8</summary>

```sql
-- Why: We measure children per parent for every link and show freight summed before and after the
--      join, so that the size of the trap is visible in your own numbers.

-- 1) Child rows per parent row, for every relationship
SELECT 'order_details.order_id -> orders' AS relationship,
       COUNT(*) AS parents,
       SUM(CASE WHEN children > 0 THEN 1 ELSE 0 END) AS parents_with_children,
       MIN(children) AS min_children,
       ROUND(AVG(children), 2) AS avg_children,
       MAX(children) AS max_children
FROM   (SELECT p.order_id, COUNT(c.order_id) AS children
        FROM   orders p
        LEFT   JOIN order_details c ON c.order_id = p.order_id
        GROUP  BY p.order_id) per_parent
UNION ALL
SELECT 'order_details.product_id -> products' AS relationship,
       COUNT(*) AS parents,
       SUM(CASE WHEN children > 0 THEN 1 ELSE 0 END) AS parents_with_children,
       MIN(children) AS min_children,
       ROUND(AVG(children), 2) AS avg_children,
       MAX(children) AS max_children
FROM   (SELECT p.product_id, COUNT(c.product_id) AS children
        FROM   products p
        LEFT   JOIN order_details c ON c.product_id = p.product_id
        GROUP  BY p.product_id) per_parent
UNION ALL
SELECT 'orders.customer_id -> customers' AS relationship,
       COUNT(*) AS parents,
       SUM(CASE WHEN children > 0 THEN 1 ELSE 0 END) AS parents_with_children,
       MIN(children) AS min_children,
       ROUND(AVG(children), 2) AS avg_children,
       MAX(children) AS max_children
FROM   (SELECT p.customer_id, COUNT(c.customer_id) AS children
        FROM   customers p
        LEFT   JOIN orders c ON c.customer_id = p.customer_id
        GROUP  BY p.customer_id) per_parent
UNION ALL
SELECT 'orders.employee_id -> employees' AS relationship,
       COUNT(*) AS parents,
       SUM(CASE WHEN children > 0 THEN 1 ELSE 0 END) AS parents_with_children,
       MIN(children) AS min_children,
       ROUND(AVG(children), 2) AS avg_children,
       MAX(children) AS max_children
FROM   (SELECT p.employee_id, COUNT(c.employee_id) AS children
        FROM   employees p
        LEFT   JOIN orders c ON c.employee_id = p.employee_id
        GROUP  BY p.employee_id) per_parent
UNION ALL
SELECT 'products.supplier_id -> suppliers' AS relationship,
       COUNT(*) AS parents,
       SUM(CASE WHEN children > 0 THEN 1 ELSE 0 END) AS parents_with_children,
       MIN(children) AS min_children,
       ROUND(AVG(children), 2) AS avg_children,
       MAX(children) AS max_children
FROM   (SELECT p.supplier_id, COUNT(c.supplier_id) AS children
        FROM   suppliers p
        LEFT   JOIN products c ON c.supplier_id = p.supplier_id
        GROUP  BY p.supplier_id) per_parent
UNION ALL
SELECT 'products.category_id -> categories' AS relationship,
       COUNT(*) AS parents,
       SUM(CASE WHEN children > 0 THEN 1 ELSE 0 END) AS parents_with_children,
       MIN(children) AS min_children,
       ROUND(AVG(children), 2) AS avg_children,
       MAX(children) AS max_children
FROM   (SELECT p.category_id, COUNT(c.category_id) AS children
        FROM   categories p
        LEFT   JOIN products c ON c.category_id = p.category_id
        GROUP  BY p.category_id) per_parent
UNION ALL
SELECT 'employees.reports_to -> employees' AS relationship,
       COUNT(*) AS parents,
       SUM(CASE WHEN children > 0 THEN 1 ELSE 0 END) AS parents_with_children,
       MIN(children) AS min_children,
       ROUND(AVG(children), 2) AS avg_children,
       MAX(children) AS max_children
FROM   (SELECT p.employee_id, COUNT(c.reports_to) AS children
        FROM   employees p
        LEFT   JOIN employees c ON c.reports_to = p.employee_id
        GROUP  BY p.employee_id) per_parent;

-- 2) Fan-out demo: `orders.freight` before and after joining `order_details`
-- freight is stored once per orders row; joining order_details repeats it on every order_details row.
SELECT (SELECT SUM(freight) FROM orders) AS total_counted_once,
       (SELECT SUM(o.freight)
        FROM   orders o
        JOIN   order_details od ON od.order_id = o.order_id) AS total_after_join;

-- 3) Fan-out demo: `products.units_in_stock` before and after joining `order_details`
-- units_in_stock is stored once per products row; joining order_details repeats it on every order_details row.
SELECT (SELECT SUM(units_in_stock) FROM products) AS total_counted_once,
       (SELECT SUM(p.units_in_stock)
        FROM   products p
        JOIN   order_details od ON od.product_id = p.product_id) AS total_after_join;
```
</details>

<details><summary>PostgreSQL</summary>

```sql
-- Why: We measure children per parent for every link and show freight summed before and after the
--      join, so that the size of the trap is visible in your own numbers.

-- 1) Child rows per parent row, for every relationship
SELECT 'order_details.order_id -> orders' AS relationship,
       COUNT(*) AS parents,
       SUM(CASE WHEN children > 0 THEN 1 ELSE 0 END) AS parents_with_children,
       MIN(children) AS min_children,
       ROUND((AVG(children))::numeric, 2) AS avg_children,
       MAX(children) AS max_children
FROM   (SELECT p.order_id, COUNT(c.order_id) AS children
        FROM   orders p
        LEFT   JOIN order_details c ON c.order_id = p.order_id
        GROUP  BY p.order_id) per_parent
UNION ALL
SELECT 'order_details.product_id -> products' AS relationship,
       COUNT(*) AS parents,
       SUM(CASE WHEN children > 0 THEN 1 ELSE 0 END) AS parents_with_children,
       MIN(children) AS min_children,
       ROUND((AVG(children))::numeric, 2) AS avg_children,
       MAX(children) AS max_children
FROM   (SELECT p.product_id, COUNT(c.product_id) AS children
        FROM   products p
        LEFT   JOIN order_details c ON c.product_id = p.product_id
        GROUP  BY p.product_id) per_parent
UNION ALL
SELECT 'orders.customer_id -> customers' AS relationship,
       COUNT(*) AS parents,
       SUM(CASE WHEN children > 0 THEN 1 ELSE 0 END) AS parents_with_children,
       MIN(children) AS min_children,
       ROUND((AVG(children))::numeric, 2) AS avg_children,
       MAX(children) AS max_children
FROM   (SELECT p.customer_id, COUNT(c.customer_id) AS children
        FROM   customers p
        LEFT   JOIN orders c ON c.customer_id = p.customer_id
        GROUP  BY p.customer_id) per_parent
UNION ALL
SELECT 'orders.employee_id -> employees' AS relationship,
       COUNT(*) AS parents,
       SUM(CASE WHEN children > 0 THEN 1 ELSE 0 END) AS parents_with_children,
       MIN(children) AS min_children,
       ROUND((AVG(children))::numeric, 2) AS avg_children,
       MAX(children) AS max_children
FROM   (SELECT p.employee_id, COUNT(c.employee_id) AS children
        FROM   employees p
        LEFT   JOIN orders c ON c.employee_id = p.employee_id
        GROUP  BY p.employee_id) per_parent
UNION ALL
SELECT 'products.supplier_id -> suppliers' AS relationship,
       COUNT(*) AS parents,
       SUM(CASE WHEN children > 0 THEN 1 ELSE 0 END) AS parents_with_children,
       MIN(children) AS min_children,
       ROUND((AVG(children))::numeric, 2) AS avg_children,
       MAX(children) AS max_children
FROM   (SELECT p.supplier_id, COUNT(c.supplier_id) AS children
        FROM   suppliers p
        LEFT   JOIN products c ON c.supplier_id = p.supplier_id
        GROUP  BY p.supplier_id) per_parent
UNION ALL
SELECT 'products.category_id -> categories' AS relationship,
       COUNT(*) AS parents,
       SUM(CASE WHEN children > 0 THEN 1 ELSE 0 END) AS parents_with_children,
       MIN(children) AS min_children,
       ROUND((AVG(children))::numeric, 2) AS avg_children,
       MAX(children) AS max_children
FROM   (SELECT p.category_id, COUNT(c.category_id) AS children
        FROM   categories p
        LEFT   JOIN products c ON c.category_id = p.category_id
        GROUP  BY p.category_id) per_parent
UNION ALL
SELECT 'employees.reports_to -> employees' AS relationship,
       COUNT(*) AS parents,
       SUM(CASE WHEN children > 0 THEN 1 ELSE 0 END) AS parents_with_children,
       MIN(children) AS min_children,
       ROUND((AVG(children))::numeric, 2) AS avg_children,
       MAX(children) AS max_children
FROM   (SELECT p.employee_id, COUNT(c.reports_to) AS children
        FROM   employees p
        LEFT   JOIN employees c ON c.reports_to = p.employee_id
        GROUP  BY p.employee_id) per_parent;

-- 2) Fan-out demo: `orders.freight` before and after joining `order_details`
-- freight is stored once per orders row; joining order_details repeats it on every order_details row.
SELECT (SELECT SUM(freight) FROM orders) AS total_counted_once,
       (SELECT SUM(o.freight)
        FROM   orders o
        JOIN   order_details od ON od.order_id = o.order_id) AS total_after_join;

-- 3) Fan-out demo: `products.units_in_stock` before and after joining `order_details`
-- units_in_stock is stored once per products row; joining order_details repeats it on every order_details row.
SELECT (SELECT SUM(units_in_stock) FROM products) AS total_counted_once,
       (SELECT SUM(p.units_in_stock)
        FROM   products p
        JOIN   order_details od ON od.product_id = p.product_id) AS total_after_join;
```
</details>

In [8]:
outputs = {}
# Child rows per parent row, for every relationship
outputs['cardinality'] = sql(r"""
SELECT 'order_details.order_id -> orders' AS relationship,
       COUNT(*) AS parents,
       SUM(CASE WHEN children > 0 THEN 1 ELSE 0 END) AS parents_with_children,
       MIN(children) AS min_children,
       ROUND(AVG(children), 2) AS avg_children,
       MAX(children) AS max_children
FROM   (SELECT p.order_id, COUNT(c.order_id) AS children
        FROM   orders p
        LEFT   JOIN order_details c ON c.order_id = p.order_id
        GROUP  BY p.order_id) per_parent
UNION ALL
SELECT 'order_details.product_id -> products' AS relationship,
       COUNT(*) AS parents,
       SUM(CASE WHEN children > 0 THEN 1 ELSE 0 END) AS parents_with_children,
       MIN(children) AS min_children,
       ROUND(AVG(children), 2) AS avg_children,
       MAX(children) AS max_children
FROM   (SELECT p.product_id, COUNT(c.product_id) AS children
        FROM   products p
        LEFT   JOIN order_details c ON c.product_id = p.product_id
        GROUP  BY p.product_id) per_parent
UNION ALL
SELECT 'orders.customer_id -> customers' AS relationship,
       COUNT(*) AS parents,
       SUM(CASE WHEN children > 0 THEN 1 ELSE 0 END) AS parents_with_children,
       MIN(children) AS min_children,
       ROUND(AVG(children), 2) AS avg_children,
       MAX(children) AS max_children
FROM   (SELECT p.customer_id, COUNT(c.customer_id) AS children
        FROM   customers p
        LEFT   JOIN orders c ON c.customer_id = p.customer_id
        GROUP  BY p.customer_id) per_parent
UNION ALL
SELECT 'orders.employee_id -> employees' AS relationship,
       COUNT(*) AS parents,
       SUM(CASE WHEN children > 0 THEN 1 ELSE 0 END) AS parents_with_children,
       MIN(children) AS min_children,
       ROUND(AVG(children), 2) AS avg_children,
       MAX(children) AS max_children
FROM   (SELECT p.employee_id, COUNT(c.employee_id) AS children
        FROM   employees p
        LEFT   JOIN orders c ON c.employee_id = p.employee_id
        GROUP  BY p.employee_id) per_parent
UNION ALL
SELECT 'products.supplier_id -> suppliers' AS relationship,
       COUNT(*) AS parents,
       SUM(CASE WHEN children > 0 THEN 1 ELSE 0 END) AS parents_with_children,
       MIN(children) AS min_children,
       ROUND(AVG(children), 2) AS avg_children,
       MAX(children) AS max_children
FROM   (SELECT p.supplier_id, COUNT(c.supplier_id) AS children
        FROM   suppliers p
        LEFT   JOIN products c ON c.supplier_id = p.supplier_id
        GROUP  BY p.supplier_id) per_parent
UNION ALL
SELECT 'products.category_id -> categories' AS relationship,
       COUNT(*) AS parents,
       SUM(CASE WHEN children > 0 THEN 1 ELSE 0 END) AS parents_with_children,
       MIN(children) AS min_children,
       ROUND(AVG(children), 2) AS avg_children,
       MAX(children) AS max_children
FROM   (SELECT p.category_id, COUNT(c.category_id) AS children
        FROM   categories p
        LEFT   JOIN products c ON c.category_id = p.category_id
        GROUP  BY p.category_id) per_parent
UNION ALL
SELECT 'employees.reports_to -> employees' AS relationship,
       COUNT(*) AS parents,
       SUM(CASE WHEN children > 0 THEN 1 ELSE 0 END) AS parents_with_children,
       MIN(children) AS min_children,
       ROUND(AVG(children), 2) AS avg_children,
       MAX(children) AS max_children
FROM   (SELECT p.employee_id, COUNT(c.reports_to) AS children
        FROM   employees p
        LEFT   JOIN employees c ON c.reports_to = p.employee_id
        GROUP  BY p.employee_id) per_parent
""")
display(outputs['cardinality'].head(20))
# Fan-out demo: `orders.freight` before and after joining `order_details`
outputs['fanout_1'] = sql(r"""
-- freight is stored once per orders row; joining order_details repeats it on every order_details row.
SELECT (SELECT SUM(freight) FROM orders) AS total_counted_once,
       (SELECT SUM(o.freight)
        FROM   orders o
        JOIN   order_details od ON od.order_id = o.order_id) AS total_after_join
""")
display(outputs['fanout_1'].head(20))
# Fan-out demo: `products.units_in_stock` before and after joining `order_details`
outputs['fanout_2'] = sql(r"""
-- units_in_stock is stored once per products row; joining order_details repeats it on every order_details row.
SELECT (SELECT SUM(units_in_stock) FROM products) AS total_counted_once,
       (SELECT SUM(p.units_in_stock)
        FROM   products p
        JOIN   order_details od ON od.product_id = p.product_id) AS total_after_join
""")
display(outputs['fanout_2'].head(20))
result = list(outputs.values())[-1] if outputs else pd.DataFrame()
# 📌 What we found
rel = outputs["cardinality"]
for row in rel.itertuples():
    kind = "1:1" if row.max_children <= 1 else "1:N"
    metrics.setdefault("cardinality", {})[row.relationship] = {"kind": kind, "max": int(row.max_children), "avg": float(row.avg_children)}
widest = rel.sort_values("max_children", ascending=False).iloc[0]
note(f"Widest join: `{widest.relationship}`, up to {int(widest.max_children):,} child rows for one parent "
     f"({float(widest.avg_children):.2f} on average; {int(widest.parents - widest.parents_with_children):,} parent(s) have none).")
for label, name in [('fanout_1', 'orders.freight'), ('fanout_2', 'products.units_in_stock')]:
    row = outputs[label].iloc[0]
    once, joined = float(row.total_counted_once), float(row.total_after_join)
    factor = joined / once if once else float("nan")
    metrics.setdefault("fanout", {"column": name, "total": once, "joined": joined, "factor": factor})
    note(f"**Fan-out:** `{name}` totals **{once:,.2f}** counted once, but **{joined:,.2f}** after the join "
         f"({factor:.2f}× too high). It must be summed before joining, or at its own grain.")

,relationship,parents,parents_with_children,min_children,avg_children,max_children
0,order_details.order_id -> orders,830,830.0,1,2.60,25
1,order_details.product_id -> products,77,77.0,5,27.99,54
2,orders.customer_id -> customers,91,89.0,0,9.12,31
3,orders.employee_id -> employees,9,9.0,42,92.22,156
4,products.supplier_id -> suppliers,29,29.0,1,2.66,5
5,products.category_id -> categories,8,8.0,5,9.63,13
6,employees.reports_to -> employees,9,2.0,0,0.89,5


,total_counted_once,total_after_join
0,64942.69,207306.1


,total_counted_once,total_after_join
0,3119.0,85760.0


📌 Widest join: `orders.employee_id -> employees`, up to 156 child rows for one parent (92.22 on average; 0 parent(s) have none).

📌 **Fan-out:** `orders.freight` totals **64,942.69** counted once, but **207,306.10** after the join (3.19× too high). It must be summed before joining, or at its own grain.

📌 **Fan-out:** `products.units_in_stock` totals **3,119.00** counted once, but **85,760.00** after the join (27.50× too high). It must be summed before joining, or at its own grain.

## 4. Find duplicates

*2 · Validate*

🔍 **What we found before:** Profiling found no repeated key, but a profile is only as good as the query behind it.  
💡 **Why it matters:** A duplicated key is either a double-loaded row or a sign that the table's grain is finer than it looks. Either way it inflates every count and sum that joins to it.  
🎯 **What we're going to do:** We list duplicated keys with GROUP BY … HAVING COUNT(*) > 1, so that each one can be traced back to the source.

🛠 **What this SQL does:** Groups each table by its key and keeps only the groups with more than one row; a table without a key is grouped by all of its columns, which finds exact copies. The summary counts the duplicated values and the extra rows they add.

<details><summary>MySQL 8</summary>

```sql
-- Why: We list duplicated keys with GROUP BY … HAVING COUNT() > 1, so that each one can be traced
--      back to the source.

-- Duplicated keys per table (GROUP BY … HAVING COUNT(*) > 1)
SELECT 'categories' AS table_name,
       'category_id' AS checked_columns,
       COUNT(*) AS duplicated_values,
       COALESCE(SUM(copies - 1), 0) AS extra_rows
FROM   (SELECT category_id, COUNT(*) AS copies
        FROM   categories
        WHERE  category_id IS NOT NULL
        GROUP  BY category_id
        HAVING COUNT(*) > 1) dup
UNION ALL
SELECT 'customers' AS table_name,
       'customer_id' AS checked_columns,
       COUNT(*) AS duplicated_values,
       COALESCE(SUM(copies - 1), 0) AS extra_rows
FROM   (SELECT customer_id, COUNT(*) AS copies
        FROM   customers
        WHERE  customer_id IS NOT NULL
        GROUP  BY customer_id
        HAVING COUNT(*) > 1) dup
UNION ALL
SELECT 'employees' AS table_name,
       'employee_id' AS checked_columns,
       COUNT(*) AS duplicated_values,
       COALESCE(SUM(copies - 1), 0) AS extra_rows
FROM   (SELECT employee_id, COUNT(*) AS copies
        FROM   employees
        WHERE  employee_id IS NOT NULL
        GROUP  BY employee_id
        HAVING COUNT(*) > 1) dup
UNION ALL
SELECT 'order_details' AS table_name,
       'order_id + product_id' AS checked_columns,
       COUNT(*) AS duplicated_values,
       COALESCE(SUM(copies - 1), 0) AS extra_rows
FROM   (SELECT order_id, product_id, COUNT(*) AS copies
        FROM   order_details
        WHERE  order_id IS NOT NULL AND product_id IS NOT NULL
        GROUP  BY order_id, product_id
        HAVING COUNT(*) > 1) dup
UNION ALL
SELECT 'orders' AS table_name,
       'order_id' AS checked_columns,
       COUNT(*) AS duplicated_values,
       COALESCE(SUM(copies - 1), 0) AS extra_rows
FROM   (SELECT order_id, COUNT(*) AS copies
        FROM   orders
        WHERE  order_id IS NOT NULL
        GROUP  BY order_id
        HAVING COUNT(*) > 1) dup
UNION ALL
SELECT 'products' AS table_name,
       'product_id' AS checked_columns,
       COUNT(*) AS duplicated_values,
       COALESCE(SUM(copies - 1), 0) AS extra_rows
FROM   (SELECT product_id, COUNT(*) AS copies
        FROM   products
        WHERE  product_id IS NOT NULL
        GROUP  BY product_id
        HAVING COUNT(*) > 1) dup
UNION ALL
SELECT 'shippers' AS table_name,
       'shipper_id' AS checked_columns,
       COUNT(*) AS duplicated_values,
       COALESCE(SUM(copies - 1), 0) AS extra_rows
FROM   (SELECT shipper_id, COUNT(*) AS copies
        FROM   shippers
        WHERE  shipper_id IS NOT NULL
        GROUP  BY shipper_id
        HAVING COUNT(*) > 1) dup
UNION ALL
SELECT 'suppliers' AS table_name,
       'supplier_id' AS checked_columns,
       COUNT(*) AS duplicated_values,
       COALESCE(SUM(copies - 1), 0) AS extra_rows
FROM   (SELECT supplier_id, COUNT(*) AS copies
        FROM   suppliers
        WHERE  supplier_id IS NOT NULL
        GROUP  BY supplier_id
        HAVING COUNT(*) > 1) dup;
```
</details>

<details><summary>PostgreSQL</summary>

```sql
-- Why: We list duplicated keys with GROUP BY … HAVING COUNT() > 1, so that each one can be traced
--      back to the source.

-- Duplicated keys per table (GROUP BY … HAVING COUNT(*) > 1)
SELECT 'categories' AS table_name,
       'category_id' AS checked_columns,
       COUNT(*) AS duplicated_values,
       COALESCE(SUM(copies - 1), 0) AS extra_rows
FROM   (SELECT category_id, COUNT(*) AS copies
        FROM   categories
        WHERE  category_id IS NOT NULL
        GROUP  BY category_id
        HAVING COUNT(*) > 1) dup
UNION ALL
SELECT 'customers' AS table_name,
       'customer_id' AS checked_columns,
       COUNT(*) AS duplicated_values,
       COALESCE(SUM(copies - 1), 0) AS extra_rows
FROM   (SELECT customer_id, COUNT(*) AS copies
        FROM   customers
        WHERE  customer_id IS NOT NULL
        GROUP  BY customer_id
        HAVING COUNT(*) > 1) dup
UNION ALL
SELECT 'employees' AS table_name,
       'employee_id' AS checked_columns,
       COUNT(*) AS duplicated_values,
       COALESCE(SUM(copies - 1), 0) AS extra_rows
FROM   (SELECT employee_id, COUNT(*) AS copies
        FROM   employees
        WHERE  employee_id IS NOT NULL
        GROUP  BY employee_id
        HAVING COUNT(*) > 1) dup
UNION ALL
SELECT 'order_details' AS table_name,
       'order_id + product_id' AS checked_columns,
       COUNT(*) AS duplicated_values,
       COALESCE(SUM(copies - 1), 0) AS extra_rows
FROM   (SELECT order_id, product_id, COUNT(*) AS copies
        FROM   order_details
        WHERE  order_id IS NOT NULL AND product_id IS NOT NULL
        GROUP  BY order_id, product_id
        HAVING COUNT(*) > 1) dup
UNION ALL
SELECT 'orders' AS table_name,
       'order_id' AS checked_columns,
       COUNT(*) AS duplicated_values,
       COALESCE(SUM(copies - 1), 0) AS extra_rows
FROM   (SELECT order_id, COUNT(*) AS copies
        FROM   orders
        WHERE  order_id IS NOT NULL
        GROUP  BY order_id
        HAVING COUNT(*) > 1) dup
UNION ALL
SELECT 'products' AS table_name,
       'product_id' AS checked_columns,
       COUNT(*) AS duplicated_values,
       COALESCE(SUM(copies - 1), 0) AS extra_rows
FROM   (SELECT product_id, COUNT(*) AS copies
        FROM   products
        WHERE  product_id IS NOT NULL
        GROUP  BY product_id
        HAVING COUNT(*) > 1) dup
UNION ALL
SELECT 'shippers' AS table_name,
       'shipper_id' AS checked_columns,
       COUNT(*) AS duplicated_values,
       COALESCE(SUM(copies - 1), 0) AS extra_rows
FROM   (SELECT shipper_id, COUNT(*) AS copies
        FROM   shippers
        WHERE  shipper_id IS NOT NULL
        GROUP  BY shipper_id
        HAVING COUNT(*) > 1) dup
UNION ALL
SELECT 'suppliers' AS table_name,
       'supplier_id' AS checked_columns,
       COUNT(*) AS duplicated_values,
       COALESCE(SUM(copies - 1), 0) AS extra_rows
FROM   (SELECT supplier_id, COUNT(*) AS copies
        FROM   suppliers
        WHERE  supplier_id IS NOT NULL
        GROUP  BY supplier_id
        HAVING COUNT(*) > 1) dup;
```
</details>

In [9]:
outputs = {}
# Duplicated keys per table (GROUP BY … HAVING COUNT(*) > 1)
outputs['duplicates'] = sql(r"""
SELECT 'categories' AS table_name,
       'category_id' AS checked_columns,
       COUNT(*) AS duplicated_values,
       COALESCE(SUM(copies - 1), 0) AS extra_rows
FROM   (SELECT category_id, COUNT(*) AS copies
        FROM   categories
        WHERE  category_id IS NOT NULL
        GROUP  BY category_id
        HAVING COUNT(*) > 1) dup
UNION ALL
SELECT 'customers' AS table_name,
       'customer_id' AS checked_columns,
       COUNT(*) AS duplicated_values,
       COALESCE(SUM(copies - 1), 0) AS extra_rows
FROM   (SELECT customer_id, COUNT(*) AS copies
        FROM   customers
        WHERE  customer_id IS NOT NULL
        GROUP  BY customer_id
        HAVING COUNT(*) > 1) dup
UNION ALL
SELECT 'employees' AS table_name,
       'employee_id' AS checked_columns,
       COUNT(*) AS duplicated_values,
       COALESCE(SUM(copies - 1), 0) AS extra_rows
FROM   (SELECT employee_id, COUNT(*) AS copies
        FROM   employees
        WHERE  employee_id IS NOT NULL
        GROUP  BY employee_id
        HAVING COUNT(*) > 1) dup
UNION ALL
SELECT 'order_details' AS table_name,
       'order_id + product_id' AS checked_columns,
       COUNT(*) AS duplicated_values,
       COALESCE(SUM(copies - 1), 0) AS extra_rows
FROM   (SELECT order_id, product_id, COUNT(*) AS copies
        FROM   order_details
        WHERE  order_id IS NOT NULL AND product_id IS NOT NULL
        GROUP  BY order_id, product_id
        HAVING COUNT(*) > 1) dup
UNION ALL
SELECT 'orders' AS table_name,
       'order_id' AS checked_columns,
       COUNT(*) AS duplicated_values,
       COALESCE(SUM(copies - 1), 0) AS extra_rows
FROM   (SELECT order_id, COUNT(*) AS copies
        FROM   orders
        WHERE  order_id IS NOT NULL
        GROUP  BY order_id
        HAVING COUNT(*) > 1) dup
UNION ALL
SELECT 'products' AS table_name,
       'product_id' AS checked_columns,
       COUNT(*) AS duplicated_values,
       COALESCE(SUM(copies - 1), 0) AS extra_rows
FROM   (SELECT product_id, COUNT(*) AS copies
        FROM   products
        WHERE  product_id IS NOT NULL
        GROUP  BY product_id
        HAVING COUNT(*) > 1) dup
UNION ALL
SELECT 'shippers' AS table_name,
       'shipper_id' AS checked_columns,
       COUNT(*) AS duplicated_values,
       COALESCE(SUM(copies - 1), 0) AS extra_rows
FROM   (SELECT shipper_id, COUNT(*) AS copies
        FROM   shippers
        WHERE  shipper_id IS NOT NULL
        GROUP  BY shipper_id
        HAVING COUNT(*) > 1) dup
UNION ALL
SELECT 'suppliers' AS table_name,
       'supplier_id' AS checked_columns,
       COUNT(*) AS duplicated_values,
       COALESCE(SUM(copies - 1), 0) AS extra_rows
FROM   (SELECT supplier_id, COUNT(*) AS copies
        FROM   suppliers
        WHERE  supplier_id IS NOT NULL
        GROUP  BY supplier_id
        HAVING COUNT(*) > 1) dup
""")
display(outputs['duplicates'].head(20))
result = list(outputs.values())[-1] if outputs else pd.DataFrame()
# 📌 What we found
dup = outputs["duplicates"]
metrics["duplicate_keys"] = {row.table_name: int(row.duplicated_values) for row in dup.itertuples()}
bad = dup[dup["duplicated_values"] > 0]
if bad.empty:
    note("No duplicates: every key value appears exactly once in its table.")
for row in bad.itertuples():
    note(f"`{row.table_name}`: {int(row.duplicated_values):,} `{row.checked_columns}` value(s) repeat, adding "
         f"{int(row.extra_rows):,} extra row(s). The analysis keeps one row per key with ROW_NUMBER().")

,table_name,checked_columns,duplicated_values,extra_rows
0,categories,category_id,0,0.0
1,customers,customer_id,0,0.0
2,employees,employee_id,0,0.0
3,order_details,order_id + product_id,0,0.0
4,orders,order_id,0,0.0
5,products,product_id,0,0.0
6,shippers,shipper_id,0,0.0
7,suppliers,supplier_id,0,0.0


📌 No duplicates: every key value appears exactly once in its table.

## 5. Profile NULLs

*2 · Validate*

🔍 **What we found before:** 11 column(s) have empty values, e.g. `suppliers.home_page` (83%), `suppliers.region` (69%), `customers.region` (66%).  
💡 **Why it matters:** NULL means unknown. COUNT(col), SUM and AVG silently skip it, so an average over a gappy column describes fewer rows than you think, and a NULL foreign key drops the row from an inner join.  
🎯 **What we're going to do:** We count NULLs in every column, so that every average and join later can be read with the right caveat.

🛠 **What this SQL does:** For every column: COUNT(*) − COUNT(column). COUNT(*) counts rows; COUNT(column) counts only rows where the column has a value, so the difference is the number of NULLs. One line per column, stacked with UNION ALL.

<details><summary>MySQL 8</summary>

```sql
-- Why: We count NULLs in every column, so that every average and join later can be read with the
--      right caveat.

-- NULLs per column: COUNT(*) − COUNT(column)
-- Only the columns that had NULLs when the data was profiled; add the others back to re-check new data.
SELECT 'customers' AS table_name, 'region' AS column_name, COUNT(*) AS row_count, COUNT(*) - COUNT(region) AS null_rows FROM customers
UNION ALL
SELECT 'customers' AS table_name, 'postal_code' AS column_name, COUNT(*) AS row_count, COUNT(*) - COUNT(postal_code) AS null_rows FROM customers
UNION ALL
SELECT 'customers' AS table_name, 'fax' AS column_name, COUNT(*) AS row_count, COUNT(*) - COUNT(fax) AS null_rows FROM customers
UNION ALL
SELECT 'employees' AS table_name, 'region' AS column_name, COUNT(*) AS row_count, COUNT(*) - COUNT(region) AS null_rows FROM employees
UNION ALL
SELECT 'employees' AS table_name, 'reports_to' AS column_name, COUNT(*) AS row_count, COUNT(*) - COUNT(reports_to) AS null_rows FROM employees
UNION ALL
SELECT 'orders' AS table_name, 'shipped_date' AS column_name, COUNT(*) AS row_count, COUNT(*) - COUNT(shipped_date) AS null_rows FROM orders
UNION ALL
SELECT 'orders' AS table_name, 'ship_region' AS column_name, COUNT(*) AS row_count, COUNT(*) - COUNT(ship_region) AS null_rows FROM orders
UNION ALL
SELECT 'orders' AS table_name, 'ship_postal_code' AS column_name, COUNT(*) AS row_count, COUNT(*) - COUNT(ship_postal_code) AS null_rows FROM orders
UNION ALL
SELECT 'suppliers' AS table_name, 'region' AS column_name, COUNT(*) AS row_count, COUNT(*) - COUNT(region) AS null_rows FROM suppliers
UNION ALL
SELECT 'suppliers' AS table_name, 'fax' AS column_name, COUNT(*) AS row_count, COUNT(*) - COUNT(fax) AS null_rows FROM suppliers
UNION ALL
SELECT 'suppliers' AS table_name, 'home_page' AS column_name, COUNT(*) AS row_count, COUNT(*) - COUNT(home_page) AS null_rows FROM suppliers;
```
</details>

<details><summary>PostgreSQL</summary>

```sql
-- Why: We count NULLs in every column, so that every average and join later can be read with the
--      right caveat.

-- NULLs per column: COUNT(*) − COUNT(column)
-- Only the columns that had NULLs when the data was profiled; add the others back to re-check new data.
SELECT 'customers' AS table_name, 'region' AS column_name, COUNT(*) AS row_count, COUNT(*) - COUNT(region) AS null_rows FROM customers
UNION ALL
SELECT 'customers' AS table_name, 'postal_code' AS column_name, COUNT(*) AS row_count, COUNT(*) - COUNT(postal_code) AS null_rows FROM customers
UNION ALL
SELECT 'customers' AS table_name, 'fax' AS column_name, COUNT(*) AS row_count, COUNT(*) - COUNT(fax) AS null_rows FROM customers
UNION ALL
SELECT 'employees' AS table_name, 'region' AS column_name, COUNT(*) AS row_count, COUNT(*) - COUNT(region) AS null_rows FROM employees
UNION ALL
SELECT 'employees' AS table_name, 'reports_to' AS column_name, COUNT(*) AS row_count, COUNT(*) - COUNT(reports_to) AS null_rows FROM employees
UNION ALL
SELECT 'orders' AS table_name, 'shipped_date' AS column_name, COUNT(*) AS row_count, COUNT(*) - COUNT(shipped_date) AS null_rows FROM orders
UNION ALL
SELECT 'orders' AS table_name, 'ship_region' AS column_name, COUNT(*) AS row_count, COUNT(*) - COUNT(ship_region) AS null_rows FROM orders
UNION ALL
SELECT 'orders' AS table_name, 'ship_postal_code' AS column_name, COUNT(*) AS row_count, COUNT(*) - COUNT(ship_postal_code) AS null_rows FROM orders
UNION ALL
SELECT 'suppliers' AS table_name, 'region' AS column_name, COUNT(*) AS row_count, COUNT(*) - COUNT(region) AS null_rows FROM suppliers
UNION ALL
SELECT 'suppliers' AS table_name, 'fax' AS column_name, COUNT(*) AS row_count, COUNT(*) - COUNT(fax) AS null_rows FROM suppliers
UNION ALL
SELECT 'suppliers' AS table_name, 'home_page' AS column_name, COUNT(*) AS row_count, COUNT(*) - COUNT(home_page) AS null_rows FROM suppliers;
```
</details>

In [10]:
outputs = {}
# NULLs per column: COUNT(*) − COUNT(column)
outputs['nulls'] = sql(r"""
-- Only the columns that had NULLs when the data was profiled; add the others back to re-check new data.
SELECT 'customers' AS table_name, 'region' AS column_name, COUNT(*) AS row_count, COUNT(*) - COUNT(region) AS null_rows FROM customers
UNION ALL
SELECT 'customers' AS table_name, 'postal_code' AS column_name, COUNT(*) AS row_count, COUNT(*) - COUNT(postal_code) AS null_rows FROM customers
UNION ALL
SELECT 'customers' AS table_name, 'fax' AS column_name, COUNT(*) AS row_count, COUNT(*) - COUNT(fax) AS null_rows FROM customers
UNION ALL
SELECT 'employees' AS table_name, 'region' AS column_name, COUNT(*) AS row_count, COUNT(*) - COUNT(region) AS null_rows FROM employees
UNION ALL
SELECT 'employees' AS table_name, 'reports_to' AS column_name, COUNT(*) AS row_count, COUNT(*) - COUNT(reports_to) AS null_rows FROM employees
UNION ALL
SELECT 'orders' AS table_name, 'shipped_date' AS column_name, COUNT(*) AS row_count, COUNT(*) - COUNT(shipped_date) AS null_rows FROM orders
UNION ALL
SELECT 'orders' AS table_name, 'ship_region' AS column_name, COUNT(*) AS row_count, COUNT(*) - COUNT(ship_region) AS null_rows FROM orders
UNION ALL
SELECT 'orders' AS table_name, 'ship_postal_code' AS column_name, COUNT(*) AS row_count, COUNT(*) - COUNT(ship_postal_code) AS null_rows FROM orders
UNION ALL
SELECT 'suppliers' AS table_name, 'region' AS column_name, COUNT(*) AS row_count, COUNT(*) - COUNT(region) AS null_rows FROM suppliers
UNION ALL
SELECT 'suppliers' AS table_name, 'fax' AS column_name, COUNT(*) AS row_count, COUNT(*) - COUNT(fax) AS null_rows FROM suppliers
UNION ALL
SELECT 'suppliers' AS table_name, 'home_page' AS column_name, COUNT(*) AS row_count, COUNT(*) - COUNT(home_page) AS null_rows FROM suppliers
""")
display(outputs['nulls'].head(20))
result = list(outputs.values())[-1] if outputs else pd.DataFrame()
# 📌 What we found
nulls = result.assign(null_pct=100 * result["null_rows"] / result["row_count"].clip(lower=1))
gappy = nulls[nulls["null_rows"] > 0].sort_values("null_pct", ascending=False)
metrics["nulls"] = {f"{r.table_name}.{r.column_name}": round(float(r.null_pct), 1) for r in gappy.itertuples()}
if gappy.empty:
    note("No NULLs anywhere: every column is filled in every row.")
else:
    top = gappy.head(12).iloc[::-1]
    plt.figure(figsize=(8, max(2.5, 0.35 * len(top))))
    plt.barh(top["table_name"] + "." + top["column_name"], top["null_pct"], color=C_SERIES, height=0.6)
    plt.xlabel("% of rows empty"); plt.title("Columns with NULLs")
    plt.tight_layout(); plt.show()
    note(f"{len(gappy)} column(s) have NULLs. Most empty: " + ", ".join(
        f"`{r.table_name}.{r.column_name}` {r.null_pct:.1f}%" for r in gappy.head(3).itertuples()) + ".")
for table, col, meaning in [('orders', 'shipped_date', 'those orders have no ship_days, so AVG(ship_days) leaves them out.'), ('employees', 'reports_to', 'those rows have no `employees` row, so an INNER JOIN would drop them.')]:
    row = nulls[(nulls["table_name"] == table) & (nulls["column_name"] == col)]
    if len(row) and int(row["null_rows"].iloc[0]):
        note(f"`{table}.{col}` is empty in {int(row['null_rows'].iloc[0]):,} row(s): {meaning}")

,table_name,column_name,row_count,null_rows
0,customers,region,91,60
1,customers,postal_code,91,1
2,customers,fax,91,22
3,employees,region,9,4
4,employees,reports_to,9,1
5,orders,shipped_date,830,21
6,orders,ship_region,830,507
7,orders,ship_postal_code,830,19
8,suppliers,region,29,20
9,suppliers,fax,29,16


📌 11 column(s) have NULLs. Most empty: `suppliers.home_page` 82.8%, `suppliers.region` 69.0%, `customers.region` 65.9%.

📌 `orders.shipped_date` is empty in 21 row(s): those orders have no ship_days, so AVG(ship_days) leaves them out.

📌 `employees.reports_to` is empty in 1 row(s): those rows have no `employees` row, so an INNER JOIN would drop them.

## 6. Date ranges and partial periods

*2 · Validate*

🔍 **What we found before:** `orders.order_date` runs from 1996-07-04 to 1998-05-06; 1998-Q2 is only partly covered.  
💡 **Why it matters:** An extract that stops mid-period makes the last period look like a collapse, and impossible dates (shipped before ordered, years in the future) point at loading errors. Both must be caught before any trend is reported.  
🎯 **What we're going to do:** We check the earliest and latest date of every date column and count rows per period, so that partial periods are labelled instead of misread.

🛠 **What this SQL does:** MIN and MAX per date column; then rows per period with the first and last date seen in each, so that the check below can tell how much of each period the data covers; then shipping dates that come before the order or more than a year after it.

<details><summary>MySQL 8</summary>

```sql
-- Why: We check the earliest and latest date of every date column and count rows per period, so
--      that partial periods are labelled instead of misread.

-- 1) Earliest and latest value of every date column
SELECT 'employees' AS table_name, 'birth_date' AS column_name,
       MIN(birth_date) AS earliest, MAX(birth_date) AS latest,
       COUNT(*) - COUNT(birth_date) AS null_rows
FROM   employees
UNION ALL
SELECT 'employees' AS table_name, 'hire_date' AS column_name,
       MIN(hire_date) AS earliest, MAX(hire_date) AS latest,
       COUNT(*) - COUNT(hire_date) AS null_rows
FROM   employees
UNION ALL
SELECT 'orders' AS table_name, 'order_date' AS column_name,
       MIN(order_date) AS earliest, MAX(order_date) AS latest,
       COUNT(*) - COUNT(order_date) AS null_rows
FROM   orders
UNION ALL
SELECT 'orders' AS table_name, 'required_date' AS column_name,
       MIN(required_date) AS earliest, MAX(required_date) AS latest,
       COUNT(*) - COUNT(required_date) AS null_rows
FROM   orders
UNION ALL
SELECT 'orders' AS table_name, 'shipped_date' AS column_name,
       MIN(shipped_date) AS earliest, MAX(shipped_date) AS latest,
       COUNT(*) - COUNT(shipped_date) AS null_rows
FROM   orders;

-- 2) Rows per quarter, with the first and last date seen
SELECT YEAR(order_date) AS order_year,
       QUARTER(order_date) AS quarter_number,
       COUNT(*) AS row_count,
       MIN(order_date) AS first_date,
       MAX(order_date) AS last_date
FROM   orders
WHERE  order_date IS NOT NULL
GROUP  BY YEAR(order_date), QUARTER(order_date)
ORDER  BY order_year, quarter_number;

-- 3) Is `shipped_date` ever before `order_date`, or implausibly late?
SELECT COUNT(*) AS row_count,
       COUNT(shipped_date) AS with_shipped_date,
       SUM(CASE WHEN shipped_date < order_date THEN 1 ELSE 0 END) AS before_order_date,
       SUM(CASE WHEN DATEDIFF(shipped_date, order_date) > 365 THEN 1 ELSE 0 END) AS over_a_year,
       MIN(DATEDIFF(shipped_date, order_date)) AS min_days,
       MAX(DATEDIFF(shipped_date, order_date)) AS max_days
FROM   orders;
```
</details>

<details><summary>PostgreSQL</summary>

```sql
-- Why: We check the earliest and latest date of every date column and count rows per period, so
--      that partial periods are labelled instead of misread.

-- 1) Earliest and latest value of every date column
SELECT 'employees' AS table_name, 'birth_date' AS column_name,
       MIN(birth_date) AS earliest, MAX(birth_date) AS latest,
       COUNT(*) - COUNT(birth_date) AS null_rows
FROM   employees
UNION ALL
SELECT 'employees' AS table_name, 'hire_date' AS column_name,
       MIN(hire_date) AS earliest, MAX(hire_date) AS latest,
       COUNT(*) - COUNT(hire_date) AS null_rows
FROM   employees
UNION ALL
SELECT 'orders' AS table_name, 'order_date' AS column_name,
       MIN(order_date) AS earliest, MAX(order_date) AS latest,
       COUNT(*) - COUNT(order_date) AS null_rows
FROM   orders
UNION ALL
SELECT 'orders' AS table_name, 'required_date' AS column_name,
       MIN(required_date) AS earliest, MAX(required_date) AS latest,
       COUNT(*) - COUNT(required_date) AS null_rows
FROM   orders
UNION ALL
SELECT 'orders' AS table_name, 'shipped_date' AS column_name,
       MIN(shipped_date) AS earliest, MAX(shipped_date) AS latest,
       COUNT(*) - COUNT(shipped_date) AS null_rows
FROM   orders;

-- 2) Rows per quarter, with the first and last date seen
SELECT EXTRACT(YEAR FROM order_date)::int AS order_year,
       EXTRACT(QUARTER FROM order_date)::int AS quarter_number,
       COUNT(*) AS row_count,
       MIN(order_date) AS first_date,
       MAX(order_date) AS last_date
FROM   orders
WHERE  order_date IS NOT NULL
GROUP  BY EXTRACT(YEAR FROM order_date)::int, EXTRACT(QUARTER FROM order_date)::int
ORDER  BY order_year, quarter_number;

-- 3) Is `shipped_date` ever before `order_date`, or implausibly late?
SELECT COUNT(*) AS row_count,
       COUNT(shipped_date) AS with_shipped_date,
       SUM(CASE WHEN shipped_date < order_date THEN 1 ELSE 0 END) AS before_order_date,
       SUM(CASE WHEN shipped_date - order_date > 365 THEN 1 ELSE 0 END) AS over_a_year,
       MIN(shipped_date - order_date) AS min_days,
       MAX(shipped_date - order_date) AS max_days
FROM   orders;
```
</details>

In [11]:
outputs = {}
# Earliest and latest value of every date column
outputs['date_ranges'] = sql(r"""
SELECT 'employees' AS table_name, 'birth_date' AS column_name,
       MIN(birth_date) AS earliest, MAX(birth_date) AS latest,
       COUNT(*) - COUNT(birth_date) AS null_rows
FROM   employees
UNION ALL
SELECT 'employees' AS table_name, 'hire_date' AS column_name,
       MIN(hire_date) AS earliest, MAX(hire_date) AS latest,
       COUNT(*) - COUNT(hire_date) AS null_rows
FROM   employees
UNION ALL
SELECT 'orders' AS table_name, 'order_date' AS column_name,
       MIN(order_date) AS earliest, MAX(order_date) AS latest,
       COUNT(*) - COUNT(order_date) AS null_rows
FROM   orders
UNION ALL
SELECT 'orders' AS table_name, 'required_date' AS column_name,
       MIN(required_date) AS earliest, MAX(required_date) AS latest,
       COUNT(*) - COUNT(required_date) AS null_rows
FROM   orders
UNION ALL
SELECT 'orders' AS table_name, 'shipped_date' AS column_name,
       MIN(shipped_date) AS earliest, MAX(shipped_date) AS latest,
       COUNT(*) - COUNT(shipped_date) AS null_rows
FROM   orders
""")
display(outputs['date_ranges'].head(20))
# Rows per quarter, with the first and last date seen
outputs['periods'] = sql(r"""
SELECT EXTRACT(YEAR FROM order_date)::int AS order_year,
       EXTRACT(QUARTER FROM order_date)::int AS quarter_number,
       COUNT(*) AS row_count,
       MIN(order_date) AS first_date,
       MAX(order_date) AS last_date
FROM   orders
WHERE  order_date IS NOT NULL
GROUP  BY EXTRACT(YEAR FROM order_date)::int, EXTRACT(QUARTER FROM order_date)::int
ORDER  BY order_year, quarter_number
""")
display(outputs['periods'].head(20))
# Is `shipped_date` ever before `order_date`, or implausibly late?
outputs['lead_sanity'] = sql(r"""
SELECT COUNT(*) AS row_count,
       COUNT(shipped_date) AS with_shipped_date,
       SUM(CASE WHEN shipped_date < order_date THEN 1 ELSE 0 END) AS before_order_date,
       SUM(CASE WHEN shipped_date - order_date > 365 THEN 1 ELSE 0 END) AS over_a_year,
       MIN(shipped_date - order_date) AS min_days,
       MAX(shipped_date - order_date) AS max_days
FROM   orders
""")
display(outputs['lead_sanity'].head(20))
result = list(outputs.values())[-1] if outputs else pd.DataFrame()
# 📌 What we found
import datetime as _dt
ranges = outputs["date_ranges"]
for row in ranges.itertuples():
    metrics.setdefault("date_ranges", {})[f"{row.table_name}.{row.column_name}"] = (str(row.earliest)[:10], str(row.latest)[:10])
if "periods" not in outputs:
    note("Date columns: " + "; ".join(f"`{r.table_name}.{r.column_name}` {str(r.earliest)[:10]} to {str(r.latest)[:10]}"
         + (f" ({int(r.null_rows):,} empty)" if r.null_rows else "") for r in ranges.itertuples()) + ".")
if "periods" in outputs:
    per = outputs["periods"]
    unit = 'quarter'
    starts, ends = [], []
    for row in per.itertuples(index=False):
        year, number = int(row[0]), (int(row[1]) if unit != "year" else 1)
        first_month = {"quarter": 3 * number - 2, "month": number, "year": 1}[unit]
        start = _dt.date(year, first_month, 1)
        end = (pd.Timestamp(start) + {"quarter": pd.offsets.QuarterEnd(0), "month": pd.offsets.MonthEnd(0),
                                      "year": pd.offsets.YearEnd(0)}[unit]).date()
        starts.append(start); ends.append(end)
    per = per.assign(period_start=starts, period_end=ends)
    first_seen = pd.to_datetime(per["first_date"]).dt.date
    last_seen = pd.to_datetime(per["last_date"]).dt.date
    covered = [((min(l, e) - max(f, s)).days + 1) / ((e - s).days + 1) if i in (0, len(per) - 1) else 1.0
               for i, (f, l, s, e) in enumerate(zip(first_seen, last_seen, per["period_start"], per["period_end"]))]
    per = per.assign(coverage=covered)
    label = (lambda r: f"{int(r.iloc[0])}-Q{int(r.iloc[1])}") if unit == "quarter" else \
            (lambda r: f"{int(r.iloc[0])}-{int(r.iloc[1]):02d}") if unit == "month" else (lambda r: str(int(r.iloc[0])))
    partial = [label(r) for _, r in per.iterrows() if r["coverage"] < 0.9]
    metrics["partial_periods"] = partial
    metrics["periods_seen"] = len(per)
    per_year = {"quarter": 4, "month": 12, "year": 1}[unit]
    index = [int(r.iloc[0]) * per_year + (int(r.iloc[1]) - 1 if unit != "year" else 0) for _, r in per.iterrows()]
    missing = sorted(set(range(min(index), max(index) + 1)) - set(index)) if index else []
    missing = [f"{i // 4}-Q{i % 4 + 1}" if unit == "quarter" else f"{i // 12}-{i % 12 + 1:02d}" if unit == "month" else str(i)
               for i in missing]
    metrics["missing_periods"] = missing
    if missing:
        note(f"⚠️ **No rows at all in {', '.join(missing[:6])}**. A gap is either a real stop or a loading problem; "
             f"period-over-period changes are only computed between neighbouring {unit}s.")
    span = f"{str(ranges['earliest'].min())[:10]} to {str(ranges['latest'].max())[:10]}"
    note(f"{len(per)} {unit}s of data. " + (f"**Partial:** {', '.join(partial)} (less than {0.9:.0%} of its days covered), so "
         "it is labelled and left out of period-over-period verdicts." if partial else "The first and last periods are fully covered."))
if "lead_sanity" in outputs:
    row = outputs["lead_sanity"].iloc[0]
    metrics["negative_lead"] = int(row.iloc[2] or 0)
    end_col, start_col = 'shipped_date', 'order_date'
    note(f"{int(row.iloc[2] or 0):,} row(s) where `{end_col}` is before `{start_col}`, {int(row.iloc[3] or 0):,} more than a year later; "
         f"lead times run from {row.min_days} to {row.max_days} days.")
for label in [k for k in outputs if k.startswith("part_")]:
    row = outputs[label].iloc[0]
    metrics["date_part_mismatches"] = metrics.get("date_part_mismatches", 0) + int(row.mismatched or 0)
    note(f"{label[5:]}: {int(row.mismatched or 0):,} row(s) disagree with the date they were derived from.")

,table_name,column_name,earliest,latest,null_rows
0,employees,birth_date,1937-09-19,1966-01-27,0
1,employees,hire_date,1992-04-01,1994-11-15,0
2,orders,order_date,1996-07-04,1998-05-06,0
3,orders,required_date,1996-07-24,1998-06-11,0
4,orders,shipped_date,1996-07-10,1998-05-06,21


,order_year,quarter_number,row_count,first_date,last_date
0,1996,3,70,1996-07-04,1996-09-30
1,1996,4,82,1996-10-01,1996-12-31
2,1997,1,92,1997-01-01,1997-03-31
3,1997,2,93,1997-04-01,1997-06-30
4,1997,3,103,1997-07-01,1997-09-30
5,1997,4,120,1997-10-01,1997-12-31
6,1998,1,182,1998-01-01,1998-03-31
7,1998,2,88,1998-04-01,1998-05-06


,row_count,with_shipped_date,before_order_date,over_a_year,min_days,max_days
0,830,809,0.0,0.0,1,37


📌 8 quarters of data. **Partial:** 1998-Q2 (less than 90% of its days covered), so it is labelled and left out of period-over-period verdicts.

📌 0 row(s) where `shipped_date` is before `order_date`, 0 more than a year later; lead times run from 1.0 to 37.0 days.

## 7. Find orphans

*2 · Validate*

🔍 **What we found before:** 7 trusted relationship(s); profiling counted 0 orphan child row(s) in total.  
💡 **Why it matters:** An orphan (a child whose parent is missing) vanishes from every INNER JOIN, so revenue quietly drops. Parents without children are not errors, but they matter for rates: a customer who never ordered still counts as a customer.  
🎯 **What we're going to do:** We count orphans in both directions with an anti-join, so that every join later can be trusted or its losses explained.

🛠 **What this SQL does:** LEFT JOIN child to parent and keep the rows where the parent key came back NULL: those children have no parent. The second query uses NOT EXISTS to find parents that no child points at. Both are anti-joins; NOT EXISTS is safe when the child column has NULLs.

<details><summary>MySQL 8</summary>

```sql
-- Why: We count orphans in both directions with an anti-join, so that every join later can be
--      trusted or its losses explained.

-- 1) Orphans in both directions, for every relationship
SELECT 'order_details.order_id -> orders' AS relationship,
       'child rows with no parent' AS problem,
       COUNT(*) AS row_count
FROM   order_details c
LEFT   JOIN orders p ON p.order_id = c.order_id
WHERE  p.order_id IS NULL
  AND  c.order_id IS NOT NULL
UNION ALL
SELECT 'order_details.order_id -> orders',
       'parents with no child rows (information)',
       COUNT(*)
FROM   orders p
WHERE  NOT EXISTS (SELECT 1 FROM order_details c WHERE p.order_id = c.order_id)
UNION ALL
SELECT 'order_details.product_id -> products' AS relationship,
       'child rows with no parent' AS problem,
       COUNT(*) AS row_count
FROM   order_details c
LEFT   JOIN products p ON p.product_id = c.product_id
WHERE  p.product_id IS NULL
  AND  c.product_id IS NOT NULL
UNION ALL
SELECT 'order_details.product_id -> products',
       'parents with no child rows (information)',
       COUNT(*)
FROM   products p
WHERE  NOT EXISTS (SELECT 1 FROM order_details c WHERE p.product_id = c.product_id)
UNION ALL
SELECT 'orders.customer_id -> customers' AS relationship,
       'child rows with no parent' AS problem,
       COUNT(*) AS row_count
FROM   orders c
LEFT   JOIN customers p ON p.customer_id = c.customer_id
WHERE  p.customer_id IS NULL
  AND  c.customer_id IS NOT NULL
UNION ALL
SELECT 'orders.customer_id -> customers',
       'parents with no child rows (information)',
       COUNT(*)
FROM   customers p
WHERE  NOT EXISTS (SELECT 1 FROM orders c WHERE p.customer_id = c.customer_id)
UNION ALL
SELECT 'orders.employee_id -> employees' AS relationship,
       'child rows with no parent' AS problem,
       COUNT(*) AS row_count
FROM   orders c
LEFT   JOIN employees p ON p.employee_id = c.employee_id
WHERE  p.employee_id IS NULL
  AND  c.employee_id IS NOT NULL
UNION ALL
SELECT 'orders.employee_id -> employees',
       'parents with no child rows (information)',
       COUNT(*)
FROM   employees p
WHERE  NOT EXISTS (SELECT 1 FROM orders c WHERE p.employee_id = c.employee_id)
UNION ALL
SELECT 'products.supplier_id -> suppliers' AS relationship,
       'child rows with no parent' AS problem,
       COUNT(*) AS row_count
FROM   products c
LEFT   JOIN suppliers p ON p.supplier_id = c.supplier_id
WHERE  p.supplier_id IS NULL
  AND  c.supplier_id IS NOT NULL
UNION ALL
SELECT 'products.supplier_id -> suppliers',
       'parents with no child rows (information)',
       COUNT(*)
FROM   suppliers p
WHERE  NOT EXISTS (SELECT 1 FROM products c WHERE p.supplier_id = c.supplier_id)
UNION ALL
SELECT 'products.category_id -> categories' AS relationship,
       'child rows with no parent' AS problem,
       COUNT(*) AS row_count
FROM   products c
LEFT   JOIN categories p ON p.category_id = c.category_id
WHERE  p.category_id IS NULL
  AND  c.category_id IS NOT NULL
UNION ALL
SELECT 'products.category_id -> categories',
       'parents with no child rows (information)',
       COUNT(*)
FROM   categories p
WHERE  NOT EXISTS (SELECT 1 FROM products c WHERE p.category_id = c.category_id)
UNION ALL
SELECT 'employees.reports_to -> employees' AS relationship,
       'child rows with no parent' AS problem,
       COUNT(*) AS row_count
FROM   employees c
LEFT   JOIN employees p ON p.employee_id = c.reports_to
WHERE  p.employee_id IS NULL
  AND  c.reports_to IS NOT NULL
UNION ALL
SELECT 'employees.reports_to -> employees',
       'parents with no child rows (information)',
       COUNT(*)
FROM   employees p
WHERE  NOT EXISTS (SELECT 1 FROM employees c WHERE p.employee_id = c.reports_to);

-- 2) `customers` rows that no `orders` row points at
SELECT p.customer_id, p.company_name
FROM   customers p
WHERE  NOT EXISTS (SELECT 1 FROM orders c WHERE p.customer_id = c.customer_id)
ORDER  BY 1
LIMIT  20;
```
</details>

<details><summary>PostgreSQL</summary>

```sql
-- Why: We count orphans in both directions with an anti-join, so that every join later can be
--      trusted or its losses explained.

-- 1) Orphans in both directions, for every relationship
SELECT 'order_details.order_id -> orders' AS relationship,
       'child rows with no parent' AS problem,
       COUNT(*) AS row_count
FROM   order_details c
LEFT   JOIN orders p ON p.order_id = c.order_id
WHERE  p.order_id IS NULL
  AND  c.order_id IS NOT NULL
UNION ALL
SELECT 'order_details.order_id -> orders',
       'parents with no child rows (information)',
       COUNT(*)
FROM   orders p
WHERE  NOT EXISTS (SELECT 1 FROM order_details c WHERE p.order_id = c.order_id)
UNION ALL
SELECT 'order_details.product_id -> products' AS relationship,
       'child rows with no parent' AS problem,
       COUNT(*) AS row_count
FROM   order_details c
LEFT   JOIN products p ON p.product_id = c.product_id
WHERE  p.product_id IS NULL
  AND  c.product_id IS NOT NULL
UNION ALL
SELECT 'order_details.product_id -> products',
       'parents with no child rows (information)',
       COUNT(*)
FROM   products p
WHERE  NOT EXISTS (SELECT 1 FROM order_details c WHERE p.product_id = c.product_id)
UNION ALL
SELECT 'orders.customer_id -> customers' AS relationship,
       'child rows with no parent' AS problem,
       COUNT(*) AS row_count
FROM   orders c
LEFT   JOIN customers p ON p.customer_id = c.customer_id
WHERE  p.customer_id IS NULL
  AND  c.customer_id IS NOT NULL
UNION ALL
SELECT 'orders.customer_id -> customers',
       'parents with no child rows (information)',
       COUNT(*)
FROM   customers p
WHERE  NOT EXISTS (SELECT 1 FROM orders c WHERE p.customer_id = c.customer_id)
UNION ALL
SELECT 'orders.employee_id -> employees' AS relationship,
       'child rows with no parent' AS problem,
       COUNT(*) AS row_count
FROM   orders c
LEFT   JOIN employees p ON p.employee_id = c.employee_id
WHERE  p.employee_id IS NULL
  AND  c.employee_id IS NOT NULL
UNION ALL
SELECT 'orders.employee_id -> employees',
       'parents with no child rows (information)',
       COUNT(*)
FROM   employees p
WHERE  NOT EXISTS (SELECT 1 FROM orders c WHERE p.employee_id = c.employee_id)
UNION ALL
SELECT 'products.supplier_id -> suppliers' AS relationship,
       'child rows with no parent' AS problem,
       COUNT(*) AS row_count
FROM   products c
LEFT   JOIN suppliers p ON p.supplier_id = c.supplier_id
WHERE  p.supplier_id IS NULL
  AND  c.supplier_id IS NOT NULL
UNION ALL
SELECT 'products.supplier_id -> suppliers',
       'parents with no child rows (information)',
       COUNT(*)
FROM   suppliers p
WHERE  NOT EXISTS (SELECT 1 FROM products c WHERE p.supplier_id = c.supplier_id)
UNION ALL
SELECT 'products.category_id -> categories' AS relationship,
       'child rows with no parent' AS problem,
       COUNT(*) AS row_count
FROM   products c
LEFT   JOIN categories p ON p.category_id = c.category_id
WHERE  p.category_id IS NULL
  AND  c.category_id IS NOT NULL
UNION ALL
SELECT 'products.category_id -> categories',
       'parents with no child rows (information)',
       COUNT(*)
FROM   categories p
WHERE  NOT EXISTS (SELECT 1 FROM products c WHERE p.category_id = c.category_id)
UNION ALL
SELECT 'employees.reports_to -> employees' AS relationship,
       'child rows with no parent' AS problem,
       COUNT(*) AS row_count
FROM   employees c
LEFT   JOIN employees p ON p.employee_id = c.reports_to
WHERE  p.employee_id IS NULL
  AND  c.reports_to IS NOT NULL
UNION ALL
SELECT 'employees.reports_to -> employees',
       'parents with no child rows (information)',
       COUNT(*)
FROM   employees p
WHERE  NOT EXISTS (SELECT 1 FROM employees c WHERE p.employee_id = c.reports_to);

-- 2) `customers` rows that no `orders` row points at
SELECT p.customer_id, p.company_name
FROM   customers p
WHERE  NOT EXISTS (SELECT 1 FROM orders c WHERE p.customer_id = c.customer_id)
ORDER  BY 1
LIMIT  20;
```
</details>

In [12]:
outputs = {}
# Orphans in both directions, for every relationship
outputs['orphans'] = sql(r"""
SELECT 'order_details.order_id -> orders' AS relationship,
       'child rows with no parent' AS problem,
       COUNT(*) AS row_count
FROM   order_details c
LEFT   JOIN orders p ON p.order_id = c.order_id
WHERE  p.order_id IS NULL
  AND  c.order_id IS NOT NULL
UNION ALL
SELECT 'order_details.order_id -> orders',
       'parents with no child rows (information)',
       COUNT(*)
FROM   orders p
WHERE  NOT EXISTS (SELECT 1 FROM order_details c WHERE p.order_id = c.order_id)
UNION ALL
SELECT 'order_details.product_id -> products' AS relationship,
       'child rows with no parent' AS problem,
       COUNT(*) AS row_count
FROM   order_details c
LEFT   JOIN products p ON p.product_id = c.product_id
WHERE  p.product_id IS NULL
  AND  c.product_id IS NOT NULL
UNION ALL
SELECT 'order_details.product_id -> products',
       'parents with no child rows (information)',
       COUNT(*)
FROM   products p
WHERE  NOT EXISTS (SELECT 1 FROM order_details c WHERE p.product_id = c.product_id)
UNION ALL
SELECT 'orders.customer_id -> customers' AS relationship,
       'child rows with no parent' AS problem,
       COUNT(*) AS row_count
FROM   orders c
LEFT   JOIN customers p ON p.customer_id = c.customer_id
WHERE  p.customer_id IS NULL
  AND  c.customer_id IS NOT NULL
UNION ALL
SELECT 'orders.customer_id -> customers',
       'parents with no child rows (information)',
       COUNT(*)
FROM   customers p
WHERE  NOT EXISTS (SELECT 1 FROM orders c WHERE p.customer_id = c.customer_id)
UNION ALL
SELECT 'orders.employee_id -> employees' AS relationship,
       'child rows with no parent' AS problem,
       COUNT(*) AS row_count
FROM   orders c
LEFT   JOIN employees p ON p.employee_id = c.employee_id
WHERE  p.employee_id IS NULL
  AND  c.employee_id IS NOT NULL
UNION ALL
SELECT 'orders.employee_id -> employees',
       'parents with no child rows (information)',
       COUNT(*)
FROM   employees p
WHERE  NOT EXISTS (SELECT 1 FROM orders c WHERE p.employee_id = c.employee_id)
UNION ALL
SELECT 'products.supplier_id -> suppliers' AS relationship,
       'child rows with no parent' AS problem,
       COUNT(*) AS row_count
FROM   products c
LEFT   JOIN suppliers p ON p.supplier_id = c.supplier_id
WHERE  p.supplier_id IS NULL
  AND  c.supplier_id IS NOT NULL
UNION ALL
SELECT 'products.supplier_id -> suppliers',
       'parents with no child rows (information)',
       COUNT(*)
FROM   suppliers p
WHERE  NOT EXISTS (SELECT 1 FROM products c WHERE p.supplier_id = c.supplier_id)
UNION ALL
SELECT 'products.category_id -> categories' AS relationship,
       'child rows with no parent' AS problem,
       COUNT(*) AS row_count
FROM   products c
LEFT   JOIN categories p ON p.category_id = c.category_id
WHERE  p.category_id IS NULL
  AND  c.category_id IS NOT NULL
UNION ALL
SELECT 'products.category_id -> categories',
       'parents with no child rows (information)',
       COUNT(*)
FROM   categories p
WHERE  NOT EXISTS (SELECT 1 FROM products c WHERE p.category_id = c.category_id)
UNION ALL
SELECT 'employees.reports_to -> employees' AS relationship,
       'child rows with no parent' AS problem,
       COUNT(*) AS row_count
FROM   employees c
LEFT   JOIN employees p ON p.employee_id = c.reports_to
WHERE  p.employee_id IS NULL
  AND  c.reports_to IS NOT NULL
UNION ALL
SELECT 'employees.reports_to -> employees',
       'parents with no child rows (information)',
       COUNT(*)
FROM   employees p
WHERE  NOT EXISTS (SELECT 1 FROM employees c WHERE p.employee_id = c.reports_to)
""")
display(outputs['orphans'].head(20))
# `customers` rows that no `orders` row points at
outputs['childless_examples'] = sql(r"""
SELECT p.customer_id, p.company_name
FROM   customers p
WHERE  NOT EXISTS (SELECT 1 FROM orders c WHERE p.customer_id = c.customer_id)
ORDER  BY 1
LIMIT  20
""")
display(outputs['childless_examples'].head(20))
result = list(outputs.values())[-1] if outputs else pd.DataFrame()
# 📌 What we found
orphans = outputs["orphans"]
bad = orphans[(orphans["problem"] == "child rows with no parent") & (orphans["row_count"] > 0)]
lonely = orphans[(orphans["problem"] != "child rows with no parent") & (orphans["row_count"] > 0)]
metrics["orphans"] = {r.relationship: int(r.row_count) for r in bad.itertuples()}
metrics["childless"] = {r.relationship: int(r.row_count) for r in lonely.itertuples()}
if bad.empty:
    note("No orphans: every foreign key value points at an existing row, so no row will vanish from an inner join.")
for r in bad.itertuples():
    note(f"`{r.relationship}`: **{int(r.row_count):,} orphan row(s)**. They are kept (LEFT JOIN, labelled 'Unknown') so that no revenue is lost.")
for r in lonely.itertuples():
    note(f"`{r.relationship}`: {int(r.row_count):,} parent row(s) have no child rows.")
if "childless_examples" in outputs:
    ex = outputs["childless_examples"]
    note("For example: " + ", ".join(" / ".join(str(v) for v in row) for row in ex.head(5).itertuples(index=False)) + ".")

,relationship,problem,row_count
0,order_details.order_id -> orders,child rows with no parent,0
1,order_details.order_id -> orders,parents with no child rows (information),0
2,order_details.product_id -> products,child rows with no parent,0
3,order_details.product_id -> products,parents with no child rows (information),0
4,orders.customer_id -> customers,child rows with no parent,0
5,orders.customer_id -> customers,parents with no child rows (information),2
6,orders.employee_id -> employees,child rows with no parent,0
7,orders.employee_id -> employees,parents with no child rows (information),0
8,products.supplier_id -> suppliers,child rows with no parent,0
9,products.supplier_id -> suppliers,parents with no child rows (information),0


,customer_id,company_name
0,FISSA,FISSA Fabrica Inter. Salchichas S.A.
1,PARIS,Paris spécialités


📌 No orphans: every foreign key value points at an existing row, so no row will vanish from an inner join.

📌 `orders.customer_id -> customers`: 2 parent row(s) have no child rows.

📌 `employees.reports_to -> employees`: 7 parent row(s) have no child rows.

📌 For example: FISSA / FISSA Fabrica Inter. Salchichas S.A., PARIS / Paris spécialités.

## 8. Build the analysis views

*3 · Build with CTEs*

🔍 **What we found before:** The revenue formula (unit_price × quantity × (1 − discount)), the order grain and the tables to join (`categories`, `customers`, `employees`, `orders` and 1 more) are known.  
💡 **Why it matters:** Writing every KPI from raw tables repeats the same joins and invites the same mistakes each time. One tested view per grain fixes them once.  
🎯 **What we're going to do:** We build two views, `order_facts` (one row per order) and `line_facts` (one row per line), from small CTEs that each do one job, so that every later query starts from checked, business-named columns.

🛠 **What this SQL does:** `line_revenue` computes revenue per line; `order_totals` rolls lines up to one row per order; the final SELECT joins the order table and dimension tables with LEFT JOIN (so nothing is dropped; a missing match becomes 'Unknown'), and adds the period columns, ship_days and shipped_late. `line_facts` keeps one row per line for product questions.

<details><summary>MySQL 8</summary>

```sql
-- Why: We build two views, order_facts (one row per order) and line_facts (one row per line), from
--      small CTEs that each do one job, so that every later query starts from checked, business-
--      named columns.

-- 1) View `order_facts`: one row per order
-- Grain: one row per order (`order_id`). Lines are rolled up BEFORE joining order-level tables, so nothing is repeated.
CREATE OR REPLACE VIEW order_facts AS
WITH line_revenue AS (                -- 1) one row per order_details line, with its revenue
    SELECT od.order_id,
           od.unit_price * od.quantity * (1 - COALESCE(od.discount, 0)) AS revenue,
           od.quantity AS units
    FROM   order_details od
),
order_totals AS (                -- 2) one row per order: roll the lines up BEFORE joining order-level tables
    SELECT order_id,
           SUM(revenue) AS revenue,
           SUM(units) AS units,
           COUNT(*) AS line_count
    FROM   line_revenue
    GROUP  BY order_id
)
SELECT t.order_id,
       o.customer_id,
       o.order_date,
       YEAR(o.order_date) AS order_year,
       QUARTER(o.order_date) AS quarter_number,
       t.revenue,
       t.units,
       t.line_count,
       o.freight,
       DATEDIFF(o.shipped_date, o.order_date) AS ship_days,
       CASE WHEN o.shipped_date IS NULL OR o.required_date IS NULL THEN NULL
            WHEN o.shipped_date > o.required_date THEN 1
            ELSE 0
       END AS shipped_late,
       COALESCE(c.country, 'Unknown') AS customer_country,
       COALESCE(e.last_name, 'Unknown') AS employee_last_name
FROM   order_totals t
LEFT   JOIN orders o ON o.order_id = t.order_id
LEFT   JOIN customers c ON c.customer_id = o.customer_id
LEFT   JOIN employees e ON e.employee_id = o.employee_id;

-- 2) View `line_facts`: one row per line
-- Grain: one row per order_details line. Use it for product-level questions; use order_facts for order-level KPIs.
CREATE OR REPLACE VIEW line_facts AS
SELECT od.order_id,
       od.product_id,
       o.customer_id,
       o.order_date,
       YEAR(o.order_date) AS order_year,
       QUARTER(o.order_date) AS quarter_number,
       od.unit_price * od.quantity * (1 - COALESCE(od.discount, 0)) AS revenue,
       od.quantity AS units,
       COALESCE(p.product_name, 'Unknown') AS product_name,
       COALESCE(cat.category_name, 'Unknown') AS category_name
FROM   order_details od
LEFT   JOIN orders o ON o.order_id = od.order_id
LEFT   JOIN products p ON p.product_id = od.product_id
LEFT   JOIN categories cat ON cat.category_id = p.category_id;

-- 3) The first five rows of `order_facts`
SELECT *
FROM   order_facts
ORDER  BY order_date, order_id
LIMIT  5;
```
</details>

<details><summary>PostgreSQL</summary>

```sql
-- Why: We build two views, order_facts (one row per order) and line_facts (one row per line), from
--      small CTEs that each do one job, so that every later query starts from checked, business-
--      named columns.

-- 1) View `order_facts`: one row per order
-- Grain: one row per order (`order_id`). Lines are rolled up BEFORE joining order-level tables, so nothing is repeated.
CREATE OR REPLACE VIEW order_facts AS
WITH line_revenue AS (                -- 1) one row per order_details line, with its revenue
    SELECT od.order_id,
           od.unit_price * od.quantity * (1 - COALESCE(od.discount, 0)) AS revenue,
           od.quantity AS units
    FROM   order_details od
),
order_totals AS (                -- 2) one row per order: roll the lines up BEFORE joining order-level tables
    SELECT order_id,
           SUM(revenue) AS revenue,
           SUM(units) AS units,
           COUNT(*) AS line_count
    FROM   line_revenue
    GROUP  BY order_id
)
SELECT t.order_id,
       o.customer_id,
       o.order_date,
       EXTRACT(YEAR FROM o.order_date)::int AS order_year,
       EXTRACT(QUARTER FROM o.order_date)::int AS quarter_number,
       t.revenue,
       t.units,
       t.line_count,
       o.freight,
       o.shipped_date - o.order_date AS ship_days,
       CASE WHEN o.shipped_date IS NULL OR o.required_date IS NULL THEN NULL
            WHEN o.shipped_date > o.required_date THEN 1
            ELSE 0
       END AS shipped_late,
       COALESCE(c.country, 'Unknown') AS customer_country,
       COALESCE(e.last_name, 'Unknown') AS employee_last_name
FROM   order_totals t
LEFT   JOIN orders o ON o.order_id = t.order_id
LEFT   JOIN customers c ON c.customer_id = o.customer_id
LEFT   JOIN employees e ON e.employee_id = o.employee_id;

-- 2) View `line_facts`: one row per line
-- Grain: one row per order_details line. Use it for product-level questions; use order_facts for order-level KPIs.
CREATE OR REPLACE VIEW line_facts AS
SELECT od.order_id,
       od.product_id,
       o.customer_id,
       o.order_date,
       EXTRACT(YEAR FROM o.order_date)::int AS order_year,
       EXTRACT(QUARTER FROM o.order_date)::int AS quarter_number,
       od.unit_price * od.quantity * (1 - COALESCE(od.discount, 0)) AS revenue,
       od.quantity AS units,
       COALESCE(p.product_name, 'Unknown') AS product_name,
       COALESCE(cat.category_name, 'Unknown') AS category_name
FROM   order_details od
LEFT   JOIN orders o ON o.order_id = od.order_id
LEFT   JOIN products p ON p.product_id = od.product_id
LEFT   JOIN categories cat ON cat.category_id = p.category_id;

-- 3) The first five rows of `order_facts`
SELECT *
FROM   order_facts
ORDER  BY order_date, order_id
LIMIT  5;
```
</details>

In [13]:
outputs = {}
# View `order_facts`: one row per order
outputs['order_facts'] = sql(r"""
-- Grain: one row per order (`order_id`). Lines are rolled up BEFORE joining order-level tables, so nothing is repeated.
CREATE OR REPLACE VIEW order_facts AS
WITH line_revenue AS (                -- 1) one row per order_details line, with its revenue
    SELECT od.order_id,
           od.unit_price * od.quantity * (1 - COALESCE(od.discount, 0)) AS revenue,
           od.quantity AS units
    FROM   order_details od
),
order_totals AS (                -- 2) one row per order: roll the lines up BEFORE joining order-level tables
    SELECT order_id,
           SUM(revenue) AS revenue,
           SUM(units) AS units,
           COUNT(*) AS line_count
    FROM   line_revenue
    GROUP  BY order_id
)
SELECT t.order_id,
       o.customer_id,
       o.order_date,
       EXTRACT(YEAR FROM o.order_date)::int AS order_year,
       EXTRACT(QUARTER FROM o.order_date)::int AS quarter_number,
       t.revenue,
       t.units,
       t.line_count,
       o.freight,
       o.shipped_date - o.order_date AS ship_days,
       CASE WHEN o.shipped_date IS NULL OR o.required_date IS NULL THEN NULL
            WHEN o.shipped_date > o.required_date THEN 1
            ELSE 0
       END AS shipped_late,
       COALESCE(c.country, 'Unknown') AS customer_country,
       COALESCE(e.last_name, 'Unknown') AS employee_last_name
FROM   order_totals t
LEFT   JOIN orders o ON o.order_id = t.order_id
LEFT   JOIN customers c ON c.customer_id = o.customer_id
LEFT   JOIN employees e ON e.employee_id = o.employee_id
""")
display(outputs['order_facts'].head(20))
# View `line_facts`: one row per line
outputs['line_facts'] = sql(r"""
-- Grain: one row per order_details line. Use it for product-level questions; use order_facts for order-level KPIs.
CREATE OR REPLACE VIEW line_facts AS
SELECT od.order_id,
       od.product_id,
       o.customer_id,
       o.order_date,
       EXTRACT(YEAR FROM o.order_date)::int AS order_year,
       EXTRACT(QUARTER FROM o.order_date)::int AS quarter_number,
       od.unit_price * od.quantity * (1 - COALESCE(od.discount, 0)) AS revenue,
       od.quantity AS units,
       COALESCE(p.product_name, 'Unknown') AS product_name,
       COALESCE(cat.category_name, 'Unknown') AS category_name
FROM   order_details od
LEFT   JOIN orders o ON o.order_id = od.order_id
LEFT   JOIN products p ON p.product_id = od.product_id
LEFT   JOIN categories cat ON cat.category_id = p.category_id
""")
display(outputs['line_facts'].head(20))
# The first five rows of `order_facts`
outputs['preview'] = sql(r"""
SELECT *
FROM   order_facts
ORDER  BY order_date, order_id
LIMIT  5
""")
display(outputs['preview'].head(20))
result = list(outputs.values())[-1] if outputs else pd.DataFrame()
# 📌 What we found
preview = outputs["preview"]
note(f"`order_facts` has these columns: " + ", ".join(f"`{c}`" for c in preview.columns) + ".")

,Count


,Count


,order_id,customer_id,order_date,order_year,quarter_number,revenue,units,line_count,freight,ship_days,shipped_late,customer_country,employee_last_name
0,10248,VINET,1996-07-04,1996,3,440.00,27.0,3,32.38,12,0,France,Buchanan
1,10249,TOMSP,1996-07-05,1996,3,1863.40,49.0,2,11.61,5,0,Germany,Suyama
2,10250,HANAR,1996-07-08,1996,3,1552.60,60.0,3,65.83,4,0,Brazil,Peacock
3,10251,VICTE,1996-07-08,1996,3,654.06,41.0,3,41.34,7,0,France,Leverling
4,10252,SUPRD,1996-07-09,1996,3,3597.90,105.0,3,51.30,2,0,Belgium,Peacock


📌 `order_facts` has these columns: `order_id`, `customer_id`, `order_date`, `order_year`, `quarter_number`, `revenue`, `units`, `line_count`, `freight`, `ship_days`, `shipped_late`, `customer_country`, `employee_last_name`.

## 9. Reconcile the totals

*2 · Validate*

🔍 **What we found before:** The views are built; their totals have not been checked yet.  
💡 **Why it matters:** A join that drops orphans, a duplicate that slips through or a filter in the wrong place all change the total. Reconciling against the source is the only proof that the views neither lose nor invent revenue.  
🎯 **What we're going to do:** We compute rows, orders and revenue from the raw table, from each view, and independently in pandas from the uploaded file, so that every number shared later is traceable to the source.

🛠 **What this SQL does:** Three SELECTs stacked with UNION ALL compute the same totals three ways; they must agree to the cent. A second query splits revenue into gross and discount. The check below also recomputes revenue in pandas straight from the uploaded rows, outside SQL.

<details><summary>MySQL 8</summary>

```sql
-- Why: We compute rows, orders and revenue from the raw table, from each view, and independently in
--      pandas from the uploaded file, so that every number shared later is traceable to the source.

-- 1) The same totals computed from the raw table and from each view
SELECT 'order_details (raw table)' AS computed_from,
       COUNT(*) AS rows_read,
       COUNT(DISTINCT od.order_id) AS orders,
       ROUND(SUM(od.unit_price * od.quantity * (1 - COALESCE(od.discount, 0))), 2) AS revenue
FROM   order_details od
UNION ALL
SELECT 'order_facts (one row per order)',
       SUM(line_count),
       COUNT(*),
       ROUND(SUM(revenue), 2)
FROM   order_facts
UNION ALL
SELECT 'line_facts (one row per line)',
       COUNT(*),
       COUNT(DISTINCT order_id),
       ROUND(SUM(revenue), 2)
FROM   line_facts;

-- 2) Gross vs net revenue (the formula is net of discount)
SELECT ROUND(SUM(od.unit_price * od.quantity), 2) AS gross_revenue,
       ROUND(SUM(od.unit_price * od.quantity * (1 - COALESCE(od.discount, 0))), 2) AS net_revenue,
       ROUND(SUM(od.unit_price * od.quantity - od.unit_price * od.quantity * (1 - COALESCE(od.discount, 0))), 2) AS discount_given
FROM   order_details od;

-- 3) Is `order_details.unit_price` the same as the list price `products.unit_price`?
SELECT COUNT(*) AS line_rows,
       SUM(CASE WHEN od.unit_price <> p.unit_price THEN 1 ELSE 0 END) AS price_differs_from_list,
       ROUND(100.0 * SUM(CASE WHEN od.unit_price <> p.unit_price THEN 1 ELSE 0 END) / NULLIF(COUNT(*), 0), 1) AS differs_pct
FROM   order_details od
JOIN   products p ON p.product_id = od.product_id;
```
</details>

<details><summary>PostgreSQL</summary>

```sql
-- Why: We compute rows, orders and revenue from the raw table, from each view, and independently in
--      pandas from the uploaded file, so that every number shared later is traceable to the source.

-- 1) The same totals computed from the raw table and from each view
SELECT 'order_details (raw table)' AS computed_from,
       COUNT(*) AS rows_read,
       COUNT(DISTINCT od.order_id) AS orders,
       ROUND((SUM(od.unit_price * od.quantity * (1 - COALESCE(od.discount, 0))))::numeric, 2) AS revenue
FROM   order_details od
UNION ALL
SELECT 'order_facts (one row per order)',
       SUM(line_count),
       COUNT(*),
       ROUND((SUM(revenue))::numeric, 2)
FROM   order_facts
UNION ALL
SELECT 'line_facts (one row per line)',
       COUNT(*),
       COUNT(DISTINCT order_id),
       ROUND((SUM(revenue))::numeric, 2)
FROM   line_facts;

-- 2) Gross vs net revenue (the formula is net of discount)
SELECT ROUND((SUM(od.unit_price * od.quantity))::numeric, 2) AS gross_revenue,
       ROUND((SUM(od.unit_price * od.quantity * (1 - COALESCE(od.discount, 0))))::numeric, 2) AS net_revenue,
       ROUND((SUM(od.unit_price * od.quantity - od.unit_price * od.quantity * (1 - COALESCE(od.discount, 0))))::numeric, 2) AS discount_given
FROM   order_details od;

-- 3) Is `order_details.unit_price` the same as the list price `products.unit_price`?
SELECT COUNT(*) AS line_rows,
       SUM(CASE WHEN od.unit_price <> p.unit_price THEN 1 ELSE 0 END) AS price_differs_from_list,
       ROUND((100.0 * SUM(CASE WHEN od.unit_price <> p.unit_price THEN 1 ELSE 0 END) / NULLIF(COUNT(*), 0))::numeric, 1) AS differs_pct
FROM   order_details od
JOIN   products p ON p.product_id = od.product_id;
```
</details>

In [14]:
outputs = {}
# The same totals computed from the raw table and from each view
outputs['totals'] = sql(r"""
SELECT 'order_details (raw table)' AS computed_from,
       COUNT(*) AS rows_read,
       COUNT(DISTINCT od.order_id) AS orders,
       ROUND(SUM(od.unit_price * od.quantity * (1 - COALESCE(od.discount, 0))), 2) AS revenue
FROM   order_details od
UNION ALL
SELECT 'order_facts (one row per order)',
       SUM(line_count),
       COUNT(*),
       ROUND(SUM(revenue), 2)
FROM   order_facts
UNION ALL
SELECT 'line_facts (one row per line)',
       COUNT(*),
       COUNT(DISTINCT order_id),
       ROUND(SUM(revenue), 2)
FROM   line_facts
""")
display(outputs['totals'].head(20))
# Gross vs net revenue (the formula is net of discount)
outputs['gross_net'] = sql(r"""
SELECT ROUND(SUM(od.unit_price * od.quantity), 2) AS gross_revenue,
       ROUND(SUM(od.unit_price * od.quantity * (1 - COALESCE(od.discount, 0))), 2) AS net_revenue,
       ROUND(SUM(od.unit_price * od.quantity - od.unit_price * od.quantity * (1 - COALESCE(od.discount, 0))), 2) AS discount_given
FROM   order_details od
""")
display(outputs['gross_net'].head(20))
# Is `order_details.unit_price` the same as the list price `products.unit_price`?
outputs['list_price'] = sql(r"""
SELECT COUNT(*) AS line_rows,
       SUM(CASE WHEN od.unit_price <> p.unit_price THEN 1 ELSE 0 END) AS price_differs_from_list,
       ROUND(100.0 * SUM(CASE WHEN od.unit_price <> p.unit_price THEN 1 ELSE 0 END) / NULLIF(COUNT(*), 0), 1) AS differs_pct
FROM   order_details od
JOIN   products p ON p.product_id = od.product_id
""")
display(outputs['list_price'].head(20))
result = list(outputs.values())[-1] if outputs else pd.DataFrame()
# 📌 What we found
totals = outputs["totals"]
src = source['order_details']
source_revenue = float((src['unit_price'] * src['quantity'] * (1 - src['discount'].fillna(0))).sum())
measure = totals.columns[-1]                         # revenue, or row_count when there is no money column
sql_values = [float(v) for v in totals[measure]]
deduped = False                                # the raw table still holds the duplicates the views remove
compared = sql_values[1:] if deduped else sql_values
tolerance = max(0.01, 1e-9 * max(abs(v) for v in compared + [source_revenue]))  # floating-point sums on big data
agree = max(compared + [source_revenue]) - min(compared + [source_revenue]) <= tolerance
metrics["revenue_total"] = compared[0]
metrics["revenue_source"] = source_revenue
metrics["reconciled"] = bool(agree)
metrics["main_measure"] = "revenue" if measure == "revenue" else "orders"
what = "revenue" if measure == "revenue" else "the row count"
fmt = "{:,.2f}" if measure == "revenue" else "{:,.0f}"
if agree:
    note(f"**Reconciled:** {what} is **" + fmt.format(compared[0]) + "** from " + ("every view" if deduped else "the raw table, from every view")
         + ", and from the uploaded file (recomputed in pandas). Nothing was lost or double-counted.")
else:
    note("⚠️ The totals differ: " + "; ".join(f"{name}: " + fmt.format(value) for name, value in zip(totals["computed_from"], sql_values))
         + "; uploaded file: " + fmt.format(source_revenue) + ". Look for orphans or duplicated keys before trusting any KPI.")
if deduped:
    note("The raw table still contains the duplicated keys: " + fmt.format(sql_values[0] - compared[0])
         + f" of {what} disappears when they are removed, which is the point of de-duplicating.")
if "gross_net" in outputs:
    row = outputs["gross_net"].iloc[0]
    metrics["revenue_gross"] = float(row.gross_revenue)
    note(f"Gross revenue is {float(row.gross_revenue):,.2f}; discounts take {float(row.discount_given):,.2f} "
         f"({float(row.discount_given) / float(row.gross_revenue):.1%}), leaving net {float(row.net_revenue):,.2f}.")
if "list_price" in outputs:
    row = outputs["list_price"].iloc[0]
    metrics["price_differs_pct"] = float(row.differs_pct)
    note(f"In {float(row.differs_pct):.1f}% of lines the price paid differs from today's list price, which is why revenue uses the price on the line.")
if "order_conflicts" in outputs:
    n = int(outputs["order_conflicts"].iloc[0, 0])
    note(f"{n:,} order(s) have lines with different dates" + (": the view takes the earliest." if n else "."))

,computed_from,rows_read,orders,revenue
0,order_details (raw table),2155.0,830,1265793.04
1,order_facts (one row per order),2155.0,830,1265793.04
2,line_facts (one row per line),2155.0,830,1265793.04


,gross_revenue,net_revenue,discount_given
0,1354458.59,1265793.04,88665.55


,line_rows,price_differs_from_list,differs_pct
0,2155,658.0,30.5


📌 **Reconciled:** revenue is **1,265,793.04** from the raw table, from every view, and from the uploaded file (recomputed in pandas). Nothing was lost or double-counted.

📌 Gross revenue is 1,354,458.59; discounts take 88,665.55 (6.5%), leaving net 1,265,793.04.

📌 In 30.5% of lines the price paid differs from today's list price, which is why revenue uses the price on the line.

## 10. Headline KPIs

*4 · KPIs*

🔍 **What we found before:** `order_facts` has one row per order and reconciles to the source.  
💡 **Why it matters:** Headline KPIs answer 'how big is the business?' and become the baseline for every trend. Each must be computed at the right grain: average order value is revenue ÷ distinct orders, not the average of order lines.  
🎯 **What we're going to do:** We compute the headline KPIs in one query, so that every later breakdown can be compared with the overall figure.

🛠 **What this SQL does:** One CTE summarises order_facts (SUM, COUNT, AVG); another counts customers with two or more orders (GROUP BY … HAVING COUNT(*) >= 2). The final SELECT divides with 1.0 * … / NULLIF(…, 0) and 100.0 * …, so that ratios stay decimal and never divide by zero. The average *line* value is shown next to the AOV to make the grain difference visible.

<details><summary>MySQL 8</summary>

```sql
-- Why: We compute the headline KPIs in one query, so that every later breakdown can be compared
--      with the overall figure.

-- Headline KPIs
WITH order_summary AS (
    SELECT SUM(revenue) AS revenue,
           COUNT(*) AS orders,
           COUNT(DISTINCT customer_id) AS customers,
           SUM(units) AS units,
           AVG(ship_days) AS avg_ship_days,
           COUNT(ship_days) AS orders_with_ship_days,
           SUM(shipped_late) AS late_orders,
           COUNT(shipped_late) AS checked_orders
    FROM   order_facts
),
repeat_customers AS (
    SELECT COUNT(*) AS repeat_customers
    FROM   (SELECT customer_id
            FROM   order_facts
            WHERE  customer_id IS NOT NULL
            GROUP  BY customer_id
            HAVING COUNT(*) >= 2) r
),
line_summary AS (
    SELECT AVG(revenue) AS avg_line_value   -- NOT the average order value: shown for contrast
    FROM   line_facts
)
SELECT ROUND(revenue, 2) AS revenue,
       orders,
       ROUND(1.0 * revenue / NULLIF(orders, 0), 2) AS avg_order_value,
       ROUND(avg_line_value, 2) AS avg_line_value,
       customers,
       ROUND(1.0 * orders / NULLIF(customers, 0), 2) AS orders_per_customer,
       ROUND(100.0 * repeat_customers / NULLIF(customers, 0), 1) AS repeat_customer_pct,
       ROUND(1.0 * units / NULLIF(orders, 0), 2) AS units_per_order,
       ROUND(avg_ship_days, 2) AS avg_ship_days,
       orders - orders_with_ship_days AS orders_without_ship_days,
       ROUND(100.0 * late_orders / NULLIF(checked_orders, 0), 1) AS late_pct
FROM   order_summary
CROSS  JOIN repeat_customers
CROSS  JOIN line_summary;
```
</details>

<details><summary>PostgreSQL</summary>

```sql
-- Why: We compute the headline KPIs in one query, so that every later breakdown can be compared
--      with the overall figure.

-- Headline KPIs
WITH order_summary AS (
    SELECT SUM(revenue) AS revenue,
           COUNT(*) AS orders,
           COUNT(DISTINCT customer_id) AS customers,
           SUM(units) AS units,
           AVG(ship_days) AS avg_ship_days,
           COUNT(ship_days) AS orders_with_ship_days,
           SUM(shipped_late) AS late_orders,
           COUNT(shipped_late) AS checked_orders
    FROM   order_facts
),
repeat_customers AS (
    SELECT COUNT(*) AS repeat_customers
    FROM   (SELECT customer_id
            FROM   order_facts
            WHERE  customer_id IS NOT NULL
            GROUP  BY customer_id
            HAVING COUNT(*) >= 2) r
),
line_summary AS (
    SELECT AVG(revenue) AS avg_line_value   -- NOT the average order value: shown for contrast
    FROM   line_facts
)
SELECT ROUND((revenue)::numeric, 2) AS revenue,
       orders,
       ROUND((1.0 * revenue / NULLIF(orders, 0))::numeric, 2) AS avg_order_value,
       ROUND((avg_line_value)::numeric, 2) AS avg_line_value,
       customers,
       ROUND((1.0 * orders / NULLIF(customers, 0))::numeric, 2) AS orders_per_customer,
       ROUND((100.0 * repeat_customers / NULLIF(customers, 0))::numeric, 1) AS repeat_customer_pct,
       ROUND((1.0 * units / NULLIF(orders, 0))::numeric, 2) AS units_per_order,
       ROUND((avg_ship_days)::numeric, 2) AS avg_ship_days,
       orders - orders_with_ship_days AS orders_without_ship_days,
       ROUND((100.0 * late_orders / NULLIF(checked_orders, 0))::numeric, 1) AS late_pct
FROM   order_summary
CROSS  JOIN repeat_customers
CROSS  JOIN line_summary;
```
</details>

In [15]:
outputs = {}
# Headline KPIs
outputs['kpis'] = sql(r"""
WITH order_summary AS (
    SELECT SUM(revenue) AS revenue,
           COUNT(*) AS orders,
           COUNT(DISTINCT customer_id) AS customers,
           SUM(units) AS units,
           AVG(ship_days) AS avg_ship_days,
           COUNT(ship_days) AS orders_with_ship_days,
           SUM(shipped_late) AS late_orders,
           COUNT(shipped_late) AS checked_orders
    FROM   order_facts
),
repeat_customers AS (
    SELECT COUNT(*) AS repeat_customers
    FROM   (SELECT customer_id
            FROM   order_facts
            WHERE  customer_id IS NOT NULL
            GROUP  BY customer_id
            HAVING COUNT(*) >= 2) r
),
line_summary AS (
    SELECT AVG(revenue) AS avg_line_value   -- NOT the average order value: shown for contrast
    FROM   line_facts
)
SELECT ROUND(revenue, 2) AS revenue,
       orders,
       ROUND(1.0 * revenue / NULLIF(orders, 0), 2) AS avg_order_value,
       ROUND(avg_line_value, 2) AS avg_line_value,
       customers,
       ROUND(1.0 * orders / NULLIF(customers, 0), 2) AS orders_per_customer,
       ROUND(100.0 * repeat_customers / NULLIF(customers, 0), 1) AS repeat_customer_pct,
       ROUND(1.0 * units / NULLIF(orders, 0), 2) AS units_per_order,
       ROUND(avg_ship_days, 2) AS avg_ship_days,
       orders - orders_with_ship_days AS orders_without_ship_days,
       ROUND(100.0 * late_orders / NULLIF(checked_orders, 0), 1) AS late_pct
FROM   order_summary
CROSS  JOIN repeat_customers
CROSS  JOIN line_summary
""")
display(outputs['kpis'].head(20))
result = list(outputs.values())[-1] if outputs else pd.DataFrame()
# 📌 What we found
k = result.iloc[0].to_dict()
metrics["kpis"] = {key: (float(v) if v is not None and not pd.isna(v) else None) for key, v in k.items()}
if "revenue" in k:
    note(f"Revenue **{float(k['revenue']):,.2f}** from **{int(k['orders']):,}** orders: average order value **{float(k['avg_order_value']):,.2f}**.")
else:
    note(f"**{int(k['orders']):,}** orders in total.")
if "avg_line_value" in k:
    note(f"The average *line* is only {float(k['avg_line_value']):,.2f}. Averaging lines would understate the order value "
         f"{float(k['avg_order_value']) / float(k['avg_line_value']):.1f}×: the grain matters.")
if "customers" in k:
    note(f"{int(k['customers']):,} customers place {float(k['orders_per_customer']):.2f} orders each; "
         f"{float(k['repeat_customer_pct']):.1f}% ordered more than once.")
lead = 'ship_days'
if lead and f"avg_{lead}" in k:
    missing = int(k[f"orders_without_{lead}"])
    note(f"Average {lead}: **{float(k[f'avg_{lead}']):.2f}**" + (f" (over the orders that have one; {missing:,} don't, and AVG skips them)." if missing else "."))
if "late_pct" in k:
    note(f"{float(k['late_pct']):.1f}% of orders went out after the promised date.")
if "avg_rating" in k:
    note(f"Average rating: {float(k['avg_rating']):.2f}.")

,revenue,orders,avg_order_value,avg_line_value,customers,orders_per_customer,repeat_customer_pct,units_per_order,avg_ship_days,orders_without_ship_days,late_pct
0,1265793.04,830,1525.05,587.37,89,9.33,98.9,61.83,8.49,21,4.6


📌 Revenue **1,265,793.04** from **830** orders: average order value **1,525.05**.

📌 The average *line* is only 587.37. Averaging lines would understate the order value 2.6×: the grain matters.

📌 89 customers place 9.33 orders each; 98.9% ordered more than once.

📌 Average ship_days: **8.49** (over the orders that have one; 21 don't, and AVG skips them).

📌 4.6% of orders went out after the promised date.

## 11. What discounts cost

*4 · KPIs*

🔍 **What we found before:** Revenue is net of `discount`, so every discount is revenue given away.  
💡 **Why it matters:** A discount is only worth it if it buys something: more units, bigger orders or a customer kept. Measuring what each discount level costs, and whether discounted lines are larger, is the first question of any pricing review.  
🎯 **What we're going to do:** We band the lines by discount and compare what each band costs with the quantity it sells, so that discount policy is judged on evidence.

🛠 **What this SQL does:** A CASE expression puts each line in a discount band; a second CASE gives each band a sort order. Grouping by band adds up gross revenue and the discount given (gross − net); SUM(COUNT(*)) OVER () is a window over the grouped rows, so each band's share of lines needs no second query.

<details><summary>MySQL 8</summary>

```sql
-- Why: We band the lines by discount and compare what each band costs with the quantity it sells,
--      so that discount policy is judged on evidence.

-- Discount bands: cost and quantity
-- Discount bands (the discount is a fraction of the line price); 'no discount' is its own band.
WITH banded AS (
    SELECT CASE WHEN COALESCE(od.discount, 0) = 0 THEN 'no discount'
                WHEN od.discount <= 0.05 THEN '1-5%'
                WHEN od.discount <= 0.1 THEN '6-10%'
                WHEN od.discount <= 0.2 THEN '11-20%'
                ELSE 'over 20%'
           END AS discount_band,
           CASE WHEN COALESCE(od.discount, 0) = 0 THEN 0
                WHEN od.discount <= 0.05 THEN 1
                WHEN od.discount <= 0.1 THEN 2
                WHEN od.discount <= 0.2 THEN 3
                ELSE 4
           END AS band_order,
           od.unit_price * od.quantity AS gross_revenue,
           od.unit_price * od.quantity * (1 - COALESCE(od.discount, 0)) AS net_revenue,
           od.quantity AS quantity
    FROM   order_details od
)
SELECT discount_band,
       COUNT(*) AS line_rows,
       ROUND(100.0 * COUNT(*) / NULLIF(SUM(COUNT(*)) OVER (), 0), 1) AS lines_pct,
       ROUND(SUM(gross_revenue), 2) AS gross_revenue,
       ROUND(SUM(gross_revenue - net_revenue), 2) AS discount_given,
       ROUND(100.0 * SUM(gross_revenue - net_revenue) / NULLIF(SUM(gross_revenue), 0), 1) AS discount_pct_of_gross,
       ROUND(1.0 * SUM(quantity) / NULLIF(COUNT(*), 0), 2) AS avg_quantity_per_line
FROM   banded
GROUP  BY discount_band, band_order
ORDER  BY band_order;
```
</details>

<details><summary>PostgreSQL</summary>

```sql
-- Why: We band the lines by discount and compare what each band costs with the quantity it sells,
--      so that discount policy is judged on evidence.

-- Discount bands: cost and quantity
-- Discount bands (the discount is a fraction of the line price); 'no discount' is its own band.
WITH banded AS (
    SELECT CASE WHEN COALESCE(od.discount, 0) = 0 THEN 'no discount'
                WHEN od.discount <= 0.05 THEN '1-5%'
                WHEN od.discount <= 0.1 THEN '6-10%'
                WHEN od.discount <= 0.2 THEN '11-20%'
                ELSE 'over 20%'
           END AS discount_band,
           CASE WHEN COALESCE(od.discount, 0) = 0 THEN 0
                WHEN od.discount <= 0.05 THEN 1
                WHEN od.discount <= 0.1 THEN 2
                WHEN od.discount <= 0.2 THEN 3
                ELSE 4
           END AS band_order,
           od.unit_price * od.quantity AS gross_revenue,
           od.unit_price * od.quantity * (1 - COALESCE(od.discount, 0)) AS net_revenue,
           od.quantity AS quantity
    FROM   order_details od
)
SELECT discount_band,
       COUNT(*) AS line_rows,
       ROUND((100.0 * COUNT(*) / NULLIF(SUM(COUNT(*)) OVER (), 0))::numeric, 1) AS lines_pct,
       ROUND((SUM(gross_revenue))::numeric, 2) AS gross_revenue,
       ROUND((SUM(gross_revenue - net_revenue))::numeric, 2) AS discount_given,
       ROUND((100.0 * SUM(gross_revenue - net_revenue) / NULLIF(SUM(gross_revenue), 0))::numeric, 1) AS discount_pct_of_gross,
       ROUND((1.0 * SUM(quantity) / NULLIF(COUNT(*), 0))::numeric, 2) AS avg_quantity_per_line
FROM   banded
GROUP  BY discount_band, band_order
ORDER  BY band_order;
```
</details>

In [16]:
outputs = {}
# Discount bands: cost and quantity
outputs['discounts'] = sql(r"""
-- Discount bands (the discount is a fraction of the line price); 'no discount' is its own band.
WITH banded AS (
    SELECT CASE WHEN COALESCE(od.discount, 0) = 0 THEN 'no discount'
                WHEN od.discount <= 0.05 THEN '1-5%'
                WHEN od.discount <= 0.1 THEN '6-10%'
                WHEN od.discount <= 0.2 THEN '11-20%'
                ELSE 'over 20%'
           END AS discount_band,
           CASE WHEN COALESCE(od.discount, 0) = 0 THEN 0
                WHEN od.discount <= 0.05 THEN 1
                WHEN od.discount <= 0.1 THEN 2
                WHEN od.discount <= 0.2 THEN 3
                ELSE 4
           END AS band_order,
           od.unit_price * od.quantity AS gross_revenue,
           od.unit_price * od.quantity * (1 - COALESCE(od.discount, 0)) AS net_revenue,
           od.quantity AS quantity
    FROM   order_details od
)
SELECT discount_band,
       COUNT(*) AS line_rows,
       ROUND(100.0 * COUNT(*) / NULLIF(SUM(COUNT(*)) OVER (), 0), 1) AS lines_pct,
       ROUND(SUM(gross_revenue), 2) AS gross_revenue,
       ROUND(SUM(gross_revenue - net_revenue), 2) AS discount_given,
       ROUND(100.0 * SUM(gross_revenue - net_revenue) / NULLIF(SUM(gross_revenue), 0), 1) AS discount_pct_of_gross,
       ROUND(1.0 * SUM(quantity) / NULLIF(COUNT(*), 0), 2) AS avg_quantity_per_line
FROM   banded
GROUP  BY discount_band, band_order
ORDER  BY band_order
""")
display(outputs['discounts'].head(20))
result = list(outputs.values())[-1] if outputs else pd.DataFrame()
# 📌 What we found
bands = result.copy()
total_gross = float(bands["gross_revenue"].sum())
total_given = float(bands["discount_given"].sum())
none = bands[bands["discount_band"] == "no discount"]
some = bands[bands["discount_band"] != "no discount"]
metrics["discounts"] = {
    "given": total_given, "gross": total_gross, "pct_of_gross": 100 * total_given / total_gross if total_gross else 0.0,
    "discounted_lines_pct": float(some["lines_pct"].sum()),
    "qty_discounted": float((some["avg_quantity_per_line"] * some["line_rows"]).sum() / some["line_rows"].sum()) if len(some) else None,
    "qty_full_price": float(none["avg_quantity_per_line"].iloc[0]) if len(none) else None,
}
plt.figure(figsize=(8, 3.2))
plt.bar(bands["discount_band"], bands["discount_given"].astype(float), width=0.6, color=C_SERIES)
plt.title("Revenue given away, by discount band", loc="left"); plt.tight_layout(); plt.show()
dm = metrics["discounts"]
dm["bands"] = [{"band": str(r.discount_band), "lines_pct": float(r.lines_pct), "given": float(r.discount_given),
                "qty": float(r.avg_quantity_per_line)} for r in bands.itertuples()]
note(f"Discounts gave away **{total_given:,.2f}** ({dm['pct_of_gross']:.1f}% of gross revenue) on {dm['discounted_lines_pct']:.1f}% of lines.")
if dm["qty_discounted"] is not None and dm["qty_full_price"] is not None:
    lift = 100 * (dm["qty_discounted"] / dm["qty_full_price"] - 1) if dm["qty_full_price"] else float("nan")
    note(f"Discounted lines sell **{dm['qty_discounted']:.1f}** units on average vs {dm['qty_full_price']:.1f} at full price ({lift:+.0f}%). "
         + ("Discounts go with bigger lines, though this alone doesn't show they cause them." if lift > 10 else
            "Discounts don't come with noticeably bigger lines here: worth reviewing the policy."))

,discount_band,line_rows,lines_pct,gross_revenue,discount_given,discount_pct_of_gross,avg_quantity_per_line
0,no discount,1317,61.1,750698.61,0.00,0.0,21.72
1,1-5%,192,8.9,155683.36,7777.84,5.0,27.05
2,6-10%,174,8.1,101733.71,10170.65,10.0,25.10
3,11-20%,318,14.8,214424.82,37737.54,17.6,27.69
4,over 20%,154,7.1,131918.09,32979.52,25.0,28.24


📌 Discounts gave away **88,665.55** (6.5% of gross revenue) on 38.9% of lines.

📌 Discounted lines sell **27.1** units on average vs 21.7 at full price (+25%). Discounts go with bigger lines, though this alone doesn't show they cause them.

## 12. KPIs per quarter, with the driver next to the symptom

*4 · KPIs*

🔍 **What we found before:** Orders run over 8 quarters; 1998-Q2 is partial.  
💡 **Why it matters:** A KPI only means something next to its history. Putting the suspected driver in the same query as the symptom lets one table show whether they move together. The partial period is labelled so that its drop is not mistaken for a trend.  
🎯 **What we're going to do:** We compute the KPIs per quarter with LAG() for the change from the previous quarter and from the same period a year earlier, so that trends are measured, not eyeballed.

🛠 **What this SQL does:** The `quarterly` CTE groups order_facts by order_year, quarter_number; `with_prev` adds LAG() (the previous row's value in time order); the final SELECT turns them into % changes with 100.0 * (new − old) / NULLIF(old, 0). period_status marks partial periods.

<details><summary>MySQL 8</summary>

```sql
-- Why: We compute the KPIs per quarter with LAG() for the change from the previous quarter and from
--      the same period a year earlier, so that trends are measured, not eyeballed.

-- KPIs per quarter with LAG()
-- The data spans several years, so periods are grouped and ordered by order_year, quarter_number.
-- 1998-Q2 is partial (data from 1998-04-01 to 1998-05-06): its change is missing days, not a trend.
-- period_index = order_year * 4 + quarter_number: LAG() takes the previous *row*, so prev_revenue is kept only when that row is the calendar neighbour (index - 1); after a missing quarter it is NULL, not a wrong comparison.
-- revenue_year_ago joins the same quarter a year earlier on period_index - 4, so a gap can't shift it.
-- The suspected driver (ship_days) sits next to the symptom (revenue).
WITH quarterly AS (
    SELECT order_year,
           quarter_number,
           order_year * 4 + quarter_number AS period_index,
           SUM(revenue) AS revenue,
           COUNT(*) AS orders,
           COUNT(DISTINCT customer_id) AS customers,
           AVG(ship_days) AS avg_ship_days,
           SUM(shipped_late) AS late_orders,
           COUNT(shipped_late) AS checked_orders
    FROM   order_facts
    WHERE  order_year IS NOT NULL
    GROUP  BY order_year, quarter_number
),
with_prev AS (
    SELECT p.*,
           CASE WHEN p.period_index - LAG(p.period_index) OVER (ORDER BY p.period_index) = 1
                THEN LAG(p.revenue) OVER (ORDER BY p.period_index)
           END AS prev_revenue,
           y.revenue AS revenue_year_ago
    FROM   quarterly p
    LEFT   JOIN quarterly y ON y.period_index = p.period_index - 4
)
SELECT order_year,
       quarter_number,
       CASE WHEN (order_year = 1998 AND quarter_number = 2) THEN 'partial' ELSE 'complete' END AS period_status,
       ROUND(revenue, 2) AS revenue,
       orders,
       ROUND(1.0 * revenue / NULLIF(orders, 0), 2) AS avg_order_value,
       customers,
       ROUND(avg_ship_days, 1) AS avg_ship_days,
       ROUND(100.0 * late_orders / NULLIF(checked_orders, 0), 1) AS late_pct,
       ROUND(100.0 * (revenue - prev_revenue) / NULLIF(prev_revenue, 0), 1) AS revenue_qoq_pct,
       ROUND(100.0 * (revenue - revenue_year_ago) / NULLIF(revenue_year_ago, 0), 1) AS revenue_yoy_pct
FROM   with_prev
ORDER  BY order_year, quarter_number;
```
</details>

<details><summary>PostgreSQL</summary>

```sql
-- Why: We compute the KPIs per quarter with LAG() for the change from the previous quarter and from
--      the same period a year earlier, so that trends are measured, not eyeballed.

-- KPIs per quarter with LAG()
-- The data spans several years, so periods are grouped and ordered by order_year, quarter_number.
-- 1998-Q2 is partial (data from 1998-04-01 to 1998-05-06): its change is missing days, not a trend.
-- period_index = order_year * 4 + quarter_number: LAG() takes the previous *row*, so prev_revenue is kept only when that row is the calendar neighbour (index - 1); after a missing quarter it is NULL, not a wrong comparison.
-- revenue_year_ago joins the same quarter a year earlier on period_index - 4, so a gap can't shift it.
-- The suspected driver (ship_days) sits next to the symptom (revenue).
WITH quarterly AS (
    SELECT order_year,
           quarter_number,
           order_year * 4 + quarter_number AS period_index,
           SUM(revenue) AS revenue,
           COUNT(*) AS orders,
           COUNT(DISTINCT customer_id) AS customers,
           AVG(ship_days) AS avg_ship_days,
           SUM(shipped_late) AS late_orders,
           COUNT(shipped_late) AS checked_orders
    FROM   order_facts
    WHERE  order_year IS NOT NULL
    GROUP  BY order_year, quarter_number
),
with_prev AS (
    SELECT p.*,
           CASE WHEN p.period_index - LAG(p.period_index) OVER (ORDER BY p.period_index) = 1
                THEN LAG(p.revenue) OVER (ORDER BY p.period_index)
           END AS prev_revenue,
           y.revenue AS revenue_year_ago
    FROM   quarterly p
    LEFT   JOIN quarterly y ON y.period_index = p.period_index - 4
)
SELECT order_year,
       quarter_number,
       CASE WHEN (order_year = 1998 AND quarter_number = 2) THEN 'partial' ELSE 'complete' END AS period_status,
       ROUND((revenue)::numeric, 2) AS revenue,
       orders,
       ROUND((1.0 * revenue / NULLIF(orders, 0))::numeric, 2) AS avg_order_value,
       customers,
       ROUND((avg_ship_days)::numeric, 1) AS avg_ship_days,
       ROUND((100.0 * late_orders / NULLIF(checked_orders, 0))::numeric, 1) AS late_pct,
       ROUND((100.0 * (revenue - prev_revenue) / NULLIF(prev_revenue, 0))::numeric, 1) AS revenue_qoq_pct,
       ROUND((100.0 * (revenue - revenue_year_ago) / NULLIF(revenue_year_ago, 0))::numeric, 1) AS revenue_yoy_pct
FROM   with_prev
ORDER  BY order_year, quarter_number;
```
</details>

In [17]:
outputs = {}
# KPIs per quarter with LAG()
outputs['kpi_by_period'] = sql(r"""
-- The data spans several years, so periods are grouped and ordered by order_year, quarter_number.
-- 1998-Q2 is partial (data from 1998-04-01 to 1998-05-06): its change is missing days, not a trend.
-- period_index = order_year * 4 + quarter_number: LAG() takes the previous *row*, so prev_revenue is kept only when that row is the calendar neighbour (index - 1); after a missing quarter it is NULL, not a wrong comparison.
-- revenue_year_ago joins the same quarter a year earlier on period_index - 4, so a gap can't shift it.
-- The suspected driver (ship_days) sits next to the symptom (revenue).
WITH quarterly AS (
    SELECT order_year,
           quarter_number,
           order_year * 4 + quarter_number AS period_index,
           SUM(revenue) AS revenue,
           COUNT(*) AS orders,
           COUNT(DISTINCT customer_id) AS customers,
           AVG(ship_days) AS avg_ship_days,
           SUM(shipped_late) AS late_orders,
           COUNT(shipped_late) AS checked_orders
    FROM   order_facts
    WHERE  order_year IS NOT NULL
    GROUP  BY order_year, quarter_number
),
with_prev AS (
    SELECT p.*,
           CASE WHEN p.period_index - LAG(p.period_index) OVER (ORDER BY p.period_index) = 1
                THEN LAG(p.revenue) OVER (ORDER BY p.period_index)
           END AS prev_revenue,
           y.revenue AS revenue_year_ago
    FROM   quarterly p
    LEFT   JOIN quarterly y ON y.period_index = p.period_index - 4
)
SELECT order_year,
       quarter_number,
       CASE WHEN (order_year = 1998 AND quarter_number = 2) THEN 'partial' ELSE 'complete' END AS period_status,
       ROUND(revenue, 2) AS revenue,
       orders,
       ROUND(1.0 * revenue / NULLIF(orders, 0), 2) AS avg_order_value,
       customers,
       ROUND(avg_ship_days, 1) AS avg_ship_days,
       ROUND(100.0 * late_orders / NULLIF(checked_orders, 0), 1) AS late_pct,
       ROUND(100.0 * (revenue - prev_revenue) / NULLIF(prev_revenue, 0), 1) AS revenue_qoq_pct,
       ROUND(100.0 * (revenue - revenue_year_ago) / NULLIF(revenue_year_ago, 0), 1) AS revenue_yoy_pct
FROM   with_prev
ORDER  BY order_year, quarter_number
""")
display(outputs['kpi_by_period'].head(20))
result = list(outputs.values())[-1] if outputs else pd.DataFrame()
# 📌 What we found
per = result.copy()
unit, main = 'quarter', 'revenue'
cols = list(per.columns)
def label(row):
    if unit == "quarter":
        return f"{int(row[cols[0]])}-Q{int(row[cols[1]])}"
    if unit == "month":
        return f"{int(row[cols[0]])}-{int(row[cols[1]]):02d}"
    return str(int(row[cols[0]]))
per["period"] = per.apply(label, axis=1)
status = per["period_status"] if "period_status" in per else pd.Series("complete", index=per.index)
complete = per[status == "complete"]
change = f"{main}_qoq_pct"
metrics["periods"] = per[["period", main]].assign(status=status).to_dict("records")
metrics["period_table"] = [{k: (None if pd.isna(v) else (str(v) if isinstance(v, str) else float(v))) for k, v in row.items()}
                           for row in per.assign(period_status=status).to_dict("records")]
driver = 'avg_ship_days'
# Two stacked panels on one time axis (never two y-scales on one plot): the symptom on top, the driver below.
panels = 2 if driver and driver in per else 1
fig, axes = plt.subplots(panels, 1, figsize=(10, 2.9 * panels + 0.6), sharex=True, squeeze=False)
ax = axes[0][0]
ax.bar(per["period"], per[main].astype(float), width=0.6, color=[C_SERIES if s == "complete" else C_MUTED for s in status])
ax.set_title(f"{main.capitalize()} per {unit}" + (" (grey = partial period)" if (status != "complete").any() else ""), loc="left")
if panels == 2:
    axes[1][0].plot(per["period"], per[driver].astype(float), color=C_DRIVER, marker="o", markersize=5)
    axes[1][0].set_title(f"{driver.replace('_', ' ')} per {unit} (the suspected driver)", loc="left")
axes[-1][0].tick_params(axis="x", labelrotation=45)
plt.tight_layout(); plt.show()
if len(complete) >= 2:
    prev, last = complete.iloc[-2], complete.iloc[-1]
    pct = 100 * (float(last[main]) - float(prev[main])) / float(prev[main]) if float(prev[main]) else float("nan")
    metrics["qoq_last_complete"] = {"from": prev["period"], "to": last["period"], "pct": pct}
    show = (lambda v: f"{float(v):,.2f}") if main == "revenue" else (lambda v: f"{float(v):,.0f}")
    note(f"Last two complete {unit}s: {main} went from {show(prev[main])} ({prev['period']}) to {show(last[main])} "
         f"({last['period']}), **{pct:+.1f}%**.")
else:
    note(f"Only {len(complete)} complete {unit} so far: nothing to compare yet. The table still gives the level per {unit}; "
         f"period-over-period verdicts start once a second complete {unit} arrives.")
if (status != "complete").any():
    part = per[status != "complete"].iloc[-1]
    idx = per.index.get_loc(part.name)
    if idx > 0 and not pd.isna(part[change]):
        metrics["naive_change"] = {"period": part["period"], "pct": float(part[change])}
        note(f"⚠️ A naive report would say {main} changed **{float(part[change]):+.1f}%** in {part['period']}, but that {unit} is partial: "
             "the drop is missing days, not a trend.")
yoy = f"{main}_yoy_pct"
if yoy in per and change != yoy:
    rows = complete[complete[yoy].notna()]
    if len(rows):
        last = rows.iloc[-1]
        metrics["yoy"] = {"period": last["period"], "pct": float(last[yoy])}
        note(f"Year over year, {last['period']} is **{float(last[yoy]):+.1f}%** against the same {unit} a year earlier, which removes seasonality.")
if driver and driver in per and len(complete) >= 2:
    note(f"{driver}: {float(complete.iloc[-2][driver]):.1f} → {float(complete.iloc[-1][driver]):.1f} over the same two {unit}s.")

,order_year,quarter_number,period_status,revenue,orders,avg_order_value,customers,avg_ship_days,late_pct,revenue_qoq_pct,revenue_yoy_pct
0,1996,3,complete,79728.57,70,1138.98,43,8.9,7.1,NaN,NaN
1,1996,4,complete,128355.40,82,1565.31,45,7.5,2.4,61.0,NaN
2,1997,1,complete,138288.93,92,1503.14,54,9.2,5.4,7.7,NaN
3,1997,2,complete,143177.04,93,1539.54,56,9.0,4.3,3.5,NaN
4,1997,3,complete,153937.77,103,1494.54,56,8.2,4.9,7.5,93.1
5,1997,4,complete,181681.46,120,1514.01,60,9.2,6.7,18.0,41.5
6,1998,1,complete,298491.55,182,1640.06,72,8.6,4.4,64.3,115.8
7,1998,2,partial,142132.31,88,1615.14,56,6.4,0.0,-52.4,-0.7


📌 Last two complete quarters: revenue went from 181,681.46 (1997-Q4) to 298,491.55 (1998-Q1), **+64.3%**.

📌 ⚠️ A naive report would say revenue changed **-52.4%** in 1998-Q2, but that quarter is partial: the drop is missing days, not a trend.

📌 Year over year, 1998-Q1 is **+115.8%** against the same quarter a year earlier, which removes seasonality.

📌 avg_ship_days: 9.2 → 8.6 over the same two quarters.

## 13. Running total

*5 · Window functions*

🔍 **What we found before:** Revenue per quarter is known.  
💡 **Why it matters:** A running total shows how quickly the total builds up and when half of it was reached, which a per-period chart hides.  
🎯 **What we're going to do:** We add a running total with a window frame, so that cumulative progress can be read directly.

🛠 **What this SQL does:** SUM(…) OVER (ORDER BY period ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) adds up every period so far, without collapsing the rows like GROUP BY would; SUM(…) OVER () is the grand total, so their ratio is the share reached so far.

<details><summary>MySQL 8</summary>

```sql
-- Why: We add a running total with a window frame, so that cumulative progress can be read
--      directly.

-- Running total with a window frame
WITH per_period AS (
    SELECT order_year,
           quarter_number,
           order_year * 4 + quarter_number AS period_index,
           SUM(revenue) AS revenue
    FROM   order_facts
    WHERE  order_year IS NOT NULL
    GROUP  BY order_year, quarter_number
)
SELECT order_year, quarter_number,
       ROUND(revenue, 2) AS revenue,
       ROUND(SUM(revenue) OVER (ORDER BY order_year, quarter_number
                             ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW), 2) AS running_revenue,
       ROUND(100.0 * SUM(revenue) OVER (ORDER BY order_year, quarter_number
                             ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) / NULLIF(SUM(revenue) OVER (), 0), 1) AS running_pct_of_total
FROM   per_period
ORDER  BY order_year, quarter_number;
```
</details>

<details><summary>PostgreSQL</summary>

```sql
-- Why: We add a running total with a window frame, so that cumulative progress can be read
--      directly.

-- Running total with a window frame
WITH per_period AS (
    SELECT order_year,
           quarter_number,
           order_year * 4 + quarter_number AS period_index,
           SUM(revenue) AS revenue
    FROM   order_facts
    WHERE  order_year IS NOT NULL
    GROUP  BY order_year, quarter_number
)
SELECT order_year, quarter_number,
       ROUND((revenue)::numeric, 2) AS revenue,
       ROUND((SUM(revenue) OVER (ORDER BY order_year, quarter_number
                             ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW))::numeric, 2) AS running_revenue,
       ROUND((100.0 * SUM(revenue) OVER (ORDER BY order_year, quarter_number
                             ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) / NULLIF(SUM(revenue) OVER (), 0))::numeric, 1) AS running_pct_of_total
FROM   per_period
ORDER  BY order_year, quarter_number;
```
</details>

In [18]:
outputs = {}
# Running total with a window frame
outputs['running_total'] = sql(r"""
WITH per_period AS (
    SELECT order_year,
           quarter_number,
           order_year * 4 + quarter_number AS period_index,
           SUM(revenue) AS revenue
    FROM   order_facts
    WHERE  order_year IS NOT NULL
    GROUP  BY order_year, quarter_number
)
SELECT order_year, quarter_number,
       ROUND(revenue, 2) AS revenue,
       ROUND(SUM(revenue) OVER (ORDER BY order_year, quarter_number
                             ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW), 2) AS running_revenue,
       ROUND(100.0 * SUM(revenue) OVER (ORDER BY order_year, quarter_number
                             ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) / NULLIF(SUM(revenue) OVER (), 0), 1) AS running_pct_of_total
FROM   per_period
ORDER  BY order_year, quarter_number
""")
display(outputs['running_total'].head(20))
result = list(outputs.values())[-1] if outputs else pd.DataFrame()
# 📌 What we found
run = result
cols = list(run.columns)
half = run[run["running_pct_of_total"].astype(float) >= 50].iloc[0]
when = "-".join(str(int(half[c])) for c in cols[:2])
metrics["half_reached"] = when
plt.figure(figsize=(9, 3.5))
plt.plot(range(len(run)), run["running_revenue"].astype(float), color=C_SERIES, marker="o", markersize=5)
plt.xticks(range(len(run)), ["-".join(str(int(v)) for v in row) for row in run[cols[:2]].itertuples(index=False)], rotation=45)
plt.title("Running total of revenue"); plt.tight_layout(); plt.show()
note(f"Half of all revenue had been reached by **{when}** ({float(half['running_pct_of_total']):.1f}% at that point).")

,order_year,quarter_number,revenue,running_revenue,running_pct_of_total
0,1996,3,79728.57,79728.57,6.3
1,1996,4,128355.40,208083.97,16.4
2,1997,1,138288.93,346372.90,27.4
3,1997,2,143177.04,489549.94,38.7
4,1997,3,153937.77,643487.71,50.8
5,1997,4,181681.46,825169.17,65.2
6,1998,1,298491.55,1123660.73,88.8
7,1998,2,142132.31,1265793.04,100.0


📌 Half of all revenue had been reached by **1997-3** (50.8% at that point).

## 14. Moving average and unusual periods

*5 · Window functions*

🔍 **What we found before:** There are 8 quarters of revenue.  
💡 **Why it matters:** One period up or down proves little. A moving average shows the underlying direction, and comparing each period with the spread of the previous ones (a z-score) separates an unusual period from normal noise. A fixed limit such as |z| > 2 is wrong on short series: a spread measured on 4 points is so noisy that it is crossed by chance in about one period in six. Frames matter too: ROWS counts rows, RANGE counts values, so RANGE stays correct when a period is missing.  
🎯 **What we're going to do:** We add a 3-quarter moving average and flag a quarter only when it falls outside the 99% prediction interval of up to 8 earlier complete quarters (at least 4), so that normal noise is flagged about once in a hundred quarters whatever the baseline length.

🛠 **What this SQL does:** AVG(…) OVER (ORDER BY period_index RANGE BETWEEN 2 PRECEDING AND CURRENT ROW) is the moving average. The window '8 PRECEDING AND 1 PRECEDING' over baseline_value (NULL for partial periods, which AVG, STDDEV_SAMP and COUNT skip) gives the trailing average, its spread and how many periods it rests on; the z-score is (value − trailing average) ÷ trailing standard deviation, guarded with NULLIF and left NULL below 4 baseline periods. The check compares |z| with t(0.995, n − 1) × √(1 + 1/n) for a baseline of n periods (6.53 for 4, 4.36 for 6, 3.71 for 8).

<details><summary>MySQL 8</summary>

```sql
-- Why: We add a 3-quarter moving average and flag a quarter only when it falls outside the 99%
--      prediction interval of up to 8 earlier complete quarters (at least 4), so that normal noise
--      is flagged about once in a hundred quarters whatever the baseline length.

-- Moving average and z-score per quarter (RANGE frames)
-- RANGE frames count period_index values, not rows, so a missing period is skipped instead of pulling an older one in.
-- Partial periods are shown but never enter a baseline: baseline_value is NULL for them, and AVG / STDDEV / COUNT skip NULLs.
WITH per_period AS (
    SELECT order_year,
           quarter_number,
           order_year * 4 + quarter_number AS period_index,
           SUM(revenue) AS revenue
    FROM   order_facts
    WHERE  order_year IS NOT NULL
    GROUP  BY order_year, quarter_number
),
marked AS (
    SELECT p.*, CASE WHEN (p.order_year = 1998 AND p.quarter_number = 2) THEN NULL ELSE p.revenue END AS baseline_value
    FROM   per_period p
),
framed AS (
    SELECT m.*,
           AVG(revenue) OVER (ORDER BY period_index RANGE BETWEEN 2 PRECEDING AND CURRENT ROW) AS moving_avg_3,
           AVG(baseline_value) OVER (ORDER BY period_index RANGE BETWEEN 8 PRECEDING AND 1 PRECEDING) AS trailing_avg,
           STDDEV_SAMP(baseline_value) OVER (ORDER BY period_index RANGE BETWEEN 8 PRECEDING AND 1 PRECEDING) AS trailing_sd,
           COUNT(baseline_value) OVER (ORDER BY period_index RANGE BETWEEN 8 PRECEDING AND 1 PRECEDING) AS baseline_periods
    FROM   marked m
)
SELECT p.order_year, p.quarter_number,
       ROUND(p.revenue, 2) AS revenue,
       ROUND(p.moving_avg_3, 2) AS moving_avg_3,
       ROUND(p.trailing_avg, 2) AS trailing_avg,
       p.baseline_periods,
       CASE WHEN p.baseline_periods >= 4 THEN ROUND((p.revenue - p.trailing_avg) / NULLIF(p.trailing_sd, 0), 2) END AS z_score
FROM   framed p
ORDER  BY p.order_year, p.quarter_number;
```
</details>

<details><summary>PostgreSQL</summary>

```sql
-- Why: We add a 3-quarter moving average and flag a quarter only when it falls outside the 99%
--      prediction interval of up to 8 earlier complete quarters (at least 4), so that normal noise
--      is flagged about once in a hundred quarters whatever the baseline length.

-- Moving average and z-score per quarter (RANGE frames)
-- RANGE frames count period_index values, not rows, so a missing period is skipped instead of pulling an older one in.
-- Partial periods are shown but never enter a baseline: baseline_value is NULL for them, and AVG / STDDEV / COUNT skip NULLs.
WITH per_period AS (
    SELECT order_year,
           quarter_number,
           order_year * 4 + quarter_number AS period_index,
           SUM(revenue) AS revenue
    FROM   order_facts
    WHERE  order_year IS NOT NULL
    GROUP  BY order_year, quarter_number
),
marked AS (
    SELECT p.*, CASE WHEN (p.order_year = 1998 AND p.quarter_number = 2) THEN NULL ELSE p.revenue END AS baseline_value
    FROM   per_period p
),
framed AS (
    SELECT m.*,
           AVG(revenue) OVER (ORDER BY period_index RANGE BETWEEN 2 PRECEDING AND CURRENT ROW) AS moving_avg_3,
           AVG(baseline_value) OVER (ORDER BY period_index RANGE BETWEEN 8 PRECEDING AND 1 PRECEDING) AS trailing_avg,
           STDDEV_SAMP(baseline_value) OVER (ORDER BY period_index RANGE BETWEEN 8 PRECEDING AND 1 PRECEDING) AS trailing_sd,
           COUNT(baseline_value) OVER (ORDER BY period_index RANGE BETWEEN 8 PRECEDING AND 1 PRECEDING) AS baseline_periods
    FROM   marked m
)
SELECT p.order_year, p.quarter_number,
       ROUND((p.revenue)::numeric, 2) AS revenue,
       ROUND((p.moving_avg_3)::numeric, 2) AS moving_avg_3,
       ROUND((p.trailing_avg)::numeric, 2) AS trailing_avg,
       p.baseline_periods,
       CASE WHEN p.baseline_periods >= 4 THEN ROUND(((p.revenue - p.trailing_avg) / NULLIF(p.trailing_sd, 0))::numeric, 2) END AS z_score
FROM   framed p
ORDER  BY p.order_year, p.quarter_number;
```
</details>

In [19]:
outputs = {}
# Moving average and z-score per quarter (RANGE frames)
outputs['trend_signals'] = sql(r"""
-- RANGE frames count period_index values, not rows, so a missing period is skipped instead of pulling an older one in.
-- Partial periods are shown but never enter a baseline: baseline_value is NULL for them, and AVG / STDDEV / COUNT skip NULLs.
WITH per_period AS (
    SELECT order_year,
           quarter_number,
           order_year * 4 + quarter_number AS period_index,
           SUM(revenue) AS revenue
    FROM   order_facts
    WHERE  order_year IS NOT NULL
    GROUP  BY order_year, quarter_number
),
marked AS (
    SELECT p.*, CASE WHEN (p.order_year = 1998 AND p.quarter_number = 2) THEN NULL ELSE p.revenue END AS baseline_value
    FROM   per_period p
),
framed AS (
    SELECT m.*,
           AVG(revenue) OVER (ORDER BY period_index RANGE BETWEEN 2 PRECEDING AND CURRENT ROW) AS moving_avg_3,
           AVG(baseline_value) OVER (ORDER BY period_index RANGE BETWEEN 8 PRECEDING AND 1 PRECEDING) AS trailing_avg,
           STDDEV_SAMP(baseline_value) OVER (ORDER BY period_index RANGE BETWEEN 8 PRECEDING AND 1 PRECEDING) AS trailing_sd,
           COUNT(baseline_value) OVER (ORDER BY period_index RANGE BETWEEN 8 PRECEDING AND 1 PRECEDING) AS baseline_periods
    FROM   marked m
)
SELECT p.order_year, p.quarter_number,
       ROUND(p.revenue, 2) AS revenue,
       ROUND(p.moving_avg_3, 2) AS moving_avg_3,
       ROUND(p.trailing_avg, 2) AS trailing_avg,
       p.baseline_periods,
       CASE WHEN p.baseline_periods >= 4 THEN ROUND((p.revenue - p.trailing_avg) / NULLIF(p.trailing_sd, 0), 2) END AS z_score
FROM   framed p
ORDER  BY p.order_year, p.quarter_number
""")
display(outputs['trend_signals'].head(20))
result = list(outputs.values())[-1] if outputs else pd.DataFrame()
# 📌 What we found
sig = result.copy()
cols = list(sig.columns)
n = 2
sig["period"] = ["-Q".join(str(int(v)) for v in r) if True else "-".join(f"{int(v):02d}" if i else str(int(v)) for i, v in enumerate(r))
                 for r in sig[cols[:n]].itertuples(index=False)]
partial = set(['1998-Q2'])
sig["partial"] = sig["period"].isin(partial)
z = sig["z_score"].astype(float)
limits = {4: 6.53, 5: 5.04, 6: 4.36, 7: 3.96, 8: 3.71}   # 99% prediction interval: t(0.995, n - 1) * sqrt(1 + 1/n) for a baseline of n periods
sig["z_limit"] = [limits.get(int(k)) if pd.notna(k) else None for k in sig["baseline_periods"]]
unusual = sig[(z.abs() >= sig["z_limit"].astype(float)) & ~sig["partial"]]
metrics["unusual_periods"] = [{"period": r.period, "z": float(r.z_score), "value": float(r.revenue),
                               "baseline": int(r.baseline_periods), "limit": float(r.z_limit)} for r in unusual.itertuples()]
fig, ax = plt.subplots(figsize=(10, 3.4))
ax.bar(sig["period"], sig["revenue"].astype(float), width=0.6, color=[C_MUTED if p else C_SERIES for p in sig["partial"]])
ax.plot(sig["period"], sig["moving_avg_3"].astype(float), color=C_TOTAL, marker="o", markersize=4)
ax.set_title("revenue per quarter (bars) and its 3-quarter moving average (line)", loc="left")
ax.tick_params(axis="x", labelrotation=45); plt.tight_layout(); plt.show()
last = sig[~sig["partial"]].iloc[-1]
judged = int(z[~sig["partial"]].notna().sum())
note(f"Moving average at the last complete quarter ({last.period}): **{float(last.moving_avg_3):,.2f}**.")
if len(unusual):
    note(f"Unusual quarters (outside the 99% prediction interval of up to 8 earlier complete quarters): "
         + ", ".join(f"**{r.period}** (z = {r.z_score:+.1f}; the limit for a baseline of {int(r.baseline_periods)} is {r.z_limit:.2f})"
                     for r in unusual.itertuples())
         + ". Look for a one-off event (a big order, a promotion, a loading problem) before reading a trend into it.")
elif judged:
    note(f"None of the {judged} quarters with a baseline of 4+ earlier periods falls outside its 99% prediction interval: "
         "the movement is within normal noise.")
else:
    note(f"No quarter has 4 earlier complete quarters yet, so no z-score is computed; the moving average still shows the direction.")

,order_year,quarter_number,revenue,moving_avg_3,trailing_avg,baseline_periods,z_score
0,1996,3,79728.57,79728.57,NaN,0,NaN
1,1996,4,128355.40,104041.99,79728.57,1,NaN
2,1997,1,138288.93,115457.63,104041.99,2,NaN
3,1997,2,143177.04,136607.12,115457.63,3,NaN
4,1997,3,153937.77,145134.58,122387.49,4,1.08
5,1997,4,181681.46,159598.76,128697.54,5,1.83
6,1998,1,298491.55,211370.26,137528.20,6,4.78
7,1998,2,142132.31,207435.11,160522.96,7,-0.27


📌 Moving average at the last complete quarter (1998-Q1): **211,370.26**.

📌 Unusual quarters (outside the 99% prediction interval of up to 8 earlier complete quarters): **1998-Q1** (z = +4.8; the limit for a baseline of 6 is 4.36). Look for a one-off event (a big order, a promotion, a loading problem) before reading a trend into it.

## 15. Top 3 `product_name` per `category_name`

*5 · Window functions*

🔍 **What we found before:** Each `product_name` belongs to one `category_name`.  
💡 **Why it matters:** 'Top 3 per group' is the classic window-function task. RANK and DENSE_RANK give tied rows the same number (so a group can return more than 3); ROW_NUMBER never ties, so it returns exactly 3.  
🎯 **What we're going to do:** We rank `product_name` within each `category_name` by revenue, so that the leaders of every group are visible side by side.

🛠 **What this SQL does:** `item_totals` adds up revenue per group and item; `ranked` adds DENSE_RANK() and ROW_NUMBER() OVER (PARTITION BY the group ORDER BY revenue DESC). PARTITION BY restarts the numbering in every group. The outer query keeps the top ranks: a window result can't be filtered in the same SELECT's WHERE, which is why it sits in a CTE.

<details><summary>MySQL 8</summary>

```sql
-- Why: We rank product_name within each category_name by revenue, so that the leaders of every
--      group are visible side by side.

-- Top 3 per group with DENSE_RANK and ROW_NUMBER
WITH item_totals AS (
    SELECT category_name,
           product_name,
           SUM(revenue) AS revenue
    FROM   line_facts
    GROUP  BY category_name, product_name
),
ranked AS (
    SELECT t.*,
           DENSE_RANK() OVER (PARTITION BY category_name ORDER BY revenue DESC) AS revenue_rank,
           ROW_NUMBER() OVER (PARTITION BY category_name ORDER BY revenue DESC, product_name) AS row_num
    FROM   item_totals t
)
SELECT category_name,
       revenue_rank,
       row_num,
       product_name,
       ROUND(revenue, 2) AS revenue
FROM   ranked
WHERE  revenue_rank <= 3   -- DENSE_RANK keeps ties, so a group can return more than 3 rows
ORDER  BY category_name, revenue_rank, product_name;
```
</details>

<details><summary>PostgreSQL</summary>

```sql
-- Why: We rank product_name within each category_name by revenue, so that the leaders of every
--      group are visible side by side.

-- Top 3 per group with DENSE_RANK and ROW_NUMBER
WITH item_totals AS (
    SELECT category_name,
           product_name,
           SUM(revenue) AS revenue
    FROM   line_facts
    GROUP  BY category_name, product_name
),
ranked AS (
    SELECT t.*,
           DENSE_RANK() OVER (PARTITION BY category_name ORDER BY revenue DESC) AS revenue_rank,
           ROW_NUMBER() OVER (PARTITION BY category_name ORDER BY revenue DESC, product_name) AS row_num
    FROM   item_totals t
)
SELECT category_name,
       revenue_rank,
       row_num,
       product_name,
       ROUND((revenue)::numeric, 2) AS revenue
FROM   ranked
WHERE  revenue_rank <= 3   -- DENSE_RANK keeps ties, so a group can return more than 3 rows
ORDER  BY category_name, revenue_rank, product_name;
```
</details>

In [20]:
outputs = {}
# Top 3 per group with DENSE_RANK and ROW_NUMBER
outputs['top_n'] = sql(r"""
WITH item_totals AS (
    SELECT category_name,
           product_name,
           SUM(revenue) AS revenue
    FROM   line_facts
    GROUP  BY category_name, product_name
),
ranked AS (
    SELECT t.*,
           DENSE_RANK() OVER (PARTITION BY category_name ORDER BY revenue DESC) AS revenue_rank,
           ROW_NUMBER() OVER (PARTITION BY category_name ORDER BY revenue DESC, product_name) AS row_num
    FROM   item_totals t
)
SELECT category_name,
       revenue_rank,
       row_num,
       product_name,
       ROUND(revenue, 2) AS revenue
FROM   ranked
WHERE  revenue_rank <= 3   -- DENSE_RANK keeps ties, so a group can return more than 3 rows
ORDER  BY category_name, revenue_rank, product_name
""")
display(outputs['top_n'].head(20))
result = list(outputs.values())[-1] if outputs else pd.DataFrame()
# 📌 What we found
top = result
g, item, main = 'category_name', 'product_name', 'revenue'
sizes = top.groupby(g).size()
ties = sizes[sizes > 3]
metrics["top_items"] = {str(k): v[item].astype(str).tolist() for k, v in top.groupby(g)}
leader = top.sort_values(main, ascending=False).iloc[0]
note(f"Biggest single item: `{leader[item]}` in `{leader[g]}` ({float(leader[main]):,.2f}).")
if len(ties):
    note(f"Ties: {len(ties)} group(s) return more than 3 rows with DENSE_RANK; use ROW_NUMBER if exactly 3 are needed.")
else:
    note("No ties: DENSE_RANK and ROW_NUMBER pick the same items here, but only ROW_NUMBER guarantees exactly 3 per group.")

,category_name,revenue_rank,row_num,product_name,revenue
0,Beverages,1,1,Côte de Blaye,141396.73
1,Beverages,2,2,Ipoh Coffee,23526.70
2,Beverages,3,3,Chang,16355.96
3,Condiments,1,1,Vegie-spread,16701.10
4,Condiments,2,2,Sirop d'érable,14352.60
5,Condiments,3,3,Louisiana Fiery Hot Pepper Sauce,13869.89
6,Confections,1,1,Tarte au sucre,47234.97
7,Confections,2,2,Sir Rodney's Marmalade,22563.36
8,Confections,3,3,Gumbär Gummibärchen,19849.14
9,Dairy Products,1,1,Raclette Courdavault,71155.70


📌 Biggest single item: `Côte de Blaye` in `Beverages` (141,396.73).

📌 No ties: DENSE_RANK and ROW_NUMBER pick the same items here, but only ROW_NUMBER guarantees exactly 3 per group.

## 16. Revenue concentration (Pareto / ABC)

*5 · Window functions*

🔍 **What we found before:** Revenue is known per customer and per item.  
💡 **Why it matters:** Revenue is rarely spread evenly: often about 20% of customers bring about 80% of it (the Pareto pattern). How concentrated it is decides where account management, stock and service effort should go, and how exposed the business is if a few large customers leave.  
🎯 **What we're going to do:** We rank customers and items by revenue, add a running share with a window SUM, and class them A / B / C, so that concentration is a number, not an impression.

🛠 **What this SQL does:** SUM(revenue) OVER (ORDER BY revenue DESC ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) is the running total from the biggest down; dividing by SUM(revenue) OVER () gives the cumulative share. ABC classes come from a CASE on the share *before* each entity, so the entity that crosses 80% is still A. The class summary uses SUM(SUM(revenue)) OVER (): a window over grouped rows.

<details><summary>MySQL 8</summary>

```sql
-- Why: We rank customers and items by revenue, add a running share with a window SUM, and class
--      them A / B / C, so that concentration is a number, not an impression.

-- 1) Cumulative revenue share of customers (`customer_id`), largest first
-- ABC: an entity is A while the revenue before it is under 80% of the total, B under 95%, C after that.
WITH entity_revenue AS (
    SELECT customer_id, SUM(revenue) AS revenue
    FROM   order_facts
    WHERE  customer_id IS NOT NULL
    GROUP  BY customer_id
),
ranked AS (
    SELECT e.*,
           ROW_NUMBER() OVER (ORDER BY revenue DESC, customer_id) AS revenue_rank,
           COUNT(*) OVER () AS entities,
           SUM(revenue) OVER (ORDER BY revenue DESC, customer_id ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) AS running_revenue,
           SUM(revenue) OVER () AS total_revenue
    FROM   entity_revenue e
),
classed AS (
    SELECT r.*,
           CASE WHEN running_revenue - revenue < 0.80 * total_revenue THEN 'A'
                WHEN running_revenue - revenue < 0.95 * total_revenue THEN 'B'
                ELSE 'C'
           END AS abc_class
    FROM   ranked r
)
SELECT customer_id,
       revenue_rank,
       ROUND(revenue, 2) AS revenue,
       ROUND(100.0 * revenue_rank / NULLIF(entities, 0), 1) AS entity_pct,
       ROUND(100.0 * running_revenue / NULLIF(total_revenue, 0), 1) AS cumulative_revenue_pct,
       abc_class
FROM   classed
ORDER  BY revenue_rank;

-- 2) ABC classes of customers
WITH entity_revenue AS (
    SELECT customer_id, SUM(revenue) AS revenue
    FROM   order_facts
    WHERE  customer_id IS NOT NULL
    GROUP  BY customer_id
),
ranked AS (
    SELECT e.*,
           ROW_NUMBER() OVER (ORDER BY revenue DESC, customer_id) AS revenue_rank,
           COUNT(*) OVER () AS entities,
           SUM(revenue) OVER (ORDER BY revenue DESC, customer_id ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) AS running_revenue,
           SUM(revenue) OVER () AS total_revenue
    FROM   entity_revenue e
),
classed AS (
    SELECT r.*,
           CASE WHEN running_revenue - revenue < 0.80 * total_revenue THEN 'A'
                WHEN running_revenue - revenue < 0.95 * total_revenue THEN 'B'
                ELSE 'C'
           END AS abc_class
    FROM   ranked r
)
SELECT abc_class,
       COUNT(*) AS entities,
       ROUND(100.0 * COUNT(*) / NULLIF(SUM(COUNT(*)) OVER (), 0), 1) AS entities_pct,
       ROUND(SUM(revenue), 2) AS revenue,
       ROUND(100.0 * SUM(revenue) / NULLIF(SUM(SUM(revenue)) OVER (), 0), 1) AS revenue_pct
FROM   classed
GROUP  BY abc_class
ORDER  BY abc_class;

-- 3) Cumulative revenue share of items (`product_name`), largest first
-- ABC: an entity is A while the revenue before it is under 80% of the total, B under 95%, C after that.
WITH entity_revenue AS (
    SELECT product_name, SUM(revenue) AS revenue
    FROM   line_facts
    WHERE  product_name IS NOT NULL
    GROUP  BY product_name
),
ranked AS (
    SELECT e.*,
           ROW_NUMBER() OVER (ORDER BY revenue DESC, product_name) AS revenue_rank,
           COUNT(*) OVER () AS entities,
           SUM(revenue) OVER (ORDER BY revenue DESC, product_name ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) AS running_revenue,
           SUM(revenue) OVER () AS total_revenue
    FROM   entity_revenue e
),
classed AS (
    SELECT r.*,
           CASE WHEN running_revenue - revenue < 0.80 * total_revenue THEN 'A'
                WHEN running_revenue - revenue < 0.95 * total_revenue THEN 'B'
                ELSE 'C'
           END AS abc_class
    FROM   ranked r
)
SELECT product_name,
       revenue_rank,
       ROUND(revenue, 2) AS revenue,
       ROUND(100.0 * revenue_rank / NULLIF(entities, 0), 1) AS entity_pct,
       ROUND(100.0 * running_revenue / NULLIF(total_revenue, 0), 1) AS cumulative_revenue_pct,
       abc_class
FROM   classed
ORDER  BY revenue_rank;

-- 4) ABC classes of items
WITH entity_revenue AS (
    SELECT product_name, SUM(revenue) AS revenue
    FROM   line_facts
    WHERE  product_name IS NOT NULL
    GROUP  BY product_name
),
ranked AS (
    SELECT e.*,
           ROW_NUMBER() OVER (ORDER BY revenue DESC, product_name) AS revenue_rank,
           COUNT(*) OVER () AS entities,
           SUM(revenue) OVER (ORDER BY revenue DESC, product_name ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) AS running_revenue,
           SUM(revenue) OVER () AS total_revenue
    FROM   entity_revenue e
),
classed AS (
    SELECT r.*,
           CASE WHEN running_revenue - revenue < 0.80 * total_revenue THEN 'A'
                WHEN running_revenue - revenue < 0.95 * total_revenue THEN 'B'
                ELSE 'C'
           END AS abc_class
    FROM   ranked r
)
SELECT abc_class,
       COUNT(*) AS entities,
       ROUND(100.0 * COUNT(*) / NULLIF(SUM(COUNT(*)) OVER (), 0), 1) AS entities_pct,
       ROUND(SUM(revenue), 2) AS revenue,
       ROUND(100.0 * SUM(revenue) / NULLIF(SUM(SUM(revenue)) OVER (), 0), 1) AS revenue_pct
FROM   classed
GROUP  BY abc_class
ORDER  BY abc_class;
```
</details>

<details><summary>PostgreSQL</summary>

```sql
-- Why: We rank customers and items by revenue, add a running share with a window SUM, and class
--      them A / B / C, so that concentration is a number, not an impression.

-- 1) Cumulative revenue share of customers (`customer_id`), largest first
-- ABC: an entity is A while the revenue before it is under 80% of the total, B under 95%, C after that.
WITH entity_revenue AS (
    SELECT customer_id, SUM(revenue) AS revenue
    FROM   order_facts
    WHERE  customer_id IS NOT NULL
    GROUP  BY customer_id
),
ranked AS (
    SELECT e.*,
           ROW_NUMBER() OVER (ORDER BY revenue DESC, customer_id) AS revenue_rank,
           COUNT(*) OVER () AS entities,
           SUM(revenue) OVER (ORDER BY revenue DESC, customer_id ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) AS running_revenue,
           SUM(revenue) OVER () AS total_revenue
    FROM   entity_revenue e
),
classed AS (
    SELECT r.*,
           CASE WHEN running_revenue - revenue < 0.80 * total_revenue THEN 'A'
                WHEN running_revenue - revenue < 0.95 * total_revenue THEN 'B'
                ELSE 'C'
           END AS abc_class
    FROM   ranked r
)
SELECT customer_id,
       revenue_rank,
       ROUND((revenue)::numeric, 2) AS revenue,
       ROUND((100.0 * revenue_rank / NULLIF(entities, 0))::numeric, 1) AS entity_pct,
       ROUND((100.0 * running_revenue / NULLIF(total_revenue, 0))::numeric, 1) AS cumulative_revenue_pct,
       abc_class
FROM   classed
ORDER  BY revenue_rank;

-- 2) ABC classes of customers
WITH entity_revenue AS (
    SELECT customer_id, SUM(revenue) AS revenue
    FROM   order_facts
    WHERE  customer_id IS NOT NULL
    GROUP  BY customer_id
),
ranked AS (
    SELECT e.*,
           ROW_NUMBER() OVER (ORDER BY revenue DESC, customer_id) AS revenue_rank,
           COUNT(*) OVER () AS entities,
           SUM(revenue) OVER (ORDER BY revenue DESC, customer_id ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) AS running_revenue,
           SUM(revenue) OVER () AS total_revenue
    FROM   entity_revenue e
),
classed AS (
    SELECT r.*,
           CASE WHEN running_revenue - revenue < 0.80 * total_revenue THEN 'A'
                WHEN running_revenue - revenue < 0.95 * total_revenue THEN 'B'
                ELSE 'C'
           END AS abc_class
    FROM   ranked r
)
SELECT abc_class,
       COUNT(*) AS entities,
       ROUND((100.0 * COUNT(*) / NULLIF(SUM(COUNT(*)) OVER (), 0))::numeric, 1) AS entities_pct,
       ROUND((SUM(revenue))::numeric, 2) AS revenue,
       ROUND((100.0 * SUM(revenue) / NULLIF(SUM(SUM(revenue)) OVER (), 0))::numeric, 1) AS revenue_pct
FROM   classed
GROUP  BY abc_class
ORDER  BY abc_class;

-- 3) Cumulative revenue share of items (`product_name`), largest first
-- ABC: an entity is A while the revenue before it is under 80% of the total, B under 95%, C after that.
WITH entity_revenue AS (
    SELECT product_name, SUM(revenue) AS revenue
    FROM   line_facts
    WHERE  product_name IS NOT NULL
    GROUP  BY product_name
),
ranked AS (
    SELECT e.*,
           ROW_NUMBER() OVER (ORDER BY revenue DESC, product_name) AS revenue_rank,
           COUNT(*) OVER () AS entities,
           SUM(revenue) OVER (ORDER BY revenue DESC, product_name ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) AS running_revenue,
           SUM(revenue) OVER () AS total_revenue
    FROM   entity_revenue e
),
classed AS (
    SELECT r.*,
           CASE WHEN running_revenue - revenue < 0.80 * total_revenue THEN 'A'
                WHEN running_revenue - revenue < 0.95 * total_revenue THEN 'B'
                ELSE 'C'
           END AS abc_class
    FROM   ranked r
)
SELECT product_name,
       revenue_rank,
       ROUND((revenue)::numeric, 2) AS revenue,
       ROUND((100.0 * revenue_rank / NULLIF(entities, 0))::numeric, 1) AS entity_pct,
       ROUND((100.0 * running_revenue / NULLIF(total_revenue, 0))::numeric, 1) AS cumulative_revenue_pct,
       abc_class
FROM   classed
ORDER  BY revenue_rank;

-- 4) ABC classes of items
WITH entity_revenue AS (
    SELECT product_name, SUM(revenue) AS revenue
    FROM   line_facts
    WHERE  product_name IS NOT NULL
    GROUP  BY product_name
),
ranked AS (
    SELECT e.*,
           ROW_NUMBER() OVER (ORDER BY revenue DESC, product_name) AS revenue_rank,
           COUNT(*) OVER () AS entities,
           SUM(revenue) OVER (ORDER BY revenue DESC, product_name ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) AS running_revenue,
           SUM(revenue) OVER () AS total_revenue
    FROM   entity_revenue e
),
classed AS (
    SELECT r.*,
           CASE WHEN running_revenue - revenue < 0.80 * total_revenue THEN 'A'
                WHEN running_revenue - revenue < 0.95 * total_revenue THEN 'B'
                ELSE 'C'
           END AS abc_class
    FROM   ranked r
)
SELECT abc_class,
       COUNT(*) AS entities,
       ROUND((100.0 * COUNT(*) / NULLIF(SUM(COUNT(*)) OVER (), 0))::numeric, 1) AS entities_pct,
       ROUND((SUM(revenue))::numeric, 2) AS revenue,
       ROUND((100.0 * SUM(revenue) / NULLIF(SUM(SUM(revenue)) OVER (), 0))::numeric, 1) AS revenue_pct
FROM   classed
GROUP  BY abc_class
ORDER  BY abc_class;
```
</details>

In [21]:
outputs = {}
# Cumulative revenue share of customers (`customer_id`), largest first
outputs['curve_customers'] = sql(r"""
-- ABC: an entity is A while the revenue before it is under 80% of the total, B under 95%, C after that.
WITH entity_revenue AS (
    SELECT customer_id, SUM(revenue) AS revenue
    FROM   order_facts
    WHERE  customer_id IS NOT NULL
    GROUP  BY customer_id
),
ranked AS (
    SELECT e.*,
           ROW_NUMBER() OVER (ORDER BY revenue DESC, customer_id) AS revenue_rank,
           COUNT(*) OVER () AS entities,
           SUM(revenue) OVER (ORDER BY revenue DESC, customer_id ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) AS running_revenue,
           SUM(revenue) OVER () AS total_revenue
    FROM   entity_revenue e
),
classed AS (
    SELECT r.*,
           CASE WHEN running_revenue - revenue < 0.80 * total_revenue THEN 'A'
                WHEN running_revenue - revenue < 0.95 * total_revenue THEN 'B'
                ELSE 'C'
           END AS abc_class
    FROM   ranked r
)
SELECT customer_id,
       revenue_rank,
       ROUND(revenue, 2) AS revenue,
       ROUND(100.0 * revenue_rank / NULLIF(entities, 0), 1) AS entity_pct,
       ROUND(100.0 * running_revenue / NULLIF(total_revenue, 0), 1) AS cumulative_revenue_pct,
       abc_class
FROM   classed
ORDER  BY revenue_rank
""")
display(outputs['curve_customers'].head(20))
# ABC classes of customers
outputs['abc_customers'] = sql(r"""
WITH entity_revenue AS (
    SELECT customer_id, SUM(revenue) AS revenue
    FROM   order_facts
    WHERE  customer_id IS NOT NULL
    GROUP  BY customer_id
),
ranked AS (
    SELECT e.*,
           ROW_NUMBER() OVER (ORDER BY revenue DESC, customer_id) AS revenue_rank,
           COUNT(*) OVER () AS entities,
           SUM(revenue) OVER (ORDER BY revenue DESC, customer_id ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) AS running_revenue,
           SUM(revenue) OVER () AS total_revenue
    FROM   entity_revenue e
),
classed AS (
    SELECT r.*,
           CASE WHEN running_revenue - revenue < 0.80 * total_revenue THEN 'A'
                WHEN running_revenue - revenue < 0.95 * total_revenue THEN 'B'
                ELSE 'C'
           END AS abc_class
    FROM   ranked r
)
SELECT abc_class,
       COUNT(*) AS entities,
       ROUND(100.0 * COUNT(*) / NULLIF(SUM(COUNT(*)) OVER (), 0), 1) AS entities_pct,
       ROUND(SUM(revenue), 2) AS revenue,
       ROUND(100.0 * SUM(revenue) / NULLIF(SUM(SUM(revenue)) OVER (), 0), 1) AS revenue_pct
FROM   classed
GROUP  BY abc_class
ORDER  BY abc_class
""")
display(outputs['abc_customers'].head(20))
# Cumulative revenue share of items (`product_name`), largest first
outputs['curve_items'] = sql(r"""
-- ABC: an entity is A while the revenue before it is under 80% of the total, B under 95%, C after that.
WITH entity_revenue AS (
    SELECT product_name, SUM(revenue) AS revenue
    FROM   line_facts
    WHERE  product_name IS NOT NULL
    GROUP  BY product_name
),
ranked AS (
    SELECT e.*,
           ROW_NUMBER() OVER (ORDER BY revenue DESC, product_name) AS revenue_rank,
           COUNT(*) OVER () AS entities,
           SUM(revenue) OVER (ORDER BY revenue DESC, product_name ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) AS running_revenue,
           SUM(revenue) OVER () AS total_revenue
    FROM   entity_revenue e
),
classed AS (
    SELECT r.*,
           CASE WHEN running_revenue - revenue < 0.80 * total_revenue THEN 'A'
                WHEN running_revenue - revenue < 0.95 * total_revenue THEN 'B'
                ELSE 'C'
           END AS abc_class
    FROM   ranked r
)
SELECT product_name,
       revenue_rank,
       ROUND(revenue, 2) AS revenue,
       ROUND(100.0 * revenue_rank / NULLIF(entities, 0), 1) AS entity_pct,
       ROUND(100.0 * running_revenue / NULLIF(total_revenue, 0), 1) AS cumulative_revenue_pct,
       abc_class
FROM   classed
ORDER  BY revenue_rank
""")
display(outputs['curve_items'].head(20))
# ABC classes of items
outputs['abc_items'] = sql(r"""
WITH entity_revenue AS (
    SELECT product_name, SUM(revenue) AS revenue
    FROM   line_facts
    WHERE  product_name IS NOT NULL
    GROUP  BY product_name
),
ranked AS (
    SELECT e.*,
           ROW_NUMBER() OVER (ORDER BY revenue DESC, product_name) AS revenue_rank,
           COUNT(*) OVER () AS entities,
           SUM(revenue) OVER (ORDER BY revenue DESC, product_name ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) AS running_revenue,
           SUM(revenue) OVER () AS total_revenue
    FROM   entity_revenue e
),
classed AS (
    SELECT r.*,
           CASE WHEN running_revenue - revenue < 0.80 * total_revenue THEN 'A'
                WHEN running_revenue - revenue < 0.95 * total_revenue THEN 'B'
                ELSE 'C'
           END AS abc_class
    FROM   ranked r
)
SELECT abc_class,
       COUNT(*) AS entities,
       ROUND(100.0 * COUNT(*) / NULLIF(SUM(COUNT(*)) OVER (), 0), 1) AS entities_pct,
       ROUND(SUM(revenue), 2) AS revenue,
       ROUND(100.0 * SUM(revenue) / NULLIF(SUM(SUM(revenue)) OVER (), 0), 1) AS revenue_pct
FROM   classed
GROUP  BY abc_class
ORDER  BY abc_class
""")
display(outputs['abc_items'].head(20))
result = list(outputs.values())[-1] if outputs else pd.DataFrame()
# 📌 What we found
import numpy as _np
for name in ['customers', 'items']:
    cur = outputs[f"curve_{name}"]
    x = cur["entity_pct"].astype(float).to_numpy()
    y = cur["cumulative_revenue_pct"].astype(float).to_numpy()
    top20 = float(_np.interp(20, _np.r_[0, x], _np.r_[0, y]))
    cls = outputs[f"abc_{name}"].set_index("abc_class")
    a = cls.loc["A"] if "A" in cls.index else None
    metrics.setdefault("pareto", {})[name] = {
        "entities": int(len(cur)), "top20_revenue_pct": top20,
        "a_entities": int(a["entities"]) if a is not None else 0, "a_entities_pct": float(a["entities_pct"]) if a is not None else 0.0,
        "a_revenue_pct": float(a["revenue_pct"]) if a is not None else 0.0,
    }
    p = metrics["pareto"][name]
    step = max(1, len(x) // 60)                       # up to ~60 points for the curve in the business brief
    p["curve"] = [(0.0, 0.0)] + [(float(a), float(b)) for a, b in zip(x[::step], y[::step])] + [(float(x[-1]), float(y[-1]))]
    note(f"The top 20% of {name} bring **{top20:.0f}%** of revenue. Class A ({p['a_entities']:,} {name}, {p['a_entities_pct']:.0f}% of them) "
         f"brings {p['a_revenue_pct']:.0f}%.")
first = ['customers', 'items'][0]
cur = outputs[f"curve_{first}"]
plt.figure(figsize=(6.5, 4))
plt.plot(_np.r_[0, cur["entity_pct"].astype(float)], _np.r_[0, cur["cumulative_revenue_pct"].astype(float)], color=C_SERIES)
plt.plot([0, 100], [0, 100], color=C_MUTED, linewidth=1)
plt.axhline(80, color=C_MUTED, linewidth=1); plt.axvline(20, color=C_MUTED, linewidth=1)
plt.xlabel(f"% of {first}, largest first"); plt.ylabel("% of revenue")
plt.title(f"Revenue concentration across {first}", loc="left"); plt.tight_layout(); plt.show()

,customer_id,revenue_rank,revenue,entity_pct,cumulative_revenue_pct,abc_class
0,QUICK,1,110277.31,1.1,8.7,A
1,ERNSH,2,104874.98,2.2,17.0,A
2,SAVEA,3,104361.95,3.4,25.2,A
3,RATTC,4,51097.80,4.5,29.3,A
4,HUNGO,5,49979.91,5.6,33.2,A
5,HANAR,6,32841.37,6.7,35.8,A
6,KOENE,7,30908.38,7.9,38.3,A
7,FOLKO,8,29567.56,9.0,40.6,A
8,MEREP,9,28872.19,10.1,42.9,A
9,WHITC,10,27363.61,11.2,45.0,A


,abc_class,entities,entities_pct,revenue,revenue_pct
0,A,34,38.2,1013547.90,80.1
1,B,27,30.3,192826.66,15.2
2,C,28,31.5,59418.47,4.7


,product_name,revenue_rank,revenue,entity_pct,cumulative_revenue_pct,abc_class
0,Côte de Blaye,1,141396.73,1.3,11.2,A
1,Thüringer Rostbratwurst,2,80368.67,2.6,17.5,A
2,Raclette Courdavault,3,71155.70,3.9,23.1,A
3,Tarte au sucre,4,47234.97,5.2,26.9,A
4,Camembert Pierrot,5,46825.48,6.5,30.6,A
5,Gnocchi di nonna Alice,6,42593.06,7.8,33.9,A
6,Manjimup Dried Apples,7,41819.65,9.1,37.2,A
7,Alice Mutton,8,32698.38,10.4,39.8,A
8,Carnarvon Tigers,9,29171.88,11.7,42.1,A
9,Rössle Sauerkraut,10,25696.64,13.0,44.2,A


,abc_class,entities,entities_pct,revenue,revenue_pct
0,A,36,46.8,1015339.73,80.2
1,B,23,29.9,190243.72,15.0
2,C,18,23.4,60209.60,4.8


📌 The top 20% of customers bring **60%** of revenue. Class A (34 customers, 38% of them) brings 80%.

📌 The top 20% of items bring **54%** of revenue. Class A (36 items, 47% of them) brings 80%.

## 17. Each customer's latest order

*5 · Window functions*

🔍 **What we found before:** Customers place several orders each (`customer_id` repeats in order_facts).  
💡 **Why it matters:** 'The latest record per key' is how you de-duplicate history tables and how you measure recency. ROW_NUMBER() numbers each customer's orders from newest to oldest, so row 1 is the latest, exactly one per customer even when dates tie.  
🎯 **What we're going to do:** We keep each customer's latest order and measure the days since it, so that lapsing customers stand out.

🛠 **What this SQL does:** ROW_NUMBER() OVER (PARTITION BY customer_id ORDER BY order_date DESC, order_id DESC): the order_id tie-breaker makes the pick deterministic. Recency is measured to the latest date in the data, because measuring to today would make an old extract look like every customer had left.

<details><summary>MySQL 8</summary>

```sql
-- Why: We keep each customer's latest order and measure the days since it, so that lapsing
--      customers stand out.

-- The latest order per customer with ROW_NUMBER()
WITH ranked AS (
    SELECT customer_id, order_id, order_date, revenue,
           ROW_NUMBER() OVER (PARTITION BY customer_id ORDER BY order_date DESC, order_id DESC) AS row_num
    FROM   order_facts
    WHERE  customer_id IS NOT NULL
      AND  order_date IS NOT NULL
)
SELECT r.customer_id,
       r.order_id AS latest_order_id,
       r.order_date AS latest_order_date,
       ROUND(r.revenue, 2) AS latest_order_revenue,
       DATEDIFF((SELECT MAX(order_date) FROM order_facts), r.order_date) AS days_since_last_order   -- measured to the last date in the data, not to today
FROM   ranked r
WHERE  r.row_num = 1
ORDER  BY days_since_last_order DESC, r.customer_id;
```
</details>

<details><summary>PostgreSQL</summary>

```sql
-- Why: We keep each customer's latest order and measure the days since it, so that lapsing
--      customers stand out.

-- The latest order per customer with ROW_NUMBER()
WITH ranked AS (
    SELECT customer_id, order_id, order_date, revenue,
           ROW_NUMBER() OVER (PARTITION BY customer_id ORDER BY order_date DESC, order_id DESC) AS row_num
    FROM   order_facts
    WHERE  customer_id IS NOT NULL
      AND  order_date IS NOT NULL
)
SELECT r.customer_id,
       r.order_id AS latest_order_id,
       r.order_date AS latest_order_date,
       ROUND((r.revenue)::numeric, 2) AS latest_order_revenue,
       (SELECT MAX(order_date) FROM order_facts) - r.order_date AS days_since_last_order   -- measured to the last date in the data, not to today
FROM   ranked r
WHERE  r.row_num = 1
ORDER  BY days_since_last_order DESC, r.customer_id;
```
</details>

In [22]:
outputs = {}
# The latest order per customer with ROW_NUMBER()
outputs['latest_record'] = sql(r"""
WITH ranked AS (
    SELECT customer_id, order_id, order_date, revenue,
           ROW_NUMBER() OVER (PARTITION BY customer_id ORDER BY order_date DESC, order_id DESC) AS row_num
    FROM   order_facts
    WHERE  customer_id IS NOT NULL
      AND  order_date IS NOT NULL
)
SELECT r.customer_id,
       r.order_id AS latest_order_id,
       r.order_date AS latest_order_date,
       ROUND(r.revenue, 2) AS latest_order_revenue,
       (SELECT MAX(order_date) FROM order_facts) - r.order_date AS days_since_last_order   -- measured to the last date in the data, not to today
FROM   ranked r
WHERE  r.row_num = 1
ORDER  BY days_since_last_order DESC, r.customer_id
""")
display(outputs['latest_record'].head(20))
result = list(outputs.values())[-1] if outputs else pd.DataFrame()
# 📌 What we found
latest = result
days = latest["days_since_last_order"].astype(float)
metrics["recency_median"] = float(days.median())
lapsed = int((days > 180).sum())
note(f"{len(latest):,} customers (one row each). Median time since the last order: **{days.median():.0f} days**; "
     f"{lapsed:,} customer(s) have not ordered in over 180 days before the end of the data.")

,customer_id,latest_order_id,latest_order_date,latest_order_revenue,days_since_last_order
0,CENTC,10259,1996-07-18,100.80,657
1,LAZYK,10545,1997-05-22,210.00,349
2,HUNGC,10660,1997-09-08,1701.00,240
3,MEREP,10724,1997-10-30,638.50,188
4,FAMIA,10725,1997-10-31,287.80,187
5,VINET,10739,1997-11-12,240.00,175
6,GROSR,10785,1997-12-18,387.50,139
7,FOLIG,10789,1997-12-22,3687.00,135
8,LAUGB,10810,1998-01-01,187.00,125
9,TRAIH,10822,1998-01-08,237.90,118


📌 89 customers (one row each). Median time since the last order: **23 days**; 5 customer(s) have not ordered in over 180 days before the end of the data.

## 18. Time between orders, and overdue customers

*5 · Window functions*

🔍 **What we found before:** Customers order repeatedly (`customer_id` repeats in order_facts).  
💡 **Why it matters:** How long customers take to come back is the rhythm of the business. A customer who has been silent for more than twice their own usual gap is the earliest sign of churn, long before they are 'lost' by any fixed rule like 180 days.  
🎯 **What we're going to do:** We measure the days between consecutive orders with LAG and list customers who are overdue against their own rhythm, so that a win-back list exists before they are gone.

🛠 **What this SQL does:** LAG(order_date) OVER (PARTITION BY customer_id ORDER BY order_date, order_id) is each customer's previous order date (LEAD is the next one); the difference is the gap. Per customer, AVG of the gaps is their typical rhythm (the first order's NULL gap is skipped by AVG); a customer is overdue when the days since their last order exceed twice that rhythm. Only customers with 3+ orders are judged.

<details><summary>MySQL 8</summary>

```sql
-- Why: We measure the days between consecutive orders with LAG and list customers who are overdue
--      against their own rhythm, so that a win-back list exists before they are gone.

-- 1) Days between consecutive orders (LAG / LEAD)
WITH customer_orders AS (
    SELECT customer_id, order_id, order_date, revenue,
           LAG(order_date) OVER (PARTITION BY customer_id ORDER BY order_date, order_id) AS previous_order_date,
           LEAD(order_date) OVER (PARTITION BY customer_id ORDER BY order_date, order_id) AS next_order_date
    FROM   order_facts
    WHERE  customer_id IS NOT NULL
      AND  order_date IS NOT NULL
),
gaps AS (
    SELECT DATEDIFF(order_date, previous_order_date) AS days_since_previous
    FROM   customer_orders
    WHERE  previous_order_date IS NOT NULL      -- a customer's first order has no previous one
)
SELECT COUNT(*) AS repeat_orders,
       ROUND(AVG(days_since_previous), 1) AS avg_days_between_orders,
       MIN(days_since_previous) AS min_days,
       MAX(days_since_previous) AS max_days,
       ROUND(100.0 * SUM(CASE WHEN days_since_previous <= 30 THEN 1 ELSE 0 END) / NULLIF(COUNT(*), 0), 1) AS within_30_days_pct,
       ROUND(100.0 * SUM(CASE WHEN days_since_previous <= 90 THEN 1 ELSE 0 END) / NULLIF(COUNT(*), 0), 1) AS within_90_days_pct
FROM   gaps;

-- 2) Customers silent for more than twice their usual gap
WITH customer_orders AS (
    SELECT customer_id, order_id, order_date, revenue,
           LAG(order_date) OVER (PARTITION BY customer_id ORDER BY order_date, order_id) AS previous_order_date,
           LEAD(order_date) OVER (PARTITION BY customer_id ORDER BY order_date, order_id) AS next_order_date
    FROM   order_facts
    WHERE  customer_id IS NOT NULL
      AND  order_date IS NOT NULL
),
per_customer AS (
    SELECT customer_id,
           COUNT(*) AS orders,
           MAX(order_date) AS last_order_date,
           AVG(DATEDIFF(order_date, previous_order_date)) AS typical_gap_days,
           SUM(revenue) AS revenue    -- AVG skips the first order's NULL gap
    FROM   customer_orders
    GROUP  BY customer_id
),
scored AS (
    SELECT p.*,
           DATEDIFF((SELECT MAX(order_date) FROM order_facts), last_order_date) AS days_since_last_order
    FROM   per_customer p
    WHERE  orders >= 3                       -- at least two gaps, so 'typical' means something
)
SELECT customer_id,
       orders,
       last_order_date,
       ROUND(typical_gap_days, 0) AS typical_gap_days,
       days_since_last_order,
       CASE WHEN days_since_last_order > 2 * typical_gap_days THEN 'overdue' ELSE 'on track' END AS status,
       ROUND(revenue, 2) AS revenue
FROM   scored
ORDER  BY CASE WHEN days_since_last_order > 2 * typical_gap_days THEN 0 ELSE 1 END, revenue DESC, customer_id;
```
</details>

<details><summary>PostgreSQL</summary>

```sql
-- Why: We measure the days between consecutive orders with LAG and list customers who are overdue
--      against their own rhythm, so that a win-back list exists before they are gone.

-- 1) Days between consecutive orders (LAG / LEAD)
WITH customer_orders AS (
    SELECT customer_id, order_id, order_date, revenue,
           LAG(order_date) OVER (PARTITION BY customer_id ORDER BY order_date, order_id) AS previous_order_date,
           LEAD(order_date) OVER (PARTITION BY customer_id ORDER BY order_date, order_id) AS next_order_date
    FROM   order_facts
    WHERE  customer_id IS NOT NULL
      AND  order_date IS NOT NULL
),
gaps AS (
    SELECT order_date - previous_order_date AS days_since_previous
    FROM   customer_orders
    WHERE  previous_order_date IS NOT NULL      -- a customer's first order has no previous one
)
SELECT COUNT(*) AS repeat_orders,
       ROUND((AVG(days_since_previous))::numeric, 1) AS avg_days_between_orders,
       MIN(days_since_previous) AS min_days,
       MAX(days_since_previous) AS max_days,
       ROUND((100.0 * SUM(CASE WHEN days_since_previous <= 30 THEN 1 ELSE 0 END) / NULLIF(COUNT(*), 0))::numeric, 1) AS within_30_days_pct,
       ROUND((100.0 * SUM(CASE WHEN days_since_previous <= 90 THEN 1 ELSE 0 END) / NULLIF(COUNT(*), 0))::numeric, 1) AS within_90_days_pct
FROM   gaps;

-- 2) Customers silent for more than twice their usual gap
WITH customer_orders AS (
    SELECT customer_id, order_id, order_date, revenue,
           LAG(order_date) OVER (PARTITION BY customer_id ORDER BY order_date, order_id) AS previous_order_date,
           LEAD(order_date) OVER (PARTITION BY customer_id ORDER BY order_date, order_id) AS next_order_date
    FROM   order_facts
    WHERE  customer_id IS NOT NULL
      AND  order_date IS NOT NULL
),
per_customer AS (
    SELECT customer_id,
           COUNT(*) AS orders,
           MAX(order_date) AS last_order_date,
           AVG(order_date - previous_order_date) AS typical_gap_days,
           SUM(revenue) AS revenue    -- AVG skips the first order's NULL gap
    FROM   customer_orders
    GROUP  BY customer_id
),
scored AS (
    SELECT p.*,
           (SELECT MAX(order_date) FROM order_facts) - last_order_date AS days_since_last_order
    FROM   per_customer p
    WHERE  orders >= 3                       -- at least two gaps, so 'typical' means something
)
SELECT customer_id,
       orders,
       last_order_date,
       ROUND((typical_gap_days)::numeric, 0) AS typical_gap_days,
       days_since_last_order,
       CASE WHEN days_since_last_order > 2 * typical_gap_days THEN 'overdue' ELSE 'on track' END AS status,
       ROUND((revenue)::numeric, 2) AS revenue
FROM   scored
ORDER  BY CASE WHEN days_since_last_order > 2 * typical_gap_days THEN 0 ELSE 1 END, revenue DESC, customer_id;
```
</details>

In [23]:
outputs = {}
# Days between consecutive orders (LAG / LEAD)
outputs['gaps'] = sql(r"""
WITH customer_orders AS (
    SELECT customer_id, order_id, order_date, revenue,
           LAG(order_date) OVER (PARTITION BY customer_id ORDER BY order_date, order_id) AS previous_order_date,
           LEAD(order_date) OVER (PARTITION BY customer_id ORDER BY order_date, order_id) AS next_order_date
    FROM   order_facts
    WHERE  customer_id IS NOT NULL
      AND  order_date IS NOT NULL
),
gaps AS (
    SELECT order_date - previous_order_date AS days_since_previous
    FROM   customer_orders
    WHERE  previous_order_date IS NOT NULL      -- a customer's first order has no previous one
)
SELECT COUNT(*) AS repeat_orders,
       ROUND(AVG(days_since_previous), 1) AS avg_days_between_orders,
       MIN(days_since_previous) AS min_days,
       MAX(days_since_previous) AS max_days,
       ROUND(100.0 * SUM(CASE WHEN days_since_previous <= 30 THEN 1 ELSE 0 END) / NULLIF(COUNT(*), 0), 1) AS within_30_days_pct,
       ROUND(100.0 * SUM(CASE WHEN days_since_previous <= 90 THEN 1 ELSE 0 END) / NULLIF(COUNT(*), 0), 1) AS within_90_days_pct
FROM   gaps
""")
display(outputs['gaps'].head(20))
# Customers silent for more than twice their usual gap
outputs['overdue'] = sql(r"""
WITH customer_orders AS (
    SELECT customer_id, order_id, order_date, revenue,
           LAG(order_date) OVER (PARTITION BY customer_id ORDER BY order_date, order_id) AS previous_order_date,
           LEAD(order_date) OVER (PARTITION BY customer_id ORDER BY order_date, order_id) AS next_order_date
    FROM   order_facts
    WHERE  customer_id IS NOT NULL
      AND  order_date IS NOT NULL
),
per_customer AS (
    SELECT customer_id,
           COUNT(*) AS orders,
           MAX(order_date) AS last_order_date,
           AVG(order_date - previous_order_date) AS typical_gap_days,
           SUM(revenue) AS revenue    -- AVG skips the first order's NULL gap
    FROM   customer_orders
    GROUP  BY customer_id
),
scored AS (
    SELECT p.*,
           (SELECT MAX(order_date) FROM order_facts) - last_order_date AS days_since_last_order
    FROM   per_customer p
    WHERE  orders >= 3                       -- at least two gaps, so 'typical' means something
)
SELECT customer_id,
       orders,
       last_order_date,
       ROUND(typical_gap_days, 0) AS typical_gap_days,
       days_since_last_order,
       CASE WHEN days_since_last_order > 2 * typical_gap_days THEN 'overdue' ELSE 'on track' END AS status,
       ROUND(revenue, 2) AS revenue
FROM   scored
ORDER  BY CASE WHEN days_since_last_order > 2 * typical_gap_days THEN 0 ELSE 1 END, revenue DESC, customer_id
""")
display(outputs['overdue'].head(20))
result = list(outputs.values())[-1] if outputs else pd.DataFrame()
# 📌 What we found
g = outputs["gaps"].iloc[0]
od = outputs["overdue"]
late = od[od["status"] == "overdue"]
metrics["purchase_gaps"] = {"avg_days": float(g.avg_days_between_orders) if g.avg_days_between_orders is not None else None,
                            "within_90_pct": float(g.within_90_days_pct or 0), "judged": int(len(od)), "overdue": int(len(late))}
if "revenue" in od and len(od):
    metrics["purchase_gaps"]["overdue_revenue_pct"] = 100 * float(late["revenue"].sum()) / float(od["revenue"].sum())
note(f"Repeat orders come **{float(g.avg_days_between_orders):.0f} days** apart on average; {float(g.within_90_days_pct):.0f}% within 90 days.")
if len(od):
    share = metrics["purchase_gaps"].get("overdue_revenue_pct")
    note(f"**{len(late):,} of {len(od):,}** repeat customers are overdue (silent for more than twice their usual gap)"
         + (f"; they brought {share:.0f}% of these customers' revenue: the win-back list." if share is not None else "."))

,repeat_orders,avg_days_between_orders,min_days,max_days,within_30_days_pct,within_90_days_pct
0,741,58.9,0,532,46.2,79.5


,customer_id,orders,last_order_date,typical_gap_days,days_since_last_order,status,revenue
0,MEREP,13,1997-10-30,32.0,188,overdue,28872.19
1,BLONP,11,1998-01-12,54.0,114,overdue,18534.08
2,FAMIA,7,1997-10-31,60.0,187,overdue,4107.55
3,HUNGC,5,1997-09-08,69.0,240,overdue,3063.20
4,LACOR,4,1998-03-24,18.0,43,overdue,1992.05
5,QUICK,28,1998-04-14,23.0,22,on track,110277.31
6,ERNSH,30,1998-05-05,23.0,1,on track,104874.98
7,SAVEA,31,1998-05-01,19.0,5,on track,104361.95
8,RATTC,18,1998-05-06,38.0,0,on track,51097.80
9,HUNGO,19,1998-04-30,33.0,6,on track,49979.91


📌 Repeat orders come **59 days** apart on average; 80% within 90 days.

📌 **5 of 86** repeat customers are overdue (silent for more than twice their usual gap); they brought 4% of these customers' revenue: the win-back list.

## 19. RFM segments with NTILE

*5 · Window functions*

🔍 **What we found before:** Every customer has a last order date, an order count and a revenue total.  
💡 **Why it matters:** RFM (recency, frequency, monetary) turns three numbers into a segment a marketer can act on. NTILE(5) ranks customers into five equal-sized buckets per measure; ties are split arbitrarily, so the customer id is added to the ORDER BY to make the split repeatable.  
🎯 **What we're going to do:** We score every customer 1-5 on R, F and M and name the segments, so that revenue concentration and churn risk become visible.

🛠 **What this SQL does:** `customer_totals` aggregates order_facts per customer; `scored` adds three NTILE(5) windows; a CASE turns the scores into named segments. It is saved as the view customer_rfm, then summarised per segment.

<details><summary>MySQL 8</summary>

```sql
-- Why: We score every customer 1-5 on R, F and M and name the segments, so that revenue
--      concentration and churn risk become visible.

-- 1) View `customer_rfm`: NTILE scores per customer
CREATE OR REPLACE VIEW customer_rfm AS
WITH customer_totals AS (
    SELECT customer_id,
           DATEDIFF((SELECT MAX(order_date) FROM order_facts), MAX(order_date)) AS recency_days,
           COUNT(*) AS frequency,
           SUM(revenue) AS monetary
    FROM   order_facts
    WHERE  customer_id IS NOT NULL
      AND  order_date IS NOT NULL
    GROUP  BY customer_id
),
scored AS (
    SELECT t.*,
           NTILE(5) OVER (ORDER BY recency_days DESC, customer_id) AS r_score,   -- 5 = most recent
           NTILE(5) OVER (ORDER BY frequency, customer_id) AS f_score,            -- 5 = most orders
           NTILE(5) OVER (ORDER BY monetary, customer_id) AS m_score              -- 5 = most revenue
    FROM   customer_totals t
)
SELECT s.*,
       CASE WHEN r_score >= 4 AND f_score >= 4 THEN 'Champions'
            WHEN f_score >= 4 THEN 'Loyal'
            WHEN r_score <= 2 AND f_score >= 3 THEN 'At risk'
            WHEN r_score >= 4 AND f_score <= 2 THEN 'Recent, few orders'
            WHEN r_score <= 2 AND f_score <= 2 THEN 'Hibernating'
            ELSE 'Needs attention'
       END AS segment
FROM   scored s;

-- 2) Customers and revenue per segment
SELECT segment,
       COUNT(*) AS customers,
       ROUND(SUM(monetary), 2) AS revenue,
       ROUND(100.0 * SUM(monetary) / NULLIF((SELECT SUM(monetary) FROM customer_rfm), 0), 1) AS revenue_pct,
       ROUND(AVG(recency_days), 0) AS avg_recency_days
FROM   customer_rfm
GROUP  BY segment
ORDER  BY revenue DESC, segment;
```
</details>

<details><summary>PostgreSQL</summary>

```sql
-- Why: We score every customer 1-5 on R, F and M and name the segments, so that revenue
--      concentration and churn risk become visible.

-- 1) View `customer_rfm`: NTILE scores per customer
CREATE OR REPLACE VIEW customer_rfm AS
WITH customer_totals AS (
    SELECT customer_id,
           (SELECT MAX(order_date) FROM order_facts) - MAX(order_date) AS recency_days,
           COUNT(*) AS frequency,
           SUM(revenue) AS monetary
    FROM   order_facts
    WHERE  customer_id IS NOT NULL
      AND  order_date IS NOT NULL
    GROUP  BY customer_id
),
scored AS (
    SELECT t.*,
           NTILE(5) OVER (ORDER BY recency_days DESC, customer_id) AS r_score,   -- 5 = most recent
           NTILE(5) OVER (ORDER BY frequency, customer_id) AS f_score,            -- 5 = most orders
           NTILE(5) OVER (ORDER BY monetary, customer_id) AS m_score              -- 5 = most revenue
    FROM   customer_totals t
)
SELECT s.*,
       CASE WHEN r_score >= 4 AND f_score >= 4 THEN 'Champions'
            WHEN f_score >= 4 THEN 'Loyal'
            WHEN r_score <= 2 AND f_score >= 3 THEN 'At risk'
            WHEN r_score >= 4 AND f_score <= 2 THEN 'Recent, few orders'
            WHEN r_score <= 2 AND f_score <= 2 THEN 'Hibernating'
            ELSE 'Needs attention'
       END AS segment
FROM   scored s;

-- 2) Customers and revenue per segment
SELECT segment,
       COUNT(*) AS customers,
       ROUND((SUM(monetary))::numeric, 2) AS revenue,
       ROUND((100.0 * SUM(monetary) / NULLIF((SELECT SUM(monetary) FROM customer_rfm), 0))::numeric, 1) AS revenue_pct,
       ROUND((AVG(recency_days))::numeric, 0) AS avg_recency_days
FROM   customer_rfm
GROUP  BY segment
ORDER  BY revenue DESC, segment;
```
</details>

In [24]:
outputs = {}
# View `customer_rfm`: NTILE scores per customer
outputs['rfm_view'] = sql(r"""
CREATE OR REPLACE VIEW customer_rfm AS
WITH customer_totals AS (
    SELECT customer_id,
           (SELECT MAX(order_date) FROM order_facts) - MAX(order_date) AS recency_days,
           COUNT(*) AS frequency,
           SUM(revenue) AS monetary
    FROM   order_facts
    WHERE  customer_id IS NOT NULL
      AND  order_date IS NOT NULL
    GROUP  BY customer_id
),
scored AS (
    SELECT t.*,
           NTILE(5) OVER (ORDER BY recency_days DESC, customer_id) AS r_score,   -- 5 = most recent
           NTILE(5) OVER (ORDER BY frequency, customer_id) AS f_score,            -- 5 = most orders
           NTILE(5) OVER (ORDER BY monetary, customer_id) AS m_score              -- 5 = most revenue
    FROM   customer_totals t
)
SELECT s.*,
       CASE WHEN r_score >= 4 AND f_score >= 4 THEN 'Champions'
            WHEN f_score >= 4 THEN 'Loyal'
            WHEN r_score <= 2 AND f_score >= 3 THEN 'At risk'
            WHEN r_score >= 4 AND f_score <= 2 THEN 'Recent, few orders'
            WHEN r_score <= 2 AND f_score <= 2 THEN 'Hibernating'
            ELSE 'Needs attention'
       END AS segment
FROM   scored s
""")
display(outputs['rfm_view'].head(20))
# Customers and revenue per segment
outputs['rfm_segments'] = sql(r"""
SELECT segment,
       COUNT(*) AS customers,
       ROUND(SUM(monetary), 2) AS revenue,
       ROUND(100.0 * SUM(monetary) / NULLIF((SELECT SUM(monetary) FROM customer_rfm), 0), 1) AS revenue_pct,
       ROUND(AVG(recency_days), 0) AS avg_recency_days
FROM   customer_rfm
GROUP  BY segment
ORDER  BY revenue DESC, segment
""")
display(outputs['rfm_segments'].head(20))
result = list(outputs.values())[-1] if outputs else pd.DataFrame()
# 📌 What we found
seg = outputs["rfm_segments"]
metrics["rfm"] = {r.segment: {"customers": int(r.customers), "revenue_pct": float(r.revenue_pct)} for r in seg.itertuples()}
top = seg.iloc[0]
metrics["rfm_top_share"] = (str(top.segment), float(top.revenue_pct), int(top.customers))
plt.figure(figsize=(8, 3.5))
plt.barh(seg["segment"].iloc[::-1], seg["revenue_pct"].astype(float).iloc[::-1], color=C_SERIES, height=0.6)
plt.xlabel("% of revenue"); plt.title("Revenue share by RFM segment"); plt.tight_layout(); plt.show()
total = int(seg["customers"].sum())
note(f"**{top.segment}** ({int(top.customers):,} of {total:,} customers) bring **{float(top.revenue_pct):.1f}%** of revenue.")
risk = seg[seg["segment"] == "At risk"]
if len(risk):
    note(f"At risk: {int(risk.customers.iloc[0]):,} customers who used to order often but not recently ({float(risk.revenue_pct.iloc[0]):.1f}% of revenue).")

,Count


,segment,customers,revenue,revenue_pct,avg_recency_days
0,Champions,22,660972.98,52.2,6.0
1,Loyal,13,338392.03,26.7,52.0
2,Needs attention,15,109929.86,8.7,20.0
3,Hibernating,23,70761.90,5.6,124.0
4,At risk,8,64247.45,5.1,72.0
5,"Recent, few orders",8,21488.81,1.7,7.0


📌 **Champions** (22 of 89 customers) bring **52.2%** of revenue.

📌 At risk: 8 customers who used to order often but not recently (5.1% of revenue).

## 20. Driver tree: what moved revenue

*6 · Root cause*

🔍 **What we found before:** The last two complete quarters are 1997-Q4 and 1998-Q1 (1998-Q2 is partial and left out).  
💡 **Why it matters:** 'Revenue changed' is a symptom. Revenue = orders × average order value, so the change comes from more or fewer orders, bigger or smaller orders, or both, and then from particular segments. Splitting it is how a KPI move becomes a cause someone can act on.  
🎯 **What we're going to do:** We split the revenue change from 1997-Q4 to 1998-Q1 into a volume effect, an order-value effect and their interaction, and then by `customer_country`, `category_name`, `employee_last_name`, so that the change is traced to the part that moved.

🛠 **What this SQL does:** The first query computes each driver for both periods. The others use conditional aggregation (SUM(CASE WHEN period = … THEN revenue ELSE 0 END)) to put both periods side by side per segment, and a window SUM(…) OVER () to express each segment's change as a share of the total change.

<details><summary>MySQL 8</summary>

```sql
-- Why: We split the revenue change from 1997-Q4 to 1998-Q1 into a volume effect, an order-value
--      effect and their interaction, and then by customer_country, category_name,
--      employee_last_name, so that the change is traced to the part that moved.

-- 1) The drivers of revenue in 1997-Q4 and 1998-Q1
-- revenue = orders × average order value; orders = customers × orders per customer; AOV = units per order × revenue per unit
SELECT order_year,
       quarter_number,
       ROUND(SUM(revenue), 2) AS revenue,
       COUNT(*) AS orders,
       COUNT(DISTINCT customer_id) AS customers,
       ROUND(1.0 * COUNT(*) / NULLIF(COUNT(DISTINCT customer_id), 0), 4) AS orders_per_customer,
       ROUND(1.0 * SUM(revenue) / NULLIF(COUNT(*), 0), 4) AS avg_order_value,
       ROUND(1.0 * SUM(units) / NULLIF(COUNT(*), 0), 4) AS units_per_order,
       ROUND(1.0 * SUM(revenue) / NULLIF(SUM(units), 0), 4) AS revenue_per_unit
FROM   order_facts
WHERE  (order_year = 1997 AND quarter_number = 4)
   OR  (order_year = 1998 AND quarter_number = 1)
GROUP  BY order_year, quarter_number
ORDER  BY order_year, quarter_number;

-- 2) Change in revenue by `customer_country` (1997-Q4 → 1998-Q1)
WITH by_segment AS (
    SELECT customer_country,
           SUM(CASE WHEN (order_year = 1997 AND quarter_number = 4) THEN revenue ELSE 0 END) AS revenue_before,
           SUM(CASE WHEN (order_year = 1998 AND quarter_number = 1) THEN revenue ELSE 0 END) AS revenue_after
    FROM   order_facts
    GROUP  BY customer_country
)
SELECT customer_country,
       ROUND(revenue_before, 2) AS revenue_before,
       ROUND(revenue_after, 2) AS revenue_after,
       ROUND(revenue_after - revenue_before, 2) AS revenue_change,
       ROUND(100.0 * (revenue_after - revenue_before) / NULLIF(SUM(revenue_after - revenue_before) OVER (), 0), 1) AS share_of_change_pct
FROM   by_segment
ORDER  BY ABS(revenue_after - revenue_before) DESC, customer_country;

-- 3) Change in revenue by `category_name` (1997-Q4 → 1998-Q1)
WITH by_segment AS (
    SELECT category_name,
           SUM(CASE WHEN (order_year = 1997 AND quarter_number = 4) THEN revenue ELSE 0 END) AS revenue_before,
           SUM(CASE WHEN (order_year = 1998 AND quarter_number = 1) THEN revenue ELSE 0 END) AS revenue_after
    FROM   line_facts
    GROUP  BY category_name
)
SELECT category_name,
       ROUND(revenue_before, 2) AS revenue_before,
       ROUND(revenue_after, 2) AS revenue_after,
       ROUND(revenue_after - revenue_before, 2) AS revenue_change,
       ROUND(100.0 * (revenue_after - revenue_before) / NULLIF(SUM(revenue_after - revenue_before) OVER (), 0), 1) AS share_of_change_pct
FROM   by_segment
ORDER  BY ABS(revenue_after - revenue_before) DESC, category_name;

-- 4) Change in revenue by `employee_last_name` (1997-Q4 → 1998-Q1)
WITH by_segment AS (
    SELECT employee_last_name,
           SUM(CASE WHEN (order_year = 1997 AND quarter_number = 4) THEN revenue ELSE 0 END) AS revenue_before,
           SUM(CASE WHEN (order_year = 1998 AND quarter_number = 1) THEN revenue ELSE 0 END) AS revenue_after
    FROM   order_facts
    GROUP  BY employee_last_name
)
SELECT employee_last_name,
       ROUND(revenue_before, 2) AS revenue_before,
       ROUND(revenue_after, 2) AS revenue_after,
       ROUND(revenue_after - revenue_before, 2) AS revenue_change,
       ROUND(100.0 * (revenue_after - revenue_before) / NULLIF(SUM(revenue_after - revenue_before) OVER (), 0), 1) AS share_of_change_pct
FROM   by_segment
ORDER  BY ABS(revenue_after - revenue_before) DESC, employee_last_name;
```
</details>

<details><summary>PostgreSQL</summary>

```sql
-- Why: We split the revenue change from 1997-Q4 to 1998-Q1 into a volume effect, an order-value
--      effect and their interaction, and then by customer_country, category_name,
--      employee_last_name, so that the change is traced to the part that moved.

-- 1) The drivers of revenue in 1997-Q4 and 1998-Q1
-- revenue = orders × average order value; orders = customers × orders per customer; AOV = units per order × revenue per unit
SELECT order_year,
       quarter_number,
       ROUND((SUM(revenue))::numeric, 2) AS revenue,
       COUNT(*) AS orders,
       COUNT(DISTINCT customer_id) AS customers,
       ROUND((1.0 * COUNT(*) / NULLIF(COUNT(DISTINCT customer_id), 0))::numeric, 4) AS orders_per_customer,
       ROUND((1.0 * SUM(revenue) / NULLIF(COUNT(*), 0))::numeric, 4) AS avg_order_value,
       ROUND((1.0 * SUM(units) / NULLIF(COUNT(*), 0))::numeric, 4) AS units_per_order,
       ROUND((1.0 * SUM(revenue) / NULLIF(SUM(units), 0))::numeric, 4) AS revenue_per_unit
FROM   order_facts
WHERE  (order_year = 1997 AND quarter_number = 4)
   OR  (order_year = 1998 AND quarter_number = 1)
GROUP  BY order_year, quarter_number
ORDER  BY order_year, quarter_number;

-- 2) Change in revenue by `customer_country` (1997-Q4 → 1998-Q1)
WITH by_segment AS (
    SELECT customer_country,
           SUM(CASE WHEN (order_year = 1997 AND quarter_number = 4) THEN revenue ELSE 0 END) AS revenue_before,
           SUM(CASE WHEN (order_year = 1998 AND quarter_number = 1) THEN revenue ELSE 0 END) AS revenue_after
    FROM   order_facts
    GROUP  BY customer_country
)
SELECT customer_country,
       ROUND((revenue_before)::numeric, 2) AS revenue_before,
       ROUND((revenue_after)::numeric, 2) AS revenue_after,
       ROUND((revenue_after - revenue_before)::numeric, 2) AS revenue_change,
       ROUND((100.0 * (revenue_after - revenue_before) / NULLIF(SUM(revenue_after - revenue_before) OVER (), 0))::numeric, 1) AS share_of_change_pct
FROM   by_segment
ORDER  BY ABS(revenue_after - revenue_before) DESC, customer_country;

-- 3) Change in revenue by `category_name` (1997-Q4 → 1998-Q1)
WITH by_segment AS (
    SELECT category_name,
           SUM(CASE WHEN (order_year = 1997 AND quarter_number = 4) THEN revenue ELSE 0 END) AS revenue_before,
           SUM(CASE WHEN (order_year = 1998 AND quarter_number = 1) THEN revenue ELSE 0 END) AS revenue_after
    FROM   line_facts
    GROUP  BY category_name
)
SELECT category_name,
       ROUND((revenue_before)::numeric, 2) AS revenue_before,
       ROUND((revenue_after)::numeric, 2) AS revenue_after,
       ROUND((revenue_after - revenue_before)::numeric, 2) AS revenue_change,
       ROUND((100.0 * (revenue_after - revenue_before) / NULLIF(SUM(revenue_after - revenue_before) OVER (), 0))::numeric, 1) AS share_of_change_pct
FROM   by_segment
ORDER  BY ABS(revenue_after - revenue_before) DESC, category_name;

-- 4) Change in revenue by `employee_last_name` (1997-Q4 → 1998-Q1)
WITH by_segment AS (
    SELECT employee_last_name,
           SUM(CASE WHEN (order_year = 1997 AND quarter_number = 4) THEN revenue ELSE 0 END) AS revenue_before,
           SUM(CASE WHEN (order_year = 1998 AND quarter_number = 1) THEN revenue ELSE 0 END) AS revenue_after
    FROM   order_facts
    GROUP  BY employee_last_name
)
SELECT employee_last_name,
       ROUND((revenue_before)::numeric, 2) AS revenue_before,
       ROUND((revenue_after)::numeric, 2) AS revenue_after,
       ROUND((revenue_after - revenue_before)::numeric, 2) AS revenue_change,
       ROUND((100.0 * (revenue_after - revenue_before) / NULLIF(SUM(revenue_after - revenue_before) OVER (), 0))::numeric, 1) AS share_of_change_pct
FROM   by_segment
ORDER  BY ABS(revenue_after - revenue_before) DESC, employee_last_name;
```
</details>

In [25]:
outputs = {}
# The drivers of revenue in 1997-Q4 and 1998-Q1
outputs['tree'] = sql(r"""
-- revenue = orders × average order value; orders = customers × orders per customer; AOV = units per order × revenue per unit
SELECT order_year,
       quarter_number,
       ROUND(SUM(revenue), 2) AS revenue,
       COUNT(*) AS orders,
       COUNT(DISTINCT customer_id) AS customers,
       ROUND(1.0 * COUNT(*) / NULLIF(COUNT(DISTINCT customer_id), 0), 4) AS orders_per_customer,
       ROUND(1.0 * SUM(revenue) / NULLIF(COUNT(*), 0), 4) AS avg_order_value,
       ROUND(1.0 * SUM(units) / NULLIF(COUNT(*), 0), 4) AS units_per_order,
       ROUND(1.0 * SUM(revenue) / NULLIF(SUM(units), 0), 4) AS revenue_per_unit
FROM   order_facts
WHERE  (order_year = 1997 AND quarter_number = 4)
   OR  (order_year = 1998 AND quarter_number = 1)
GROUP  BY order_year, quarter_number
ORDER  BY order_year, quarter_number
""")
display(outputs['tree'].head(20))
# Change in revenue by `customer_country` (1997-Q4 → 1998-Q1)
outputs['by_customer_country'] = sql(r"""
WITH by_segment AS (
    SELECT customer_country,
           SUM(CASE WHEN (order_year = 1997 AND quarter_number = 4) THEN revenue ELSE 0 END) AS revenue_before,
           SUM(CASE WHEN (order_year = 1998 AND quarter_number = 1) THEN revenue ELSE 0 END) AS revenue_after
    FROM   order_facts
    GROUP  BY customer_country
)
SELECT customer_country,
       ROUND(revenue_before, 2) AS revenue_before,
       ROUND(revenue_after, 2) AS revenue_after,
       ROUND(revenue_after - revenue_before, 2) AS revenue_change,
       ROUND(100.0 * (revenue_after - revenue_before) / NULLIF(SUM(revenue_after - revenue_before) OVER (), 0), 1) AS share_of_change_pct
FROM   by_segment
ORDER  BY ABS(revenue_after - revenue_before) DESC, customer_country
""")
display(outputs['by_customer_country'].head(20))
# Change in revenue by `category_name` (1997-Q4 → 1998-Q1)
outputs['by_category_name'] = sql(r"""
WITH by_segment AS (
    SELECT category_name,
           SUM(CASE WHEN (order_year = 1997 AND quarter_number = 4) THEN revenue ELSE 0 END) AS revenue_before,
           SUM(CASE WHEN (order_year = 1998 AND quarter_number = 1) THEN revenue ELSE 0 END) AS revenue_after
    FROM   line_facts
    GROUP  BY category_name
)
SELECT category_name,
       ROUND(revenue_before, 2) AS revenue_before,
       ROUND(revenue_after, 2) AS revenue_after,
       ROUND(revenue_after - revenue_before, 2) AS revenue_change,
       ROUND(100.0 * (revenue_after - revenue_before) / NULLIF(SUM(revenue_after - revenue_before) OVER (), 0), 1) AS share_of_change_pct
FROM   by_segment
ORDER  BY ABS(revenue_after - revenue_before) DESC, category_name
""")
display(outputs['by_category_name'].head(20))
# Change in revenue by `employee_last_name` (1997-Q4 → 1998-Q1)
outputs['by_employee_last_name'] = sql(r"""
WITH by_segment AS (
    SELECT employee_last_name,
           SUM(CASE WHEN (order_year = 1997 AND quarter_number = 4) THEN revenue ELSE 0 END) AS revenue_before,
           SUM(CASE WHEN (order_year = 1998 AND quarter_number = 1) THEN revenue ELSE 0 END) AS revenue_after
    FROM   order_facts
    GROUP  BY employee_last_name
)
SELECT employee_last_name,
       ROUND(revenue_before, 2) AS revenue_before,
       ROUND(revenue_after, 2) AS revenue_after,
       ROUND(revenue_after - revenue_before, 2) AS revenue_change,
       ROUND(100.0 * (revenue_after - revenue_before) / NULLIF(SUM(revenue_after - revenue_before) OVER (), 0), 1) AS share_of_change_pct
FROM   by_segment
ORDER  BY ABS(revenue_after - revenue_before) DESC, employee_last_name
""")
display(outputs['by_employee_last_name'].head(20))
result = list(outputs.values())[-1] if outputs else pd.DataFrame()
# 📌 What we found
tree = outputs["tree"]
before, after = tree.iloc[0], tree.iloc[-1]
o1, o2 = float(before.orders), float(after.orders)
r1, r2 = float(before.revenue), float(after.revenue)
a1, a2 = r1 / o1, r2 / o2
volume, value, both = (o2 - o1) * a1, o1 * (a2 - a1), (o2 - o1) * (a2 - a1)
delta = r2 - r1
metrics["driver_split"] = {"from": '1997-Q4', "to": '1998-Q1', "delta": delta, "volume": volume, "order_value": value, "interaction": both}
steps = [("before", r1), ("more/fewer orders", volume), ("order value", value), ("interaction", both)]
fig, ax = plt.subplots(figsize=(8, 3.5))
level = 0.0
for i, (name, amount) in enumerate(steps):
    bottom = 0 if i == 0 else level if amount >= 0 else level + amount
    ax.bar(name, abs(amount), bottom=bottom, width=0.6, color=C_TOTAL if i == 0 else C_UP if amount >= 0 else C_DOWN)
    level = amount if i == 0 else level + amount
ax.bar("after", r2, width=0.6, color=C_TOTAL)
ax.set_title('1997-Q4' + " → " + '1998-Q1' + ": what moved revenue"); plt.tight_layout(); plt.show()
sizes = {"orders": abs(volume), "order value": abs(value), "both": abs(both)}
main = "orders" if sizes["orders"] >= sizes["order value"] else "order value"
share = sizes[main] / (sum(sizes.values()) or 1.0)   # share of the movement by size: never above 100%
note(f"Revenue moved **{delta:+,.2f}** ({100 * delta / r1:+.1f}%): **{volume:+,.2f}** from the number of orders "
     f"({int(o1):,} → {int(o2):,}), **{value:+,.2f}** from the average order value ({a1:,.2f} → {a2:,.2f}), and {both:+,.2f} from both together. "
     f"**{main.capitalize()}** accounts for about {share:.0%} of the movement.")
if "orders_per_customer" in tree:
    note(f"Orders = customers × orders per customer: customers {int(before.customers):,} → {int(after.customers):,}, "
         f"orders per customer {float(before.orders_per_customer):.2f} → {float(after.orders_per_customer):.2f}.")
contributors = {}
for label in [k for k in outputs if k.startswith("by_")]:
    seg = outputs[label]
    col = seg.columns[0]
    top = seg.iloc[0]
    contributors[col] = {"segment": str(top[col]), "change": float(top.revenue_change), "share": float(top.share_of_change_pct)}
    metrics.setdefault("segment_changes", {})[col] = [
        {"segment": str(r[0]), "before": float(r.revenue_before), "after": float(r.revenue_after), "change": float(r.revenue_change)}
        for r in seg.itertuples(index=False)]
    note(f"By `{col}`: **{top[col]}** moved most ({float(top.revenue_change):+,.2f}, {float(top.share_of_change_pct):.0f}% of the change).")
metrics["top_contributors"] = contributors

,order_year,quarter_number,revenue,orders,customers,orders_per_customer,avg_order_value,units_per_order,revenue_per_unit
0,1997,4,181681.46,120,60,2.0000,1514.0122,60.1417,25.1741
1,1998,1,298491.55,182,72,2.5278,1640.0635,58.4945,28.0379


,customer_country,revenue_before,revenue_after,revenue_change,share_of_change_pct
0,USA,31877.45,55943.22,24065.77,20.6
1,Brazil,14179.00,37557.34,23378.34,20.0
2,Germany,39797.80,54893.31,15095.51,12.9
3,Ireland,5094.98,19059.17,13964.19,12.0
4,Belgium,2945.00,13301.20,10356.20,8.9
5,Venezuela,6482.29,13048.01,6565.72,5.6
6,France,11886.27,17683.43,5797.16,5.0
7,Argentina,718.50,5921.50,5203.00,4.5
8,Denmark,8523.69,3343.50,-5180.19,-4.4
9,Switzerland,7478.28,2356.42,-5121.87,-4.4


,category_name,revenue_before,revenue_after,revenue_change,share_of_change_pct
0,Beverages,23102.55,89606.13,66503.58,56.9
1,Confections,21660.70,43785.80,22125.10,18.9
2,Seafood,20675.60,34395.90,13720.31,11.7
3,Dairy Products,35706.32,42831.18,7124.85,6.1
4,Produce,22301.01,15730.00,-6571.01,-5.6
5,Condiments,16174.79,21805.14,5630.35,4.8
6,Grains/Cereals,14060.70,19408.24,5347.54,4.6
7,Meat/Poultry,27999.80,30929.18,2929.38,2.5


,employee_last_name,revenue_before,revenue_after,revenue_change,share_of_change_pct
0,Leverling,34861.70,63605.37,28743.67,24.6
1,Dodsworth,9405.36,31601.66,22196.30,19.0
2,Fuller,21272.04,41416.30,20144.26,17.2
3,King,3404.50,19113.48,15708.98,13.4
4,Callahan,19082.08,32097.84,13015.76,11.1
5,Davolio,31844.51,44090.32,12245.81,10.5
6,Suyama,19939.27,8897.20,-11042.06,-9.5
7,Buchanan,8572.59,19481.90,10909.31,9.3
8,Peacock,33299.42,38187.48,4888.06,4.2


📌 Revenue moved **+116,810.09** (+64.3%): **+93,868.75** from the number of orders (120 → 182), **+15,126.16** from the average order value (1,514.01 → 1,640.06), and +7,815.18 from both together. **Orders** accounts for about 80% of the movement.

📌 Orders = customers × orders per customer: customers 60 → 72, orders per customer 2.00 → 2.53.

📌 By `customer_country`: **USA** moved most (+24,065.77, 21% of the change).

📌 By `category_name`: **Beverages** moved most (+66,503.58, 57% of the change).

📌 By `employee_last_name`: **Leverling** moved most (+28,743.67, 25% of the change).

## 21. Drill down: symptom next to driver

*6 · Root cause*

🔍 **What we found before:** The suspected driver(s) are `avg_ship_days`, `late_pct`.  
💡 **Why it matters:** A driver only explains a symptom if they move together, over time and across segments. Putting both in the same query, period by period and segment by segment, is how 'revenue fell because shipping got slower' is checked instead of assumed.  
🎯 **What we're going to do:** We line the symptom up with the driver for every complete period and for every `customer_country`, so that the data says whether they move together. Correlation is not proof: the next step is a controlled test.

🛠 **What this SQL does:** The first query groups order_facts per complete period and returns the symptom next to the driver. The second compares the last two complete periods per segment with conditional aggregation, so that the segment where the symptom moved can be checked for a matching move in the driver.

<details><summary>MySQL 8</summary>

```sql
-- Why: We line the symptom up with the driver for every complete period and for every
--      customer_country, so that the data says whether they move together. Correlation is not
--      proof: the next step is a controlled test.

-- 1) Symptom and driver per complete quarter
-- Complete periods only: a partial period would show a fake drop in the symptom.
SELECT order_year,
       quarter_number,
       ROUND(SUM(revenue), 2) AS revenue,
       ROUND(AVG(ship_days), 1) AS avg_ship_days,
       ROUND(100.0 * SUM(shipped_late) / NULLIF(COUNT(shipped_late), 0), 1) AS late_pct
FROM   order_facts
WHERE  order_year IS NOT NULL
  AND  NOT (order_year = 1998 AND quarter_number = 2)
GROUP  BY order_year, quarter_number
ORDER  BY order_year, quarter_number;

-- 2) Symptom and driver per `customer_country`, 1997-Q4 vs 1998-Q1
-- AVG(CASE WHEN … THEN x END) averages only the rows of that period: CASE without ELSE gives NULL, and AVG skips NULLs.
SELECT customer_country,
       ROUND(SUM(CASE WHEN (order_year = 1997 AND quarter_number = 4) THEN revenue ELSE 0 END), 2) AS revenue_before,
       ROUND(SUM(CASE WHEN (order_year = 1998 AND quarter_number = 1) THEN revenue ELSE 0 END), 2) AS revenue_after,
       ROUND(100.0 * (SUM(CASE WHEN (order_year = 1998 AND quarter_number = 1) THEN revenue ELSE 0 END) - SUM(CASE WHEN (order_year = 1997 AND quarter_number = 4) THEN revenue ELSE 0 END)) / NULLIF(SUM(CASE WHEN (order_year = 1997 AND quarter_number = 4) THEN revenue ELSE 0 END), 0), 1) AS revenue_change_pct,
       ROUND(AVG(CASE WHEN (order_year = 1997 AND quarter_number = 4) THEN ship_days END), 1) AS avg_ship_days_before,
       ROUND(100.0 * SUM(CASE WHEN (order_year = 1997 AND quarter_number = 4) THEN shipped_late END) / NULLIF(COUNT(CASE WHEN (order_year = 1997 AND quarter_number = 4) THEN shipped_late END), 0), 1) AS late_pct_before,
       ROUND(AVG(CASE WHEN (order_year = 1998 AND quarter_number = 1) THEN ship_days END), 1) AS avg_ship_days_after,
       ROUND(100.0 * SUM(CASE WHEN (order_year = 1998 AND quarter_number = 1) THEN shipped_late END) / NULLIF(COUNT(CASE WHEN (order_year = 1998 AND quarter_number = 1) THEN shipped_late END), 0), 1) AS late_pct_after
FROM   order_facts
WHERE  (order_year = 1997 AND quarter_number = 4)
   OR  (order_year = 1998 AND quarter_number = 1)
GROUP  BY customer_country
ORDER  BY ABS(SUM(CASE WHEN (order_year = 1998 AND quarter_number = 1) THEN revenue ELSE 0 END) - SUM(CASE WHEN (order_year = 1997 AND quarter_number = 4) THEN revenue ELSE 0 END)) DESC, customer_country;
```
</details>

<details><summary>PostgreSQL</summary>

```sql
-- Why: We line the symptom up with the driver for every complete period and for every
--      customer_country, so that the data says whether they move together. Correlation is not
--      proof: the next step is a controlled test.

-- 1) Symptom and driver per complete quarter
-- Complete periods only: a partial period would show a fake drop in the symptom.
SELECT order_year,
       quarter_number,
       ROUND((SUM(revenue))::numeric, 2) AS revenue,
       ROUND((AVG(ship_days))::numeric, 1) AS avg_ship_days,
       ROUND((100.0 * SUM(shipped_late) / NULLIF(COUNT(shipped_late), 0))::numeric, 1) AS late_pct
FROM   order_facts
WHERE  order_year IS NOT NULL
  AND  NOT (order_year = 1998 AND quarter_number = 2)
GROUP  BY order_year, quarter_number
ORDER  BY order_year, quarter_number;

-- 2) Symptom and driver per `customer_country`, 1997-Q4 vs 1998-Q1
-- AVG(CASE WHEN … THEN x END) averages only the rows of that period: CASE without ELSE gives NULL, and AVG skips NULLs.
SELECT customer_country,
       ROUND((SUM(CASE WHEN (order_year = 1997 AND quarter_number = 4) THEN revenue ELSE 0 END))::numeric, 2) AS revenue_before,
       ROUND((SUM(CASE WHEN (order_year = 1998 AND quarter_number = 1) THEN revenue ELSE 0 END))::numeric, 2) AS revenue_after,
       ROUND((100.0 * (SUM(CASE WHEN (order_year = 1998 AND quarter_number = 1) THEN revenue ELSE 0 END) - SUM(CASE WHEN (order_year = 1997 AND quarter_number = 4) THEN revenue ELSE 0 END)) / NULLIF(SUM(CASE WHEN (order_year = 1997 AND quarter_number = 4) THEN revenue ELSE 0 END), 0))::numeric, 1) AS revenue_change_pct,
       ROUND((AVG(CASE WHEN (order_year = 1997 AND quarter_number = 4) THEN ship_days END))::numeric, 1) AS avg_ship_days_before,
       ROUND((100.0 * SUM(CASE WHEN (order_year = 1997 AND quarter_number = 4) THEN shipped_late END) / NULLIF(COUNT(CASE WHEN (order_year = 1997 AND quarter_number = 4) THEN shipped_late END), 0))::numeric, 1) AS late_pct_before,
       ROUND((AVG(CASE WHEN (order_year = 1998 AND quarter_number = 1) THEN ship_days END))::numeric, 1) AS avg_ship_days_after,
       ROUND((100.0 * SUM(CASE WHEN (order_year = 1998 AND quarter_number = 1) THEN shipped_late END) / NULLIF(COUNT(CASE WHEN (order_year = 1998 AND quarter_number = 1) THEN shipped_late END), 0))::numeric, 1) AS late_pct_after
FROM   order_facts
WHERE  (order_year = 1997 AND quarter_number = 4)
   OR  (order_year = 1998 AND quarter_number = 1)
GROUP  BY customer_country
ORDER  BY ABS(SUM(CASE WHEN (order_year = 1998 AND quarter_number = 1) THEN revenue ELSE 0 END) - SUM(CASE WHEN (order_year = 1997 AND quarter_number = 4) THEN revenue ELSE 0 END)) DESC, customer_country;
```
</details>

In [26]:
outputs = {}
# Symptom and driver per complete quarter
outputs['series'] = sql(r"""
-- Complete periods only: a partial period would show a fake drop in the symptom.
SELECT order_year,
       quarter_number,
       ROUND(SUM(revenue), 2) AS revenue,
       ROUND(AVG(ship_days), 1) AS avg_ship_days,
       ROUND(100.0 * SUM(shipped_late) / NULLIF(COUNT(shipped_late), 0), 1) AS late_pct
FROM   order_facts
WHERE  order_year IS NOT NULL
  AND  NOT (order_year = 1998 AND quarter_number = 2)
GROUP  BY order_year, quarter_number
ORDER  BY order_year, quarter_number
""")
display(outputs['series'].head(20))
# Symptom and driver per `customer_country`, 1997-Q4 vs 1998-Q1
outputs['by_segment'] = sql(r"""
-- AVG(CASE WHEN … THEN x END) averages only the rows of that period: CASE without ELSE gives NULL, and AVG skips NULLs.
SELECT customer_country,
       ROUND(SUM(CASE WHEN (order_year = 1997 AND quarter_number = 4) THEN revenue ELSE 0 END), 2) AS revenue_before,
       ROUND(SUM(CASE WHEN (order_year = 1998 AND quarter_number = 1) THEN revenue ELSE 0 END), 2) AS revenue_after,
       ROUND(100.0 * (SUM(CASE WHEN (order_year = 1998 AND quarter_number = 1) THEN revenue ELSE 0 END) - SUM(CASE WHEN (order_year = 1997 AND quarter_number = 4) THEN revenue ELSE 0 END)) / NULLIF(SUM(CASE WHEN (order_year = 1997 AND quarter_number = 4) THEN revenue ELSE 0 END), 0), 1) AS revenue_change_pct,
       ROUND(AVG(CASE WHEN (order_year = 1997 AND quarter_number = 4) THEN ship_days END), 1) AS avg_ship_days_before,
       ROUND(100.0 * SUM(CASE WHEN (order_year = 1997 AND quarter_number = 4) THEN shipped_late END) / NULLIF(COUNT(CASE WHEN (order_year = 1997 AND quarter_number = 4) THEN shipped_late END), 0), 1) AS late_pct_before,
       ROUND(AVG(CASE WHEN (order_year = 1998 AND quarter_number = 1) THEN ship_days END), 1) AS avg_ship_days_after,
       ROUND(100.0 * SUM(CASE WHEN (order_year = 1998 AND quarter_number = 1) THEN shipped_late END) / NULLIF(COUNT(CASE WHEN (order_year = 1998 AND quarter_number = 1) THEN shipped_late END), 0), 1) AS late_pct_after
FROM   order_facts
WHERE  (order_year = 1997 AND quarter_number = 4)
   OR  (order_year = 1998 AND quarter_number = 1)
GROUP  BY customer_country
ORDER  BY ABS(SUM(CASE WHEN (order_year = 1998 AND quarter_number = 1) THEN revenue ELSE 0 END) - SUM(CASE WHEN (order_year = 1997 AND quarter_number = 4) THEN revenue ELSE 0 END)) DESC, customer_country
""")
display(outputs['by_segment'].head(20))
result = list(outputs.values())[-1] if outputs else pd.DataFrame()
# 📌 What we found
s = outputs["series"].copy()
main, drivers = 'revenue', ['avg_ship_days', 'late_pct']
drivers = [x for x in drivers if x in s]
if not drivers and len(s):
    note(f"No operational driver column (shipping time, rating) to line up with {main}; over the {len(s)} complete period(s), "
         f"{main} went from {float(s[main].iloc[0]):,.2f} to {float(s[main].iloc[-1]):,.2f}. The segment view shows where it moved.")
if drivers and len(s) >= 6:
    for x in drivers:
        r = s[main].astype(float).corr(s[x].astype(float))
        metrics.setdefault("driver_corr", {})[x] = float(r)
        note(f"Across {len(s)} complete periods, {main} and {x} have a correlation of **{r:+.2f}** "
             + ("(they move against each other)" if r < -0.5 else "(they move together)" if r > 0.5 else "(little linear relationship)") + ".")
elif drivers:
    note(f"Only {len(s)} complete periods: too few for a meaningful correlation (at least 6 are needed), so compare the columns side by side.")
if drivers and len(s) >= 2:
    first, last = s.iloc[-2], s.iloc[-1]
    x = drivers[0]
    change_main = 100 * (float(last[main]) - float(first[main])) / float(first[main]) if float(first[main]) else float("nan")
    change_x = 100 * (float(last[x]) - float(first[x])) / float(first[x]) if float(first[x]) else float("nan")
    moved = abs(change_x) >= 10
    # more days or more late orders is worse; a higher rating is better. A candidate cause got worse while the symptom fell,
    # or better while it grew; a driver that moved the other way cannot explain the change.
    worse = change_x < 0 if x.endswith("rating") else change_x > 0
    candidate = moved and worse == (change_main < 0)
    metrics["driver_vs_symptom"] = {"driver": x, "driver_change_pct": change_x, "symptom_change_pct": change_main, "candidate": candidate}
    note(f"Last two complete periods: {main} **{change_main:+.1f}%**, {x} **{change_x:+.1f}%** "
         f"({float(first[x]):.1f} → {float(last[x]):.1f}). "
         + ("The driver moved the way that would explain it: a candidate cause, to confirm with a controlled test "
            "(e.g. a service-level target for a random half of orders)." if candidate else
            f"{x} moved, but in the direction that would push {main} the other way, so it does not explain this change."
            if moved else f"{x} barely moved, so it does not explain this change; look at the segments in the driver tree instead."))
if "by_segment" in outputs:
    seg = outputs["by_segment"]
    top = seg.iloc[0]
    col = seg.columns[0]
    note(f"Segment that moved most: **{top[col]}** ({main} {float(top[f'{main}_change_pct']):+.1f}%)" +
         "".join(f", {x} {top[x + '_before']} → {top[x + '_after']}" for x in drivers if x + "_before" in seg) + ".")

,order_year,quarter_number,revenue,avg_ship_days,late_pct
0,1996,3,79728.57,8.9,7.1
1,1996,4,128355.40,7.5,2.4
2,1997,1,138288.93,9.2,5.4
3,1997,2,143177.04,9.0,4.3
4,1997,3,153937.77,8.2,4.9
5,1997,4,181681.46,9.2,6.7
6,1998,1,298491.55,8.6,4.4


,customer_country,revenue_before,revenue_after,revenue_change_pct,avg_ship_days_before,late_pct_before,avg_ship_days_after,late_pct_after
0,USA,31877.45,55943.22,75.5,9.5,0.0,9.0,8.3
1,Brazil,14179.00,37557.34,164.9,11.0,14.3,7.6,0.0
2,Germany,39797.80,54893.31,37.9,9.8,5.9,5.7,0.0
3,Ireland,5094.98,19059.17,274.1,7.3,0.0,9.7,0.0
4,Belgium,2945.00,13301.20,351.7,9.5,0.0,9.4,0.0
5,Venezuela,6482.29,13048.01,101.3,14.2,16.7,9.1,8.3
6,France,11886.27,17683.43,48.8,5.6,0.0,9.6,10.0
7,Argentina,718.50,5921.50,724.1,4.0,0.0,12.0,12.5
8,Denmark,8523.69,3343.50,-60.8,5.3,0.0,6.5,0.0
9,Switzerland,7478.28,2356.42,-68.5,6.3,0.0,18.0,0.0


📌 Across 7 complete periods, revenue and avg_ship_days have a correlation of **+0.02** (little linear relationship).

📌 Across 7 complete periods, revenue and late_pct have a correlation of **-0.20** (little linear relationship).

📌 Last two complete periods: revenue **+64.3%**, avg_ship_days **-6.5%** (9.2 → 8.6). avg_ship_days barely moved, so it does not explain this change; look at the segments in the driver tree instead.

📌 Segment that moved most: **USA** (revenue +75.5%), avg_ship_days 9.5 → 9.0, late_pct 0.0 → 8.3.

## 22. Cohorts by first-order quarter

*6 · Root cause*

🔍 **What we found before:** Each customer has a first order date.  
💡 **Why it matters:** Averages mix old and new customers. A cohort table follows each group of customers from their first order, so you can see whether newer customers come back as often as older ones did.  
🎯 **What we're going to do:** We group customers by the quarter of their first order and count how many are active in each later quarter, so that retention is measured per cohort.

🛠 **What this SQL does:** `first_orders` finds each customer's first order (MIN), `cohorts` turns it into a quarter, `activity` lists the quarters in which each customer ordered, and the join counts active customers by cohort and by quarters since the first order (year × 4 + quarter arithmetic, which works in every dialect).

<details><summary>MySQL 8</summary>

```sql
-- Why: We group customers by the quarter of their first order and count how many are active in each
--      later quarter, so that retention is measured per cohort.

-- Active customers by cohort and quarters since first order
WITH first_orders AS (
    SELECT customer_id, MIN(order_date) AS first_date
    FROM   order_facts
    WHERE  customer_id IS NOT NULL AND order_date IS NOT NULL
    GROUP  BY customer_id
),
cohorts AS (
    SELECT customer_id,
           YEAR(first_date) AS cohort_year,
           QUARTER(first_date) AS cohort_number
    FROM   first_orders
),
activity AS (
    SELECT DISTINCT customer_id, order_year, quarter_number
    FROM   order_facts
    WHERE  order_year IS NOT NULL
)
SELECT c.cohort_year, c.cohort_number,
       (a.order_year * 4 + a.quarter_number) - (c.cohort_year * 4 + c.cohort_number) AS quarters_since_first,
       COUNT(*) AS active_customers
FROM   cohorts c
JOIN   activity a ON a.customer_id = c.customer_id
GROUP  BY c.cohort_year, c.cohort_number, (a.order_year * 4 + a.quarter_number) - (c.cohort_year * 4 + c.cohort_number)
ORDER  BY c.cohort_year, c.cohort_number, quarters_since_first;
```
</details>

<details><summary>PostgreSQL</summary>

```sql
-- Why: We group customers by the quarter of their first order and count how many are active in each
--      later quarter, so that retention is measured per cohort.

-- Active customers by cohort and quarters since first order
WITH first_orders AS (
    SELECT customer_id, MIN(order_date) AS first_date
    FROM   order_facts
    WHERE  customer_id IS NOT NULL AND order_date IS NOT NULL
    GROUP  BY customer_id
),
cohorts AS (
    SELECT customer_id,
           EXTRACT(YEAR FROM first_date)::int AS cohort_year,
           EXTRACT(QUARTER FROM first_date)::int AS cohort_number
    FROM   first_orders
),
activity AS (
    SELECT DISTINCT customer_id, order_year, quarter_number
    FROM   order_facts
    WHERE  order_year IS NOT NULL
)
SELECT c.cohort_year, c.cohort_number,
       (a.order_year * 4 + a.quarter_number) - (c.cohort_year * 4 + c.cohort_number) AS quarters_since_first,
       COUNT(*) AS active_customers
FROM   cohorts c
JOIN   activity a ON a.customer_id = c.customer_id
GROUP  BY c.cohort_year, c.cohort_number, (a.order_year * 4 + a.quarter_number) - (c.cohort_year * 4 + c.cohort_number)
ORDER  BY c.cohort_year, c.cohort_number, quarters_since_first;
```
</details>

In [27]:
outputs = {}
# Active customers by cohort and quarters since first order
outputs['cohort'] = sql(r"""
WITH first_orders AS (
    SELECT customer_id, MIN(order_date) AS first_date
    FROM   order_facts
    WHERE  customer_id IS NOT NULL AND order_date IS NOT NULL
    GROUP  BY customer_id
),
cohorts AS (
    SELECT customer_id,
           EXTRACT(YEAR FROM first_date)::int AS cohort_year,
           EXTRACT(QUARTER FROM first_date)::int AS cohort_number
    FROM   first_orders
),
activity AS (
    SELECT DISTINCT customer_id, order_year, quarter_number
    FROM   order_facts
    WHERE  order_year IS NOT NULL
)
SELECT c.cohort_year, c.cohort_number,
       (a.order_year * 4 + a.quarter_number) - (c.cohort_year * 4 + c.cohort_number) AS quarters_since_first,
       COUNT(*) AS active_customers
FROM   cohorts c
JOIN   activity a ON a.customer_id = c.customer_id
GROUP  BY c.cohort_year, c.cohort_number, (a.order_year * 4 + a.quarter_number) - (c.cohort_year * 4 + c.cohort_number)
ORDER  BY c.cohort_year, c.cohort_number, quarters_since_first
""")
display(outputs['cohort'].head(20))
result = list(outputs.values())[-1] if outputs else pd.DataFrame()
# 📌 What we found
coh = result.copy()
cols = list(coh.columns)
key = cols[:-2]
coh["cohort"] = coh[key].astype(int).astype(str).agg("-".join, axis=1)
grid = coh.pivot_table(index="cohort", columns=cols[-2], values="active_customers", aggfunc="sum").sort_index()
size = grid[0]
retention = grid.div(size, axis=0) * 100
metrics["cohort_sizes"] = {str(k): int(v) for k, v in size.items()}
plt.figure(figsize=(9, max(3, 0.4 * len(grid))))
plt.imshow(retention.to_numpy(dtype=float), aspect="auto", cmap=C_SEQ, vmin=0, vmax=100)
plt.colorbar(label="% of cohort active")
plt.yticks(range(len(grid)), grid.index); plt.xlabel(cols[-2]); plt.title("Retention by cohort")
plt.tight_layout(); plt.show()
biggest = size.idxmax()
note(f"Largest cohort: **{biggest}** with {int(size.max()):,} new customers. " + (
     f"A period after their first order, {retention[1].mean():.0f}% of a cohort is active again on average." if 1 in retention else ""))

,cohort_year,cohort_number,quarters_since_first,active_customers
0,1996,3,0,43
1,1996,3,1,21
2,1996,3,2,27
3,1996,3,3,29
4,1996,3,4,30
5,1996,3,5,30
6,1996,3,6,37
7,1996,3,7,27
8,1996,4,0,24
9,1996,4,1,19


📌 Largest cohort: **1996-3** with 43 new customers. A period after their first order, 63% of a cohort is active again on average.

## 23. Read the plan; keep filters sargable

*7 · Fast & correct*

🔍 **What we found before:** The analysis joins on 7 key(s) and filters on `order_date`.  
💡 **Why it matters:** On a real database, the same SQL can take milliseconds or minutes depending on indexes and on whether filters are *sargable* (able to use an index). EXPLAIN shows the plan before you wait for it.  
🎯 **What we're going to do:** We read the plan, show that a function wrapped around a column returns the same rows as a range filter but blocks the index, and propose indexes, so that the queries stay fast on large tables.

🛠 **What this SQL does:** EXPLAIN prints the operators the engine will use (scans, joins, aggregations). The second query counts the same 1997 rows twice: YEAR(order_date) = 1997 must compute YEAR() for every row, while the range order_date >= '1997-01-01' AND < '1998-01-01' can seek an index. The CREATE INDEX statements are for your MySQL / PostgreSQL database; the in-memory engine here does not need them.

<details><summary>MySQL 8</summary>

```sql
-- Why: We read the plan, show that a function wrapped around a column returns the same rows as a
--      range filter but blocks the index, and propose indexes, so that the queries stay fast on
--      large tables.

-- 1) The plan for the KPI-per-quarter query
EXPLAIN FORMAT=TREE
SELECT order_year, quarter_number, SUM(revenue) AS revenue, COUNT(*) AS orders
FROM   order_facts
GROUP  BY order_year, quarter_number;

-- 2) Same rows, two filters: YEAR(order_date) = 1997 vs a date range
SELECT (SELECT COUNT(*) FROM orders WHERE YEAR(order_date) = 1997) AS function_on_column,
       (SELECT COUNT(*) FROM orders
        WHERE  order_date >= DATE '1997-01-01' AND order_date < DATE '1998-01-01') AS sargable_range;

-- 3) Indexes for the join and filter columns  [not run here: shown for your database]
-- InnoDB already indexes foreign key columns; these are listed for completeness.
-- foreign key
CREATE INDEX idx_order_details_product_id ON order_details (product_id);
-- foreign key
CREATE INDEX idx_orders_customer_id ON orders (customer_id);
-- foreign key
CREATE INDEX idx_orders_employee_id ON orders (employee_id);
-- foreign key
CREATE INDEX idx_products_supplier_id ON products (supplier_id);
-- foreign key
CREATE INDEX idx_products_category_id ON products (category_id);
-- foreign key
CREATE INDEX idx_employees_reports_to ON employees (reports_to);
-- date filter
CREATE INDEX idx_orders_order_date ON orders (order_date);
```
</details>

<details><summary>PostgreSQL</summary>

```sql
-- Why: We read the plan, show that a function wrapped around a column returns the same rows as a
--      range filter but blocks the index, and propose indexes, so that the queries stay fast on
--      large tables.

-- 1) The plan for the KPI-per-quarter query
EXPLAIN
SELECT order_year, quarter_number, SUM(revenue) AS revenue, COUNT(*) AS orders
FROM   order_facts
GROUP  BY order_year, quarter_number;

-- 2) Same rows, two filters: YEAR(order_date) = 1997 vs a date range
SELECT (SELECT COUNT(*) FROM orders WHERE EXTRACT(YEAR FROM order_date)::int = 1997) AS function_on_column,
       (SELECT COUNT(*) FROM orders
        WHERE  order_date >= DATE '1997-01-01' AND order_date < DATE '1998-01-01') AS sargable_range;

-- 3) Indexes for the join and filter columns  [not run here: shown for your database]
-- PostgreSQL does not index foreign key columns automatically.
-- foreign key
CREATE INDEX idx_order_details_product_id ON order_details (product_id);
-- foreign key
CREATE INDEX idx_orders_customer_id ON orders (customer_id);
-- foreign key
CREATE INDEX idx_orders_employee_id ON orders (employee_id);
-- foreign key
CREATE INDEX idx_products_supplier_id ON products (supplier_id);
-- foreign key
CREATE INDEX idx_products_category_id ON products (category_id);
-- foreign key
CREATE INDEX idx_employees_reports_to ON employees (reports_to);
-- date filter
CREATE INDEX idx_orders_order_date ON orders (order_date);
```
</details>

In [28]:
outputs = {}
# The plan for the KPI-per-quarter query
outputs['plan'] = sql(r"""
EXPLAIN
SELECT order_year, quarter_number, SUM(revenue) AS revenue, COUNT(*) AS orders
FROM   order_facts
GROUP  BY order_year, quarter_number
""")
display(outputs['plan'].head(20))
# Same rows, two filters: YEAR(order_date) = 1997 vs a date range
outputs['sargable'] = sql(r"""
SELECT (SELECT COUNT(*) FROM orders WHERE EXTRACT(YEAR FROM order_date)::int = 1997) AS function_on_column,
       (SELECT COUNT(*) FROM orders
        WHERE  order_date >= DATE '1997-01-01' AND order_date < DATE '1998-01-01') AS sargable_range
""")
display(outputs['sargable'].head(20))
result = list(outputs.values())[-1] if outputs else pd.DataFrame()
# 📌 What we found
import re as _re
plan = outputs["plan"]
text = "\n".join(str(v) for v in plan.iloc[:, -1]) if len(plan.columns) else ""
ops = [op.strip() for op in _re.findall(r"[╭┌]─+\s*([A-Za-z_][A-Za-z_ ]*?)\s*─", text)]
metrics["plan_ops"] = ops
if ops:
    note("Plan operators, top to bottom: " + " → ".join(dict.fromkeys(ops)) + ". Full scans are normal here: the query reads whole "
         "tables to aggregate them. Indexes matter for selective lookups and joins on large tables.")
if "sargable" in outputs:
    row = outputs["sargable"].iloc[0]
    same = int(row.function_on_column) == int(row.sargable_range)
    metrics["sargable_equal"] = same
    note(f"Both filters return **{int(row.sargable_range):,}** rows" + (" (identical)" if same else " (they differ: check time zones or timestamps)")
         + ". Only the range version can use an index on the date column.")

,explain_key,explain_value
0,physical_plan,╭─ Projection ────────────────────────────────...


,function_on_column,sargable_range
0,408,408


📌 Plan operators, top to bottom: Projection → Perfect Hash Group By → Hash Join → Seq Scan. Full scans are normal here: the query reads whole tables to aggregate them. Indexes matter for selective lookups and joins on large tables.

📌 Both filters return **408** rows (identical). Only the range version can use an index on the date column.

## 24. NULL traps: NOT IN, COUNT(column), integer division

*7 · Fast & correct*

🔍 **What we found before:** NULLs were found in the NULL profile, including in `employees.reports_to`.  
💡 **Why it matters:** NULL means unknown, and SQL's three-valued logic makes `x NOT IN (…, NULL)` unknown for every row, so NOT IN silently returns nothing. COUNT(column) and AVG skip NULLs. Integer division truncates in some engines. None of these raise an error; they just give wrong numbers.  
🎯 **What we're going to do:** We run each trap on your own data next to the safe version, so that the difference is visible and the safe habit sticks.

🛠 **What this SQL does:** The first query counts the same thing with NOT IN and with NOT EXISTS. NOT EXISTS checks for a matching row, so NULLs can't poison it. The second compares COUNT(*) with COUNT(column). The last block shows integer division for PostgreSQL; here it is not run because this engine never truncates.

<details><summary>MySQL 8</summary>

```sql
-- Why: We run each trap on your own data next to the safe version, so that the difference is
--      visible and the safe habit sticks.

-- 1) NOT IN vs NOT EXISTS when the sub-query can return NULL
-- Which employees rows does no employees.reports_to point at? reports_to contains NULLs.
SELECT (SELECT COUNT(*) FROM employees p
        WHERE  p.employee_id NOT IN (SELECT c.reports_to FROM employees c)) AS using_not_in,
       (SELECT COUNT(*) FROM employees p
        WHERE  NOT EXISTS (SELECT 1 FROM employees c WHERE c.reports_to = p.employee_id)) AS using_not_exists;

-- 2) COUNT(*) vs COUNT(shipped_date): aggregates skip NULLs
SELECT COUNT(*) AS all_rows,
       COUNT(shipped_date) AS rows_with_shipped_date,
       COUNT(*) - COUNT(shipped_date) AS null_shipped_date,
       ROUND(AVG(DATEDIFF(shipped_date, order_date)), 2) AS avg_ship_days_skips_nulls
FROM   orders;

-- 3) Integer division (shown for your database; this engine never truncates)  [not run here: shown for your database]
-- Integer ÷ integer truncates in PostgreSQL and SQL Server (MySQL's / and DuckDB's / are already decimal):
SELECT 2 / 3 AS integer_division,           -- 0 in PostgreSQL, 0.6667 in MySQL
       100.0 * 2 / 3 AS percentage          -- 66.67 everywhere;
```
</details>

<details><summary>PostgreSQL</summary>

```sql
-- Why: We run each trap on your own data next to the safe version, so that the difference is
--      visible and the safe habit sticks.

-- 1) NOT IN vs NOT EXISTS when the sub-query can return NULL
-- Which employees rows does no employees.reports_to point at? reports_to contains NULLs.
SELECT (SELECT COUNT(*) FROM employees p
        WHERE  p.employee_id NOT IN (SELECT c.reports_to FROM employees c)) AS using_not_in,
       (SELECT COUNT(*) FROM employees p
        WHERE  NOT EXISTS (SELECT 1 FROM employees c WHERE c.reports_to = p.employee_id)) AS using_not_exists;

-- 2) COUNT(*) vs COUNT(shipped_date): aggregates skip NULLs
SELECT COUNT(*) AS all_rows,
       COUNT(shipped_date) AS rows_with_shipped_date,
       COUNT(*) - COUNT(shipped_date) AS null_shipped_date,
       ROUND((AVG(shipped_date - order_date))::numeric, 2) AS avg_ship_days_skips_nulls
FROM   orders;

-- 3) Integer division (shown for your database; this engine never truncates)  [not run here: shown for your database]
-- Integer ÷ integer truncates in PostgreSQL and SQL Server (MySQL's / and DuckDB's / are already decimal):
SELECT 2 / 3 AS integer_division,           -- 0 in PostgreSQL, 0.6667 in MySQL
       100.0 * 2 / 3 AS percentage          -- 66.67 everywhere;
```
</details>

In [29]:
outputs = {}
# NOT IN vs NOT EXISTS when the sub-query can return NULL
outputs['not_in'] = sql(r"""
-- Which employees rows does no employees.reports_to point at? reports_to contains NULLs.
SELECT (SELECT COUNT(*) FROM employees p
        WHERE  p.employee_id NOT IN (SELECT c.reports_to FROM employees c)) AS using_not_in,
       (SELECT COUNT(*) FROM employees p
        WHERE  NOT EXISTS (SELECT 1 FROM employees c WHERE c.reports_to = p.employee_id)) AS using_not_exists
""")
display(outputs['not_in'].head(20))
# COUNT(*) vs COUNT(shipped_date): aggregates skip NULLs
outputs['count_nulls'] = sql(r"""
SELECT COUNT(*) AS all_rows,
       COUNT(shipped_date) AS rows_with_shipped_date,
       COUNT(*) - COUNT(shipped_date) AS null_shipped_date,
       ROUND(AVG(shipped_date - order_date), 2) AS avg_ship_days_skips_nulls
FROM   orders
""")
display(outputs['count_nulls'].head(20))
result = list(outputs.values())[-1] if outputs else pd.DataFrame()
# 📌 What we found
if not outputs:
    note("No foreign key and no nullable column here, so NOT IN and COUNT(column) can't go wrong on this data yet. "
         "The integer-division trap above still applies in PostgreSQL.")
if "not_in" in outputs:
    row = outputs["not_in"].iloc[0]
    values = {k: int(v) for k, v in row.items()}
    metrics["null_trap"] = values
    if "using_not_in" in values:
        note(f"**NOT IN returns {values['using_not_in']:,}, NOT EXISTS returns {values['using_not_exists']:,}.** "
             "One NULL in the sub-query makes every NOT IN comparison unknown. Use NOT EXISTS (or filter out NULLs).")
    else:
        note(f"Today NOT IN finds {values['not_in_today']:,} row(s); after a single NULL arrives it finds **{values['not_in_after_one_null']:,}**, "
             f"while NOT EXISTS keeps finding {values['not_exists']:,}. That is why NOT EXISTS is the safe habit.")
if "count_nulls" in outputs:
    row = outputs["count_nulls"].iloc[0]
    metrics["count_trap"] = {k: (float(v) if v is not None else None) for k, v in row.items()}
    note(f"COUNT(*) = {int(row.iloc[0]):,} but COUNT(column) = {int(row.iloc[1]):,}: {int(row.iloc[2]):,} NULL(s) are skipped by COUNT, SUM and AVG.")

,using_not_in,using_not_exists
0,0,7


,all_rows,rows_with_shipped_date,null_shipped_date,avg_ship_days_skips_nulls
0,830,809,21,8.49


📌 **NOT IN returns 0, NOT EXISTS returns 7.** One NULL in the sub-query makes every NOT IN comparison unknown. Use NOT EXISTS (or filter out NULLs).

📌 COUNT(*) = 830 but COUNT(column) = 809: 21 NULL(s) are skipped by COUNT, SUM and AVG.

# Part 3: the business reading

Every step above stored its key numbers in `metrics`. The business layer turns them into what a manager reads: one headline,
KPI tiles for the last complete quarter, recommendations ordered by urgency (each with its evidence, a confidence level and
the SQL step it comes from) and a data-trust score. The same page is written to `business_brief.html` next to this notebook:
open it in any browser. How to read it, and every rule behind it: `docs/business_guide.md`.

In [30]:
from autosql import decide, infer_schema, profile_catalog
from autosql.business import executive_summary
from autosql.report import to_business_brief

profile = profile_catalog(catalog)
decision = decide(catalog, profile, infer_schema(catalog, profile))
summary = executive_summary(decision, metrics, "Northwind Traders")
note(f"**{summary.headline}**")
tiles = pd.DataFrame([{"KPI": t.label, summary.period: round(t.value, 2),
                       f"vs {summary.compared_with}": (f"{t.delta_pct:+.1f} pts" if t.unit == "pct" else f"{t.delta_pct:+.1f}%")
                       if t.delta_pct is not None else "",
                       "for the business": {True: "better", False: "worse", None: ""}[t.good]} for t in summary.tiles])
display(tiles)
names = {1: "Act now", 2: "Plan", 3: "Monitor"}
display(pd.DataFrame([{"priority": names[i.priority], "recommendation": i.headline, "evidence": i.evidence, "next step": i.action,
                       "confidence": i.confidence, "SQL step": i.source} for i in summary.insights]))
note(f"Data trust: **{int(summary.trust.score)}/100, {summary.trust.grade}**. "
     + " ".join(f"{x.name} {int(x.score)}: {x.detail}" for x in summary.trust.dimensions))
Path("business_brief.html").write_text(to_business_brief(decision, metrics, title="Northwind Traders",
                                                         source="the Northwind sample (8 CSV files)"), encoding="utf-8")
note("Wrote `business_brief.html`: the one-page brief with the charts.")

📌 **Revenue reached 298,491.55 in 1998-Q1 (+64.3% on 1997-Q4, +115.8% year over year). Most of the movement came from more orders.**

,KPI,1998-Q1,vs 1997-Q4,for the business
0,Revenue,298491.55,+64.3%,better
1,Orders,182.00,+51.7%,better
2,Average order value,1640.06,+8.3%,better
3,Active customers,72.00,+20.0%,better
4,Days to ship,8.60,-6.5%,better
5,Late shipments,4.40,-2.3 pts,better


,priority,recommendation,evidence,next step,confidence,SQL step
0,Act now,1998-Q2 is incomplete,only 36 of 91 days have data; a naive report w...,"Report 1998-Q2 only once it is complete, or co...",measured,date_range
1,Plan,Revenue grew 64.3% in 1998-Q1,1997-Q4 → 1998-Q1: +64.3%; year over year +115.8%,"Use the last complete period as the headline, ...",measured,kpi_by_period
2,Plan,1998-Q1 is far from its recent trend,z = +4.8 against the 6 complete periods before...,Check the largest orders of that period before...,indicative,trend_signals
3,Plan,Order volume accounts for 80% of the movement,"1997-Q4 → 1998-Q1: 116,810.09 = volume 93,868....",Plan capacity for the higher volume; treat ord...,measured,driver_tree
4,Plan,Beverages drove 57% of the change,"by category_name: 66,503.58","Find out what changed in Beverages (customers,...",measured,driver_tree
5,Plan,The top 20% of customers bring 60% of revenue,class A: 34 customers (38%) = 80% of revenue,Give the 34 class-A customers named account ow...,measured,pareto
6,Plan,5 repeat customers are overdue,silent for more than twice their usual gap (ty...,"Contact them now (win-back), largest first; th...",measured,purchase_gaps
7,Plan,8 once-frequent customers are slipping,RFM 'At risk': 5.1% of revenue,Reactivation offer or account call; compare wi...,measured,rfm
8,Plan,Discounts cost 6.5% of gross revenue,"88,665.55 on 39% of lines; discounted lines ar...",Test discount levels with a controlled experim...,indicative,discounts
9,Monitor,ship days did not explain the change,-6.5% vs revenue +64.3%,Keep monitoring it; look at the driver tree in...,measured,drilldown


📌 Data trust: **99/100, High**. Completeness 99: 11 columns the analysis uses are 99.8% filled; most gaps in `orders.shipped_date` (2.5% empty). Uniqueness 100: 8 of 8 tables have a unique key and no copied rows. Integrity 100: all 7 relationships resolve, no orphan rows; 1 link(s) left for you to confirm. Validity 100: dates parse, and no impossible values (e.g. shipped before ordered). Consistency 100: totals agree between the raw table, the views and the uploaded file; 1998-Q2 is partial (labelled, not penalised).

📌 Wrote `business_brief.html`: the one-page brief with the charts.